### Import Libraries

In [1]:
import re
import json
import random
import pandas as pd
from elasticsearch import Elasticsearch, helpers
import copy
from thefuzz import fuzz
from thefuzz import process as process
import sqlalchemy as sa
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import os
from dotenv import load_dotenv
load_dotenv()
import time

### Read the unique values

In [2]:
# data_version = "28_11_24"
data_version = "27_01_25"

In [3]:
with open(f"unique_values_{data_version}.json", "r") as fp:
    unique_values = json.load(fp) # data not cleaned in certification

# only for certification refer his version
with open("unique_values_21_10_24.json", "r") as fp:
    unique_values_cert = json.load(fp)

with open(file="./unique_values_22_02_24.json") as f:
    OLD_UNIQUE_VALUES = json.load(f)

with open(f"all_brands_{data_version}.json", "r") as fp:
    all_brands = json.load(fp)
    
with open(f"all_cgrades_{data_version}.json", "r") as fp:
    all_cgrades = json.load(fp)
all_cgrades = [item.lower() for item in all_cgrades]  

with open(f"all_grades_{data_version}.json", "r") as fp:
    all_gradenames = json.load(fp)
all_gradenames = [str(item).lower() for item in all_gradenames]  

with open(file=f"./outOfScopeData_{data_version}.json") as f:
    OOS_Data = json.load(f)

# Application and its respective Industry are reviewed by the Business Team
app_industry_df = pd.read_excel("applications_industry_mapping_gd_reviewed.xlsx") # Index Data
app_industry_df2 = pd.read_excel("applications_industry_mapping_sq_21_11_24.xlsx") # Survey Data


am_region_df = pd.read_excel("Region_data_04_09_24.xlsx", sheet_name='americas')
emea_region_df = pd.read_excel("Region_data_04_09_24.xlsx", sheet_name='europe middle east africa')
ap_region_df = pd.read_excel("Region_data_04_09_24.xlsx", sheet_name='asia pacific')

with open(file="./ul_list_name_value.json") as f:
    ul_list_name_value = json.load(f)

random.shuffle(all_brands)
random.shuffle(all_cgrades)
random.shuffle(all_gradenames)

In [4]:
am_region_df = am_region_df[['Country', 'State', 'City', 'Continent']]
emea_region_df = emea_region_df[['Country', 'State', 'City', 'Continent']]
ap_region_df = ap_region_df[['Country', 'State', 'City', 'Continent']]

In [5]:
unique_values.keys()

dict_keys(['Property', 'Feature', 'UL', 'Certification', 'Auto_Approval', 'Brand', 'Polymer', 'Filler', 'Market', 'Industry_Group'])

In [6]:
len(OOS_Data['grades']), len(all_gradenames)

(13980, 4228)

### Validating ul_list_name_value

In [7]:
ul_list_name_value

{'arc resistance': {'synonyms': ['arc resistance',
   'hvar',
   'arc resistivity',
   'arc resist',
   'arc resistance',
   'high voltage arc resist to ignition',
   'high voltage arc resistant to ignition',
   'high voltage arc resistance to ignition',
   'high voltage arc resistivity to ignition',
   'high voltage arc resist ignition',
   'high voltage arc resistant ignition',
   'high voltage arc resistance ignition',
   'high voltage arc resistivity ignition',
   'high voltage arc',
   'arc resistant',
   'ar',
   'arc r',
   'arcres',
   'arc res'],
  'values': ['plc 0',
   'plc 1',
   'plc 2',
   'plc 3',
   'plc 4',
   'plc 5',
   'plc 6',
   'plc 7',
   'plc0',
   'plc1',
   'plc2',
   'plc3',
   'plc4',
   'plc5',
   'plc6',
   'plc7',
   'plc-0',
   'plc-1',
   'plc-2',
   'plc-3',
   'plc-4',
   'plc-5',
   'plc-6',
   'plc-7'],
  'has_thickness': False,
  'value_priority': False},
 'ball pressure test': {'synonyms': ['ball pressure test',
   'resist to heat',
   'resis to 

In [8]:
ul_list_name_value['ball pressure test']['synonyms']

['ball pressure test',
 'resist to heat',
 'resis to heat',
 'ball pressure',
 'resistance to heat',
 'resistant to heat',
 'bpt',
 'ball test']

In [9]:
ul_list_name_value['ball pressure test']['synonyms'] = [i for i in ul_list_name_value['ball pressure test']['synonyms'] if i not in ['resis to heat', 'resist to heat', 'resistant to heat', 'resistance to heat']]

In [10]:
ul_list_name_value['ball pressure test']['synonyms']

['ball pressure test', 'ball pressure', 'bpt', 'ball test']

In [11]:
list(ul_list_name_value)

['arc resistance',
 'ball pressure test',
 'Charpy Unnotched Impact Strength',
 'comparative tracking index (cti)',
 'dielectric strength',
 'dimensional change',
 'flame rating',
 'flammability classification',
 'flexural stress',
 'glow wire flammability index',
 'glow wire ignition temperature',
 'glow wire',
 'has linked tds',
 'high voltage arc tracking rate (hvtr)',
 'high-current arc ignition (hai)',
 'hot wire ignition (hwi)',
 'inclined-plane tracking',
 'mechanically recycled content',
 'non-halogenated material',
 'outdoor suitability',
 'detergent resistance',
 'relative thermal index - electrical (rti elec) (°c)',
 'relative thermal index - mechanical impact (rti imp) (°c)',
 'relative thermal index - mechanical strength (rti str) (°c)',
 'relative thermal index',
 'rohs 2011/65/eu material',
 'surface resistivity',
 'tensile impact strength']

In [12]:
del ul_list_name_value['Charpy Unnotched Impact Strength']
del ul_list_name_value['flexural stress']
del ul_list_name_value['surface resistivity']
del ul_list_name_value['mechanically recycled content']
del ul_list_name_value['has linked tds']

### Connect to Database

In [13]:
connection_string = eval(os.getenv('connection_string'))
snowflake_connection_string = connection_string['ml-gst-dev-usscc-01']

parts = snowflake_connection_string.split("//")[1].split("/")  
account = ".".join(parts[0].split('.')[:2]) 
user = parts[1].split('user=')[1].split('&')[0] 
password = parts[1].split('password=')[1].split('&')[0] 
database = parts[1].split('db=')[1].split('&')[0]
warehouse = parts[1].split('warehouse=')[1].split('&')[0]  
role = parts[1].split('role=')[1]

# connect to dev for latest version of the data
engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    database = 'ANALYTICS_DEV', #database, 
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur = engine.connect()



def read_data_from_snowflake_table(cur,query):
    df = pd.read_sql(query, cur)
    return df

### Synonym Data

#### Reading the Data

In [14]:
synonym_df = read_data_from_snowflake_table(cur,"""select * from SYNONYM""")
synonym_df.columns = [x.upper() if x.islower() else x for x in synonym_df.columns]
synonym_df = synonym_df.apply(lambda x: x.str.lower())

synonym_df2 = copy.deepcopy(synonym_df) 
synonym_df2["SYNONYMS"] = synonym_df["SYNONYMS"].str.split(";")
synonym_df2 = synonym_df2.explode("SYNONYMS")
synonym_df2['SYNONYMS'] = synonym_df2['SYNONYMS'].apply(lambda x: re.sub("(\s+)", " ", x.strip().lower()) if x else x)
synonym_df2['DEFINED_NAME'] = synonym_df2['DEFINED_NAME'].apply(lambda x: re.sub("(\s+)", " ", x.strip().lower()) if x else x)
synonym_df2 = synonym_df2.reset_index(drop=True)
synonym_df2 = synonym_df2[~synonym_df2.duplicated()]
synonym_df2

,TYPE,DEFINED_NAME,SYNONYMS
0,auto cert,mercedes-benz,mercedes-benz
1,auto cert,mercedes-benz,daimler
2,auto cert,mercedes-benz,daimler-benz
3,auto cert,mercedes-benz,daimler mercedes
4,auto cert,mercedes-benz,daimler mercedes-benz
...,...,...,...
1551,ul property,relative thermal index - mechanical strength (...,relative thermal index - mechanical strength
1552,ul property,relative thermal index - mechanical strength (...,rti str
1553,ul property,rohs 2011/65/eu material,rohs 2011/65/eu material
1554,ul property,surface resistivity,None


#### Remove Synonyms

- Incorrect
- Confusing/Ambiguity

In [15]:
remove_syns = []

# remove grades from the synonyms
for g in all_gradenames:
    for s in list(synonym_df2['SYNONYMS']):
        if s and g.lower() == s.lower():
            if s not in remove_syns:
                remove_syns.append(s)

In [16]:
remove_syns

['hhr', 'eco-b', 'htr', 'eco-r']

In [17]:
set(synonym_df2[synonym_df2['SYNONYMS'].duplicated()]['SYNONYMS'])

{'',
 None,
 'heat resistant',
 'im',
 'injection mouldable',
 'thermal conductivity',
 'tm'}

In [18]:
remove_syns += [
    '',
    None,
    'im',
    'eco',
    'injection mouldable',
    'mold shrinkage (md/td)',
    'thermal conductivity',
    'tm',
    'lt',
    'min',
    # 'steel fiber',
    'sf',
    # 'stainless steel fiber',
    # 'ss fiber',
    # 'stainless steel',
    'ss',
    'sr',
    'tp',
    'at',
    'chemical resistance', # appearing in others
    'mi', # 'apprearing in property'
    'heat resistant', # appearing in property
    'impa', # mapped to wrong meaning in the table # ignore
    'impact strength', # it can multiple properties
    'polyester', # it is a poylmer family not 'pbt'
    'htr', # it is a grade
    'hygroscopic', # it is opposite of water resistant
    'ny', # it is a region
    # 'max service temperature',
    # 'min service temperature',
    'metalx', # partial grade name

    'charpy', # charpy not charpy impact strength
    'flammability', 
    'impact', 
    'notched impact', 
    'shore', 
    'shore hardness', 
    'strain at break', 
    'strength', 
    'stress at 100% elongation', 
    'stress at break', 
    'temperature', 
    'tensile', 
    'tensile strength', 
    'tensile stress at 100% elongation', 
    'tensile stress at break', 
    'viscosity',
    'izod',
    'hr',
    'reduce carbon capture, reduced carbon footprint',
    'carbon footprint', 
    'iso 14067', 
    'carbon capture and utilization',
    'ccu',
    'solar resistant',
    'solar resistance',

]
remove_syns

['hhr',
 'eco-b',
 'htr',
 'eco-r',
 '',
 None,
 'im',
 'eco',
 'injection mouldable',
 'mold shrinkage (md/td)',
 'thermal conductivity',
 'tm',
 'lt',
 'min',
 'sf',
 'ss',
 'sr',
 'tp',
 'at',
 'chemical resistance',
 'mi',
 'heat resistant',
 'impa',
 'impact strength',
 'polyester',
 'htr',
 'hygroscopic',
 'ny',
 'metalx',
 'charpy',
 'flammability',
 'impact',
 'notched impact',
 'shore',
 'shore hardness',
 'strain at break',
 'strength',
 'stress at 100% elongation',
 'stress at break',
 'temperature',
 'tensile',
 'tensile strength',
 'tensile stress at 100% elongation',
 'tensile stress at break',
 'viscosity',
 'izod',
 'hr',
 'reduce carbon capture, reduced carbon footprint',
 'carbon footprint',
 'iso 14067',
 'carbon capture and utilization',
 'ccu',
 'solar resistant',
 'solar resistance']

In [19]:
synonym_df2 = synonym_df2[~synonym_df2['SYNONYMS'].isin(remove_syns)]


#### Correct Synonyms

In [20]:
correct_syns = [
    ['property', 'molding shrinkage', 'mold shrinkage (md/td)'],
    ['processing', 'injection molding', 'injection mouldable'],
    ['feature', 'heat stabilized', 'heat resistant'],
    ['feature', 'impact modified', 'im'],
    ['feature', 'chemical resistant', 'chemical resistance'],
    ['property', 'melt mass-flow rate iso 1133 (g/10min)', 'mi'],
    ['feature', 'low wear / low friction', 'friction'],
    ['processing', 'fiber spinning / gel spinning', 'gel extrusion'],
    ['feature', 'laser weldable', 'lt'],
    ['property', 'surface resistivity iec 62631-3-2 (ohm)', 'sr'],
    ['delivery form', 'tape', 'tp'],
    ['polymer', 'pa6t', 'pa6t'],
    ['filler', 'metal', 'sf'],
    ['filler', 'metal', 'ss'],
    ['feature', 'sustainable', 'eco'],
    ['auto cert', 'mercedes-benz', 'mercedes-benz group (daimier)'],
    ['filler', 'cfr-gf', 'cfr gf'],    
    ['filler', 'cfr-cf', 'cfr cf'], 
    ['filler', 'cfr-gf', 'cfr/gf'],    
    ['filler', 'cfr-cf', 'cfr/cf'], 
    ['delivery form', 'micropowder', 'micro powder'],  
    ['delivery form', 'micropowder', 'micro-powder'],
    ['feature', 'hr', 'hr'],
    ['feature', 'low wear / low friction', 'tf'],
    ['polymer', 'polyether', 'polyether'],
    ['filler', 'carbon', 'carbon'], 
    ['feature', 'carbon capture', 'carbon footprint'],
    ['feature', 'carbon capture', 'iso 14067'],
    ['feature', 'carbon capture', 'carbon capture and utilization'],
    ['feature', 'carbon capture', 'ccu'],
    ['feature', 'carbon capture', 'reduce carbon capture'],
    ['feature', 'carbon capture', 'reduced carbon footprint'],
    ['feature', 'u.v. stabilized or stable to weather', 'solar resistance'],

]

synonym_df2 = pd.concat([synonym_df2, pd.DataFrame(correct_syns, columns=synonym_df2.columns)], ignore_index=True)

In [21]:
synonym_df2[synonym_df2['SYNONYMS'].duplicated()]

,TYPE,DEFINED_NAME,SYNONYMS
1507,feature,low wear / low friction,tf
1508,polymer,polyether,polyether
1514,feature,carbon capture,reduce carbon capture
1515,feature,carbon capture,reduced carbon footprint


In [22]:
synonym_df2[synonym_df2['TYPE']=='property'][['DEFINED_NAME', 'SYNONYMS']]

,DEFINED_NAME,SYNONYMS
881,average molecular weight margolies' equation (...,average molecular weight
882,average molecular weight margolies' equation (...,average molar mass
883,average molecular weight margolies' equation (...,average molecular mass
884,average molecular weight margolies' equation (...,avermw
885,average molecular weight margolies' equation (...,molecular weight
...,...,...
1432,water absorption sim. to iso 62 2mm (%),moisture absorption equilibrium 23°c/50% r.h.
1433,wear by sandslurry method (based on gur 4120=100),wear by sandslurry method
1484,molding shrinkage,mold shrinkage (md/td)
1489,melt mass-flow rate iso 1133 (g/10min),mi


### Ignore Terms

Add these terms to the query but not to the output

In [23]:
ignore_terms = [
    'pb',
    'non sustainable',
    'barnacle resistance',
    'crystallinity between 10% and 20%',
    'medical & pharma - drug delivery devices',
    'low ion elution',
    'insect resistant',
    'sardine flavor',
    'steam sterilization',
    'oxidation',
    'heat aging',
    'low ion elution',
    'asa',
    'epdm',
    'pvdf',
    'pvc',
    'robotic extrusion',
    # 'overmolding',
    'pellet size of 40 pellets/g',
    'stress-strain curve',
    'shear of 453.9/mpa',
    'crush test > 20 henhouses-furlong/fortnight',
    'stress-strain curve',
    'additive',
    'additives',
    'adhesion',
    'compressive / bending',
    'crosslinked',
    'food',
    'food contact',
    'molding conditions',
    'nanotube',
    'nanotubes',
    'silicone replacement',
    'stabilisation for copper contact',
    'amorphous',
    'compatibility',
    'long term heat aging',
    'fda',
    'cfr 21',
    'drug master file',
    'device master file',
    'usp',
    'iso 10993',
    'food grade',
    # 'carbon footprint',
    # 'iso 14067',
    # 'carbon capture and utilization',
    # 'carbon capture',
    # 'carbon utilization',
    # 'ccu',
    'bonding',
    'bondable',
    'ltha',
    'dmf',
    'maf',
    'qmtt2 rated',
    'qmtt2 rating',
    'qmtt2 listing',
    'qmtt2 listed',
    'qmtt2',

    # from actual data
    'crush test',
    'stress-strain',
    'pellet size',
    'shear',
    'fmvss',
    
    # model error
    'food contact compliant',
    'fda compliant',

    # prod queries
    'process guide',
    'E140692',
    'CONCENTRATE BLEND DOWN',
    'damping behavior',

    # weightage
    'non sustainable',
    'non sustainable',
    'non sustainable',
]

In [24]:
ignore_terms = [x.strip().lower() for x in ignore_terms]

In [25]:
temp = list(synonym_df2[synonym_df2['TYPE']=='others']['DEFINED_NAME']) + list(synonym_df2[synonym_df2['TYPE']=='others']['SYNONYMS'])
ignore_terms += [x for x in list(set(temp)) if x not in ignore_terms]
ignore_terms = [x for x in ignore_terms if x not in ['toyota']]
ignore_terms

['pb',
 'non sustainable',
 'barnacle resistance',
 'crystallinity between 10% and 20%',
 'medical & pharma - drug delivery devices',
 'low ion elution',
 'insect resistant',
 'sardine flavor',
 'steam sterilization',
 'oxidation',
 'heat aging',
 'low ion elution',
 'asa',
 'epdm',
 'pvdf',
 'pvc',
 'robotic extrusion',
 'pellet size of 40 pellets/g',
 'stress-strain curve',
 'shear of 453.9/mpa',
 'crush test > 20 henhouses-furlong/fortnight',
 'stress-strain curve',
 'additive',
 'additives',
 'adhesion',
 'compressive / bending',
 'crosslinked',
 'food',
 'food contact',
 'molding conditions',
 'nanotube',
 'nanotubes',
 'silicone replacement',
 'stabilisation for copper contact',
 'amorphous',
 'compatibility',
 'long term heat aging',
 'fda',
 'cfr 21',
 'drug master file',
 'device master file',
 'usp',
 'iso 10993',
 'food grade',
 'bonding',
 'bondable',
 'ltha',
 'dmf',
 'maf',
 'qmtt2 rated',
 'qmtt2 rating',
 'qmtt2 listing',
 'qmtt2 listed',
 'qmtt2',
 'crush test',
 'stre

### Check the Defined Names (Synonym mapped to)

In [26]:
set(synonym_df2[synonym_df2['TYPE']=='feature']['DEFINED_NAME'])

{'anti-static',
 'bio-content',
 'carbon capture',
 'chemical resistant',
 'flame retardant',
 'heat stabilized',
 'high flow',
 'high gloss',
 'high viscosity',
 'hr',
 'hydrolysis resistant',
 'hydrophilic',
 'impact modified',
 'improved creep',
 'improved weld line',
 'increased electrical conductivity',
 'increased thermal conductivity',
 'laser direct structurable',
 'laser markable',
 'laser weldable',
 'lead-free soldering resistant',
 'light stabilized',
 'light weight',
 'low emissions',
 'low halide content',
 'low warpage',
 'low wear / low friction',
 'lubricants',
 'medical/healthcare',
 'non-halogenated/red phosphorous free flame retardant',
 'nucleated',
 'plasticizer',
 'platable',
 'recycled content',
 'reduced gloss',
 'release agent',
 'specialty appearance',
 'static dissipative',
 'sustainable',
 'thermal shock resistant',
 'u.v. stabilized',
 'u.v. stabilized or stable to weather',
 'ultrasonic weldable'}

In [27]:
set(synonym_df2[synonym_df2['TYPE']=='filler']['DEFINED_NAME'])

{'aramide fiber',
 'carbon',
 'carbon fiber',
 'carbon powder',
 'cfr-cf',
 'cfr-gf',
 'continuous carbon fiber',
 'continuous glass fiber',
 'glass beads',
 'glass fiber',
 'glass flake',
 'long aramide fiber',
 'long carbon fiber',
 'long glass fiber',
 'long metal fiber',
 'metal',
 'mineral',
 'unfilled',
 'whisker'}

In [28]:
for i in set(synonym_df2[synonym_df2['TYPE']=='auto cert']['DEFINED_NAME']):
    if i not in unique_values['Auto_Approval']:
        print(i)

vw group
mercedes-benz


### Filler

In [29]:
unique_values['Filler'] = [x.lower() for x in unique_values['Filler']]
unique_values['Filler']

['total load',
 'glass fiber',
 'glass beads',
 'unfilled',
 'mineral',
 'carbon fiber',
 'metal',
 'long glass fiber',
 'long carbon fiber',
 'long aramid fiber',
 'long metal fiber',
 'carbon powder',
 'aramid fiber']

In [30]:
filler_syns = synonym_df2[synonym_df2['TYPE']=='filler'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
filler_syns

{'aramide fiber': ['aramide fiber',
  'aramide fibre',
  'aramid fiber',
  'aramid fibre',
  'af',
  'aramide reinforced',
  'aramid reinforced'],
 'carbon': ['carbon'],
 'carbon fiber': ['carbon fiber',
  'carbon fibre',
  'cf',
  'carbon filled',
  'carbon reinforced'],
 'carbon powder': ['carbon powder', 'cd'],
 'cfr-cf': ['cfr cf', 'cfr/cf'],
 'cfr-gf': ['cfr gf', 'cfr/gf'],
 'continuous carbon fiber': ['continuous carbon fiber', 'cfr-cf'],
 'continuous glass fiber': ['continuous glass fiber', 'cfr-gf', 'e-glass'],
 'glass beads': ['glass beads', 'gb', 'glass sphere', 'glass balls'],
 'glass fiber': ['glass fiber',
  'glass',
  'glass filled',
  'fiber',
  'fibre',
  'gf',
  'glass reinforced',
  'reinforced',
  'gr',
  'glas fiber',
  'glass fibre',
  'glas fibre',
  'short fiber',
  'short fibre',
  'g',
  'glass reinforcement',
  'glass content',
  'fiberglass',
  'fibreglass',
  'fiber glass',
  'fibre glass'],
 'glass flake': ['glass flake', 'flake', 'gs'],
 'long aramide fibe

In [31]:
for f in filler_syns:
    abbs = []
    for abb in filler_syns[f]:
        if len(abb)<4:
            print(abb, "-", f)
            abbs.append(abb)
    filler_syns[f].extend(abbs+abbs+abbs) # adding weightage to the abrreviations


filler_syns

af - aramide fiber
cf - carbon fiber
cd - carbon powder
gb - glass beads
gf - glass fiber
gr - glass fiber
g - glass fiber
gs - glass flake
laf - long aramide fiber
lcf - long carbon fiber
lgf - long glass fiber
cft - long glass fiber
sf - metal
ss - metal
td - mineral
md - mineral
mf - mineral


{'aramide fiber': ['aramide fiber',
  'aramide fibre',
  'aramid fiber',
  'aramid fibre',
  'af',
  'aramide reinforced',
  'aramid reinforced',
  'af',
  'af',
  'af'],
 'carbon': ['carbon'],
 'carbon fiber': ['carbon fiber',
  'carbon fibre',
  'cf',
  'carbon filled',
  'carbon reinforced',
  'cf',
  'cf',
  'cf'],
 'carbon powder': ['carbon powder', 'cd', 'cd', 'cd', 'cd'],
 'cfr-cf': ['cfr cf', 'cfr/cf'],
 'cfr-gf': ['cfr gf', 'cfr/gf'],
 'continuous carbon fiber': ['continuous carbon fiber', 'cfr-cf'],
 'continuous glass fiber': ['continuous glass fiber', 'cfr-gf', 'e-glass'],
 'glass beads': ['glass beads',
  'gb',
  'glass sphere',
  'glass balls',
  'gb',
  'gb',
  'gb'],
 'glass fiber': ['glass fiber',
  'glass',
  'glass filled',
  'fiber',
  'fibre',
  'gf',
  'glass reinforced',
  'reinforced',
  'gr',
  'glas fiber',
  'glass fibre',
  'glas fibre',
  'short fiber',
  'short fibre',
  'g',
  'glass reinforcement',
  'glass content',
  'fiberglass',
  'fibreglass',
  'fib

In [32]:
OOS_Data['fillers']

['whisker', 'natural organic fiber']

In [33]:
if 'glass flake' not in filler_syns:
    filler_syns['glass flake'] = ['glass flake', 'flake', 'gs']
    
filler_syns['natural organic fiber'] = ['natural organic fiber', 'organic fiber', 'natural fiber']

In [34]:
def get_fp_range():
    diff = random.randint(5, 30)
    fp = random.randint(30, 70)    
    return fp, diff, fp-diff, fp+diff

In [35]:
get_fp_range()

(69, 18, 51, 87)

In [36]:
unfilled_syns = [
    "un filled", "un-filled", "unfilled", "not filled", "no filler", "non-filler", "non filler",
    "non filled", "no fill", "without fill", "without filler", "without filled", "no load"
]
for syn in filler_syns['unfilled']:
    if syn not in unfilled_syns:
       unfilled_syns.append(syn)

unfilled_syns

['un filled',
 'un-filled',
 'unfilled',
 'not filled',
 'no filler',
 'non-filler',
 'non filler',
 'non filled',
 'no fill',
 'without fill',
 'without filler',
 'without filled',
 'no load',
 'unreinforced',
 'without glass fiber',
 'without glass',
 'without glass fibre']

In [37]:
del filler_syns['unfilled']

In [38]:
filler_syn_mapping = {}
for f_meaning, f_syns in filler_syns.items():
    for syn in f_syns:
        if len(syn) > 1:
            filler_syn_mapping[syn] = f_meaning
        else:
            print(f"Add examples manually for {f_meaning} synonym '{syn}'", "\n\n")

filler_syn_mapping

Add examples manually for glass fiber synonym 'g' 


Add examples manually for glass fiber synonym 'g' 


Add examples manually for glass fiber synonym 'g' 


Add examples manually for glass fiber synonym 'g' 




{'aramide fiber': 'aramide fiber',
 'aramide fibre': 'aramide fiber',
 'aramid fiber': 'aramide fiber',
 'aramid fibre': 'aramide fiber',
 'af': 'aramide fiber',
 'aramide reinforced': 'aramide fiber',
 'aramid reinforced': 'aramide fiber',
 'carbon': 'carbon',
 'carbon fiber': 'carbon fiber',
 'carbon fibre': 'carbon fiber',
 'cf': 'carbon fiber',
 'carbon filled': 'carbon fiber',
 'carbon reinforced': 'carbon fiber',
 'carbon powder': 'carbon powder',
 'cd': 'carbon powder',
 'cfr cf': 'cfr-cf',
 'cfr/cf': 'cfr-cf',
 'cfr gf': 'cfr-gf',
 'cfr/gf': 'cfr-gf',
 'continuous carbon fiber': 'continuous carbon fiber',
 'cfr-cf': 'continuous carbon fiber',
 'continuous glass fiber': 'continuous glass fiber',
 'cfr-gf': 'continuous glass fiber',
 'e-glass': 'continuous glass fiber',
 'glass beads': 'glass beads',
 'gb': 'glass beads',
 'glass sphere': 'glass beads',
 'glass balls': 'glass beads',
 'glass fiber': 'glass fiber',
 'glass': 'glass fiber',
 'glass filled': 'glass fiber',
 'fiber':

In [39]:
filler_abbr = []
for i in list(filler_syn_mapping) + unique_values['Filler'] + OOS_Data['fillers']:
    if len(i) < 4 and i!='min':
        filler_abbr.append(i)

filler_abbr

['af',
 'cf',
 'cd',
 'gb',
 'gf',
 'gr',
 'gs',
 'laf',
 'lcf',
 'lgf',
 'cft',
 'sf',
 'ss',
 'td',
 'md',
 'mf']

In [40]:
len(list(filler_syn_mapping) + unique_values['Filler'] + OOS_Data['fillers'] + filler_abbr*5)

191

In [41]:
def get_random_filler():
    is_filler_range = False
    fp1 = random.randint(5, 50)
    fp2 = random.randint(5, 50)
    fp = fp1 + fp2

    all_fillers = unique_values['Filler'] + OOS_Data['fillers']
    fillers = [x.lower() for x in all_fillers if x not in ['Total load', 'total load']]
    filler_syns_list = list(filler_syn_mapping)

    filler_name = random.choice(fillers + filler_syns_list + filler_abbr*5 + ['unfilled']*19 + ['carbon']*4)
    if filler_name == 'unfilled':
        # print("unfilled")
        filler_details =  [{'filler_name': [filler_name]}, {'total_load': {'value': None, 'min': None, 'max': None}}]
        query = unfilled_syns 
    else:       
        filler_name2 = random.choice(fillers)
        
        if filler_name in filler_syns_list:
            syn_meaning1 = filler_syn_mapping[filler_name]
        else:
            syn_meaning1 = filler_name
                    
        if filler_name2 in filler_syns_list:
            syn_meaning2 = filler_syn_mapping[filler_name2]
        else:
            syn_meaning2 = filler_name2

        while (filler_name2 == 'unfilled') or (filler_name == filler_name2) or (syn_meaning1 == syn_meaning2) or (syn_meaning1 in syn_meaning2) or (syn_meaning2 in syn_meaning1):
            filler_name2 = random.choice(fillers)
            if filler_name2 in filler_syns_list:
                syn_meaning2 = filler_syn_mapping[filler_name2]
            else:
                syn_meaning2 = filler_name2

        fp_new, diff, fp_min, fp_max = get_fp_range()
        if len(filler_name) < 4 or len(filler_name2) < 4:
            # print("Abbreviations Scenario")
            pass

        filler_details_templates = [
            [{'filler_name': ['all']}, {'total_load': {'value': fp, 'min': fp-5, 'max': fp+5}}],
            
            [{'filler_name': [syn_meaning1]}, {'total_load': {'value': fp, 'min': fp-5, 'max': fp+5}}],
            [{'filler_name': [syn_meaning1]}, {'total_load': {'value': None, 'min': None, 'max': None}}],
            
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': fp, 'min': fp-5, 'max': fp+5}}],
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': None, 'min': None, 'max': None}}],

            [{'filler_name': ['all']}, {'total_load': {'value': fp, 'min': fp, 'max': None}}],
            [{'filler_name': ['all']}, {'total_load': {'value': fp, 'min': None, 'max': fp}}],
            [{'filler_name': [syn_meaning1]}, {'total_load': {'value': fp, 'min': fp, 'max': None}}],
            [{'filler_name': [syn_meaning1]}, {'total_load': {'value': fp, 'min': None, 'max': fp}}],
            
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': fp, 'min': fp, 'max': None}}],
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': fp, 'min': None, 'max': fp}}],

            [{'filler_name': ['all']}, {'total_load': {'value': fp_new, 'min': fp_min, 'max': fp_max}, "range": True}],
            [{'filler_name': [syn_meaning1]}, {'total_load': {'value': fp_new, 'min': fp_min, 'max': fp_max}, "range": True}],
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': fp_new, 'min': fp_min, 'max': fp_max}, "range": True}],


            # weightage - repeat template
            [{'filler_name': [syn_meaning1]}, {'total_load': {'value': None, 'min': None, 'max': None}}],
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': fp, 'min': fp-5, 'max': fp+5}}],
            [{'filler_name': [syn_meaning1, syn_meaning2]}, {'total_load': {'value': None, 'min': None, 'max': None}}],
        ]
        
        filler_details = random.choice(filler_details_templates)
                
    if filler_name != 'unfilled':
        tl = random.choice(["total load", "combined load", "full load", "total fill", "combined fill", "total filler load", "total content", "load", "filler load"])
        filler_name = random.choice([filler_name, filler_name, filler_name, filler_name, filler_name, filler_name, filler_name.replace(" ", "")])
        if filler_details[0]['filler_name'] == ['all']:
            if "range" in filler_details[1]:
                del filler_details[1]["range"]
                is_filler_range = True
                query_range_without_value = [              
                    f"{tl} of {fp_min} - {fp_max}",
                    f"{tl} {fp_min} - {fp_max}%",
                    f"{tl} of {fp_min}% - {fp_max}%",
                    f"{tl} {fp_min}% to {fp_max}%",
                    f"{tl} ranging between {fp_min}% - {fp_max}%",
                    f"{tl} in the range of {fp_min} - {fp_max}%",
                    f"{tl} range {fp_min} to {fp_max}%",

                    # weightage
                    f"{tl} of {fp_min} - {fp_max}",
                    f"{tl} {fp_min} - {fp_max}",
                    f"{tl} {fp_min} to {fp_max}",
                    f"{tl} ranging between {fp_min} - {fp_max}",
                    f"{tl} in the range of {fp_min} - {fp_max}",
                    f"{tl} range {fp_min} to {fp_max}",
                ]
                query = [
                    f"{tl} {fp_new} with range {diff}%",
                    f"{tl} of {fp_new} with range {diff}%",
                    f"{tl} {fp_new} with {diff}% range",
                    f"{tl} of {fp_new} with {diff}% range",

                    f"{tl} of {fp_new} with +/-{diff}% range",
                    f"{tl} of {fp_new} with range +/-{diff}%",
                    f"{tl} of {fp_new} with +/-{diff}",
                    f"{tl} of {fp_new} with +/-{diff}%",
                    
                    f"{tl} {fp_new} with range {diff}",
                    f"{tl} of {fp_new} with range {diff}",
                    f"{tl} {fp_new} with {diff} range",
                    f"{tl} of {fp_new} with {diff} range",

                    f"{tl} of {fp_new} with +/-{diff} range",
                    f"{tl} of {fp_new} with range +/-{diff}",
                    f"{tl} of {fp_new} with +/-{diff}",
                ]
            else:
                if not filler_details[1]['total_load']['min']:
                    query = [
                        f"{tl} max {fp}",
                        f"{tl} upto {fp}",
                        f"{tl} max {fp}%",
                        f"{tl} upto {fp}%",
                        f"{tl} less than {fp}",
                        f"{tl} less than {fp}%",
                        f"{tl} <{fp}",
                        f"{tl} <{fp}%",
                        
                        # weightage  
                        f"{tl} max {fp}",
                        f"{tl} upto {fp}",
                        f"{tl} less than {fp}",
                        f"{tl} <{fp}",
                    ]
                elif not filler_details[1]['total_load']['max']:
                    query = [
                        f"{tl} min {fp}",
                        f"{tl} at least {fp}",
                        f"{tl} min {fp}%",
                        f"{tl} at least {fp}%",
                        f"{tl} greater than {fp}",
                        f"{tl} greater than {fp}%",
                        f"{tl} >{fp}",
                        f"{tl} >{fp}%",
                        
                        # weightage 
                        f"{tl} min {fp}",
                        f"{tl} at least {fp}",
                        f"{tl} greater than {fp}",
                        f"{tl} >{fp}",
                    ]
                else:
                    query = [
                        f"{tl} {fp}",
                        f"{tl} {fp}%",
                        f"{fp} {tl}",
                        f"{fp}% {tl}",
                        f"{tl} of {fp}%",
                        f"{tl} of {fp}",

                        # weightage
                        f"{tl} {fp}",
                        f"{fp} {tl}",
                        f"{tl} of {fp}",
                    ]
                        
        elif len(filler_details[0]['filler_name']) == 2:
            filler_name2 = random.choice([filler_name2, filler_name2, filler_name2, filler_name2, filler_name2, filler_name2, filler_name2.replace(" ", "")])
            if "range" in filler_details[1]:
                del filler_details[1]["range"]
                is_filler_range = True
                query_range_without_value = [
                    f"{filler_name} {filler_name2} {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {fp_min} - {fp_max}%",
                    f"{filler_name} {filler_name2} {fp_min}% - {fp_max}%",
                    f"{filler_name} {filler_name2} {fp_min}% to {fp_max}%",
                    f"{filler_name} {filler_name2} ranging between {fp_min}% - {fp_max}%",
                    f"{filler_name} {filler_name2} with range of {fp_min} - {fp_max}%",
                    f"{filler_name} {filler_name2} range {fp_min} to {fp_max}%",

                    f"{filler_name} and {filler_name2} {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {fp_min} - {fp_max}%",
                    f"{filler_name} and {filler_name2} {fp_min}% - {fp_max}%",
                    f"{filler_name} and {filler_name2} {fp_min}% to {fp_max}%",
                    f"{filler_name} and {filler_name2} ranging between {fp_min}% - {fp_max}%",
                    f"{filler_name} and {filler_name2} with range of {fp_min} - {fp_max}%",
                    f"{filler_name} and {filler_name2} range {fp_min} to {fp_max}%",

                    f"{filler_name} {filler_name2} {tl} of {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {tl} {fp_min} - {fp_max}%",
                    f"{filler_name} {filler_name2} {tl} of {fp_min}% - {fp_max}%",
                    f"{filler_name} {filler_name2} {tl} {fp_min}% to {fp_max}%",
                    f"{filler_name} {filler_name2} {tl} ranging between {fp_min}% - {fp_max}%",
                    f"{filler_name} {filler_name2} {tl} with range of {fp_min} - {fp_max}%",
                    f"{filler_name} {filler_name2} {tl} range {fp_min} to {fp_max}%",

                    f"{filler_name} and {filler_name2} {tl} of {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {tl} {fp_min} - {fp_max}%",
                    f"{filler_name} and {filler_name2} {tl} of {fp_min}% - {fp_max}%",
                    f"{filler_name} and {filler_name2} {tl} {fp_min}% to {fp_max}%",
                    f"{filler_name} and {filler_name2} {tl} ranging between {fp_min}% - {fp_max}%",
                    f"{filler_name} and {filler_name2} {tl} with range of {fp_min} - {fp_max}%",
                    f"{filler_name} and {filler_name2} {tl} range {fp_min} to {fp_max}%",    

                    # weightage
                    f"{filler_name} {filler_name2} {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {fp_min} to {fp_max}",
                    f"{filler_name} {filler_name2} ranging between {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} with range of {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} range {fp_min} to {fp_max}",

                    f"{filler_name} and {filler_name2} {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {fp_min} to {fp_max}",
                    f"{filler_name} and {filler_name2} ranging between {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} with range of {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} range {fp_min} to {fp_max}",

                    f"{filler_name} {filler_name2} {tl} of {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {tl} {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {tl} {fp_min} to {fp_max}",
                    f"{filler_name} {filler_name2} {tl} ranging between {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {tl} with range of {fp_min} - {fp_max}",
                    f"{filler_name} {filler_name2} {tl} range {fp_min} to {fp_max}",

                    f"{filler_name} and {filler_name2} {tl} of {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {tl} {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {tl} {fp_min} to {fp_max}",
                    f"{filler_name} and {filler_name2} {tl} ranging between {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {tl} with range of {fp_min} - {fp_max}",
                    f"{filler_name} and {filler_name2} {tl} range {fp_min} to {fp_max}",
                ]
                
                query = [
                    f"{filler_name} {filler_name2} {fp_new} with {diff}% range",
                    f"{filler_name} {filler_name2} {fp_new} with range {diff}%",
                    f"{filler_name} and {filler_name2} {fp_new} with {diff}% range",
                    f"{filler_name} and {filler_name2} {fp_new} with range {diff}%",
                    f"{filler_name} {filler_name2} {fp_new}% with {diff}% range",
                    f"{filler_name} {filler_name2} {fp_new}% with range {diff}%",
                    f"{filler_name} and {filler_name2} {fp_new}% with {diff}% range",
                    f"{filler_name} and {filler_name2} {fp_new}% with range {diff}%",

                    f"{filler_name} {filler_name2} {fp_new} with +/-{diff}%",
                    f"{filler_name} {filler_name2} {fp_new} with +/-{diff}% range",
                    f"{filler_name} and {filler_name2} {fp_new} with +/-{diff}%",
                    f"{filler_name} and {filler_name2} {fp_new} with +/-{diff}% range",
                    f"{filler_name} {filler_name2} {fp_new}% with +/-{diff}%",
                    f"{filler_name} {filler_name2} {fp_new}% with +/-{diff}% range",
                    f"{filler_name} and {filler_name2} {fp_new}% with +/-{diff}%",
                    f"{filler_name} and {filler_name2} {fp_new}% with +/-{diff}% range",
                    
                    f"{filler_name} {filler_name2} {tl} of {fp_new} with {diff}% range",
                    f"{filler_name} {filler_name2} {tl} of {fp_new} with range {diff}%",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with {diff}% range",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with range {diff}%",

                    f"{filler_name} {filler_name2} {tl} of {fp_new} with +/-{diff}%",
                    f"{filler_name} {filler_name2} {tl} of {fp_new} with +/-{diff}% range",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with +/-{diff}%",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with +/-{diff}% range",

                    f"{filler_name} {filler_name2} {fp_new} with {diff} range",
                    f"{filler_name} {filler_name2} {fp_new} with range {diff}",
                    f"{filler_name} and {filler_name2} {fp_new} with {diff} range",
                    f"{filler_name} and {filler_name2} {fp_new} with range {diff}",

                    f"{filler_name} {filler_name2} {fp_new} with +/-{diff}",
                    f"{filler_name} {filler_name2} {fp_new} with +/-{diff} range",
                    f"{filler_name} and {filler_name2} {fp_new} with +/-{diff}",
                    f"{filler_name} and {filler_name2} {fp_new} with +/-{diff} range",
                    
                    f"{filler_name} {filler_name2} {tl} of {fp_new} with {diff} range",
                    f"{filler_name} {filler_name2} {tl} of {fp_new} with range {diff}",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with {diff} range",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with range {diff}",

                    f"{filler_name} {filler_name2} {tl} of {fp_new} with +/-{diff}",
                    f"{filler_name} {filler_name2} {tl} of {fp_new} with +/-{diff} range",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with +/-{diff}",
                    f"{filler_name} and {filler_name2} {tl} of {fp_new} with +/-{diff} range",
                ]
                
            elif filler_details[1]['total_load']['value']:
                if not filler_details[1]['total_load']['min']:
                    query = [
                        f"{filler_name} {filler_name2} max {fp}",
                        f"{filler_name} {filler_name2} upto {fp}",
                        f"{filler_name} {filler_name2} max {fp}%",
                        f"{filler_name} {filler_name2} upto {fp}%",
                        f"{filler_name} {filler_name2} less than {fp}",
                        f"{filler_name} {filler_name2} less than {fp}%",
                        f"{filler_name} {filler_name2} <{fp}",
                        f"{filler_name} {filler_name2} <{fp}%",

                        f"{filler_name} {filler_name2} {tl} max {fp}",
                        f"{filler_name} {filler_name2} {tl} upto {fp}",
                        f"{filler_name} {filler_name2} {tl} max {fp}%",
                        f"{filler_name} {filler_name2} {tl} upto {fp}%",
                        f"{filler_name} {filler_name2} {tl} less than {fp}",
                        f"{filler_name} {filler_name2} {tl} less than {fp}%",
                        f"{filler_name} {filler_name2} {tl} <{fp}",
                        f"{filler_name} {filler_name2} {tl} <{fp}%",

                        f"{tl} max {fp} {filler_name} {filler_name2}",
                        f"{tl} upto {fp} {filler_name} {filler_name2}",
                        f"{tl} max {fp}% {filler_name} {filler_name2}",
                        f"{tl} upto {fp}% {filler_name} {filler_name2}",
                        f"{tl} less than {fp} {filler_name} {filler_name2}",
                        f"{tl} less than {fp}% {filler_name} {filler_name2}",
                        f"{tl} <{fp} {filler_name} {filler_name2}",
                        f"{tl} <{fp}% {filler_name} {filler_name2}",

                        # weightage
                        f"{filler_name} {filler_name2} max {fp}",
                        f"{filler_name} {filler_name2} upto {fp}",

                        f"{filler_name} {filler_name2} less than {fp}",
                        f"{filler_name} {filler_name2} <{fp}",

                        f"{filler_name} {filler_name2} {tl} max {fp}",
                        f"{filler_name} {filler_name2} {tl} upto {fp}",

                        f"{filler_name} {filler_name2} {tl} less than {fp}",
                        f"{filler_name} {filler_name2} {tl} <{fp}",

                        f"{tl} max {fp} {filler_name} {filler_name2}",
                        f"{tl} upto {fp} {filler_name} {filler_name2}",

                        f"{tl} less than {fp} {filler_name} {filler_name2}",
                        f"{tl} <{fp} {filler_name} {filler_name2}",
                    ]
                elif not filler_details[1]['total_load']['max']:
                    query = [
                        f"{filler_name} {filler_name2} min {fp}",
                        f"{filler_name} {filler_name2} at least {fp}",
                        f"{filler_name} {filler_name2} min {fp}%",
                        f"{filler_name} {filler_name2} at least {fp}%",
                        f"{filler_name} {filler_name2} greater than {fp}",
                        f"{filler_name} {filler_name2} greater than {fp}%",
                        f"{filler_name} {filler_name2} >{fp}",
                        f"{filler_name} {filler_name2} >{fp}%",

                        f"{filler_name} {filler_name2} {tl} min {fp}",
                        f"{filler_name} {filler_name2} {tl} at least {fp}",
                        f"{filler_name} {filler_name2} {tl} min {fp}%",
                        f"{filler_name} {filler_name2} {tl} at least {fp}%",
                        f"{filler_name} {filler_name2} {tl} greater than {fp}",
                        f"{filler_name} {filler_name2} {tl} greater than {fp}%",
                        f"{filler_name} {filler_name2} {tl} >{fp}",
                        f"{filler_name} {filler_name2} {tl} >{fp}%",

                        f"{tl} min {fp} {filler_name} {filler_name2}",
                        f"{tl} at least {fp} {filler_name} {filler_name2}",
                        f"{tl} min {fp}% {filler_name} {filler_name2}",
                        f"{tl} at least {fp}% {filler_name} {filler_name2}",
                        f"{tl} greater than {fp} {filler_name} {filler_name2}",
                        f"{tl} greater than {fp}% {filler_name} {filler_name2}",
                        f"{tl} >{fp} {filler_name} {filler_name2}",
                        f"{tl} >{fp}% {filler_name} {filler_name2}",

                        # weightage
                        f"{filler_name} {filler_name2} min {fp}",
                        f"{filler_name} {filler_name2} at least {fp}",

                        f"{filler_name} {filler_name2} greater than {fp}",
                        f"{filler_name} {filler_name2} >{fp}",

                        f"{filler_name} {filler_name2} {tl} min {fp}",
                        f"{filler_name} {filler_name2} {tl} at least {fp}",

                        f"{filler_name} {filler_name2} {tl} greater than {fp}",
                        f"{filler_name} {filler_name2} {tl} >{fp}",

                        f"{tl} min {fp} {filler_name} {filler_name2}",
                        f"{tl} at least {fp} {filler_name} {filler_name2}",

                        f"{tl} greater than {fp} {filler_name} {filler_name2}",
                        f"{tl} >{fp} {filler_name} {filler_name2}",
                    ]
                else:
                    query = [
                        f"{filler_name} {filler_name2}{fp}",
                        f"{filler_name} {filler_name2}{fp}%",
                        f"{filler_name}{fp} {filler_name2}",
                        f"{filler_name}{fp}% {filler_name2}", 
                        f"{filler_name} {filler_name2}{fp}",
                        f"{filler_name} {filler_name2}{fp}%",
                        f"{filler_name}{fp} {filler_name2}",
                        f"{filler_name}{fp}% {filler_name2}",
                        
                        f"{filler_name} {fp}{filler_name2}",
                        f"{filler_name} {fp}%{filler_name2}",
                        f"{fp}{filler_name} {filler_name2}",
                        f"{fp}%{filler_name} {filler_name2}", 
                        f"{filler_name} {fp}{filler_name2}",
                        f"{filler_name} {fp}%{filler_name2}",
                        f"{fp}{filler_name} {filler_name2}",
                        f"{fp}%{filler_name} {filler_name2}",
                        
                        f"{filler_name} {filler_name2} {fp}",
                        f"{filler_name} {filler_name2} {fp}%",
                        f"{filler_name}, {filler_name2} {fp}",
                        f"{filler_name}, {filler_name2} {fp}%",
                        f"{filler_name} and {filler_name2} {fp}",
                        f"{filler_name} and {filler_name2} {fp}%",
                        f"{filler_name} {filler_name2} with {fp} {tl}",
                        f"{filler_name} {filler_name2} with {fp}% {tl}",
                        f"{filler_name} and {filler_name2} with {fp} {tl}",
                        f"{filler_name} and {filler_name2} with {fp}% {tl}",
                        f"{filler_name}, {filler_name2} with {fp} {tl}",
                        f"{filler_name}, {filler_name2} with {fp}% {tl}",
                        f"{filler_name} {filler_name2} with {tl} {fp}",
                        f"{filler_name} {filler_name2} with {tl} {fp}%",
                        f"{filler_name}, {filler_name2} with {tl} {fp}",
                        f"{filler_name}, {filler_name2} with {tl} {fp}%",
                        f"{filler_name} and {filler_name2} with {tl} {fp}",
                        f"{filler_name} and {filler_name2} with {tl} {fp}%",
                        f"{tl} of {fp}% with {filler_name} and {filler_name2}",
                        f"{tl} of {fp} with {filler_name} and {filler_name2}",
                        
                        f"{filler_name} {fp1} {filler_name2} {fp2}",
                        f"{filler_name} {fp1} {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}% {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}, {filler_name2} {fp2}",
                        f"{filler_name} {fp1}, {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}%, {filler_name2} {fp2}%",
                        f"{filler_name} {fp1} and {filler_name2} {fp2}",
                        f"{filler_name} {fp1} and {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}% and {filler_name2} {fp2}%",

                        f"{fp1} {filler_name} {fp2} {filler_name2}",
                        f"{fp1} {filler_name} {fp2}% {filler_name2}",
                        f"{fp1}% {filler_name} {fp2}% {filler_name2}",
                        f"{fp1} {filler_name}, {fp2} {filler_name2}",
                        f"{fp1} {filler_name}, {fp2}% {filler_name2}",
                        f"{fp1}% {filler_name}, {fp2}% {filler_name2}",
                        f"{fp1} {filler_name} and {fp2} {filler_name2}",
                        f"{fp1} {filler_name} and {fp2}% {filler_name2}",
                        f"{fp1}% {filler_name} and {fp2}% {filler_name2}",  

                        f"{fp1}% {filler_name} + {fp2}% {filler_name2}",
                        f"{filler_name} {fp1}% + {filler_name2} {fp2}%",
                        f"{fp1} {filler_name} + {fp2} {filler_name2}",
                        f"{filler_name} {fp1} + {filler_name2} {fp2}",

                        f"{fp1}% {filler_name} / {fp2}% {filler_name2}",
                        f"{filler_name} {fp1}% / {filler_name2} {fp2}%",
                        f"{fp1} {filler_name} / {fp2} {filler_name2}",
                        f"{filler_name} {fp1} / {filler_name2} {fp2}",

                        #weightage
                        f"{filler_name} {filler_name2} {fp}",
                        f"{filler_name}, {filler_name2} {fp}",
                        f"{filler_name} and {filler_name2} {fp}",
                        f"{filler_name} {filler_name2} with {fp} {tl}",

                        f"{filler_name} and {filler_name2} with {fp} {tl}",
                        f"{filler_name}, {filler_name2} with {fp} {tl}",

                        f"{filler_name} {filler_name2} with {tl} {fp}",
                        f"{filler_name}, {filler_name2} with {tl} {fp}",
                        f"{filler_name} and {filler_name2} with {tl} {fp}",
                        f"{tl} of {fp} with {filler_name} and {filler_name2}",
                        
                        f"{filler_name} {fp1} {filler_name2} {fp2}",
                        f"{filler_name} {fp1}, {filler_name2} {fp2}",
                        f"{filler_name} {fp1} and {filler_name2} {fp2}",

                        f"{fp1} {filler_name} {fp2} {filler_name2}",
                        f"{fp1} {filler_name}, {fp2} {filler_name2}",
                        f"{fp1} {filler_name} and {fp2} {filler_name2}",

                        f"{fp1} {filler_name} + {fp2} {filler_name2}",
                        f"{filler_name} {fp1} + {filler_name2} {fp2}",

                        f"{fp1} {filler_name} / {fp2} {filler_name2}",
                        f"{filler_name} {fp1} / {filler_name2} {fp2}",


                        f"{filler_name} {fp1} {filler_name2} {fp2}",
                        f"{filler_name} {fp1} {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}% {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}, {filler_name2} {fp2}",
                        f"{filler_name} {fp1}, {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}%, {filler_name2} {fp2}%",
                        f"{filler_name} {fp1} and {filler_name2} {fp2}",
                        f"{filler_name} {fp1} and {filler_name2} {fp2}%",
                        f"{filler_name} {fp1}% and {filler_name2} {fp2}%",

                        f"{fp1} {filler_name} {fp2} {filler_name2}",
                        f"{fp1} {filler_name} {fp2}% {filler_name2}",
                        f"{fp1}% {filler_name} {fp2}% {filler_name2}",
                        f"{fp1} {filler_name}, {fp2} {filler_name2}",
                        f"{fp1} {filler_name}, {fp2}% {filler_name2}",
                        f"{fp1}% {filler_name}, {fp2}% {filler_name2}",
                        f"{fp1} {filler_name} and {fp2} {filler_name2}",
                        f"{fp1} {filler_name} and {fp2}% {filler_name2}",
                        f"{fp1}% {filler_name} and {fp2}% {filler_name2}",

                        f"{fp1}% {filler_name} + {fp2}% {filler_name2}",
                        f"{filler_name} {fp1}% + {filler_name2} {fp2}%",
                        f"{fp1} {filler_name} + {fp2} {filler_name2}",
                        f"{filler_name} {fp1} + {filler_name2} {fp2}",

                        f"{fp1}% {filler_name} / {fp2}% {filler_name2}",
                        f"{filler_name} {fp1}% / {filler_name2} {fp2}%",
                        f"{fp1} {filler_name} / {fp2} {filler_name2}",
                        f"{filler_name} {fp1} / {filler_name2} {fp2}",
                    ]

                    if len(filler_name) <5 and len(filler_name2) <5:
                        # duplicated for more weightage
                        additional = [
                            f"{filler_name}{fp1} {filler_name2}{fp2}",
                            f"{filler_name}{fp1} {filler_name2}{fp2}%",
                            f"{filler_name}{fp1}% {filler_name2}{fp2}",
                            f"{filler_name}{fp1}% {filler_name2}{fp2}%",
                            f"{filler_name}{fp1} and {filler_name2}{fp2}",
                            f"{filler_name}{fp1}% and {filler_name2}{fp2}%",
                            f"{filler_name}{fp1}%, {filler_name2}{fp2}%",
                            f"{filler_name}{fp1}, {filler_name2}{fp2}",
                        
                            f"{fp1}{filler_name} {fp2}{filler_name2}",
                            f"{fp1}{filler_name} {fp2}%{filler_name2}",
                            f"{fp1}%{filler_name} {fp2}{filler_name2}",
                            f"{fp1}%{filler_name} {fp2}%{filler_name2}",
                            f"{fp1}{filler_name} and {fp2}{filler_name2}",
                            f"{fp1}%{filler_name} and {fp2}%{filler_name2}",
                            f"{fp1}{filler_name}, {fp2}{filler_name2}",
                            f"{fp1}%{filler_name}, {fp2}%{filler_name2}",
                            
                            f"{fp1}{filler_name}+{fp2}{filler_name2}",
                            f"{filler_name}{fp1}+{filler_name2}{fp2}",
                            f"{filler_name}{fp1}%+{filler_name2}{fp2}%",
                            
                            f"{fp1}{filler_name}/{fp2}{filler_name2}",
                            f"{filler_name}{fp1}/{filler_name2}{fp2}",
                            f"{filler_name}{fp1}%/{filler_name2}{fp2}%",

                            # weightage
                            f"{filler_name}{fp1} {filler_name2}{fp2}",
                            f"{filler_name}{fp1} {filler_name2}{fp2}",
                            f"{filler_name}{fp1} and {filler_name2}{fp2}",
                            f"{filler_name}{fp1}, {filler_name2}{fp2}",
                        
                            f"{fp1}{filler_name} {fp2}{filler_name2}",
                            f"{fp1}{filler_name} {fp2}{filler_name2}",

                            f"{fp1}{filler_name} and {fp2}{filler_name2}",
                            f"{fp1}{filler_name}, {fp2}{filler_name2}",

                            f"{fp1}{filler_name}+{fp2}{filler_name2}",
                            f"{filler_name}{fp1}+{filler_name2}{fp2}",

                            f"{fp1}{filler_name}/{fp2}{filler_name2}",
                            f"{filler_name}{fp1}/{filler_name2}{fp2}",

                            f"{filler_name}{fp1} {filler_name2}{fp2}",
                            f"{filler_name}{fp1} {filler_name2}{fp2}%",
                            f"{filler_name}{fp1}% {filler_name2}{fp2}",
                            f"{filler_name}{fp1}% {filler_name2}{fp2}%",
                            f"{filler_name}{fp1} and {filler_name2}{fp2}",
                            f"{filler_name}{fp1}% and {filler_name2}{fp2}%",
                            f"{filler_name}{fp1}%, {filler_name2}{fp2}%",
                            f"{filler_name}{fp1}, {filler_name2}{fp2}",
                        
                            f"{fp1}{filler_name} {fp2}{filler_name2}",
                            f"{fp1}{filler_name} {fp2}%{filler_name2}",
                            f"{fp1}%{filler_name} {fp2}{filler_name2}",
                            f"{fp1}%{filler_name} {fp2}%{filler_name2}",
                            f"{fp1}{filler_name} and {fp2}{filler_name2}",
                            f"{fp1}%{filler_name} and {fp2}%{filler_name2}",
                            f"{fp1}{filler_name}, {fp2}{filler_name2}",
                            f"{fp1}%{filler_name}, {fp2}%{filler_name2}",

                            f"{fp1}{filler_name}+{fp2}{filler_name2}",
                            f"{filler_name}{fp1}+{filler_name2}{fp2}",
                            f"{filler_name}{fp1}%+{filler_name2}{fp2}%",
                        
                            f"{fp1}{filler_name}/{fp2}{filler_name2}",
                            f"{filler_name}{fp1}/{filler_name2}{fp2}",
                            f"{filler_name}{fp1}%/{filler_name2}{fp2}%",
                            
                        ]
                        query.extend(additional)
            else:
                query = [
                    f"{filler_name} {filler_name2}",
                    f"{filler_name}, {filler_name2}",
                    f"{filler_name} and {filler_name2}",
                    f"{filler_name} or {filler_name2}",
                    f"{filler_name} / {filler_name2}",
                    f"{filler_name} + {filler_name2}",
                ]

                if len(filler_name) <5 and len(filler_name2) <5:
                    additional = [
                        f"{filler_name}+{filler_name2}",
                        f"{filler_name}+{filler_name2}",
                        f"{filler_name}/{filler_name2}",
                        f"{filler_name}/{filler_name2}",
                    ]
                    query.extend(additional)
        elif filler_details[1]['total_load']['value']:
            if "range" in filler_details[1]:
                del filler_details[1]["range"]
                is_filler_range = True

                query_range_without_value = [
                    f"{filler_name} {fp_min} - {fp_max}",
                    f"{filler_name} {fp_min} - {fp_max}%",
                    f"{filler_name} {fp_min}% - {fp_max}%",
                    f"{filler_name} {fp_min}% to {fp_max}%",
                    f"{filler_name} ranging between {fp_min}% - {fp_max}%",
                    f"{filler_name} in the range of {fp_min} - {fp_max}%",
                    f"{filler_name} range {fp_min} to {fp_max}%",

                    f"{filler_name} {tl} of {fp_min} - {fp_max}",
                    f"{filler_name} {tl} {fp_min} - {fp_max}%",
                    f"{filler_name} {tl} of {fp_min}% - {fp_max}%",
                    f"{filler_name} {tl} {fp_min}% to {fp_max}%",
                    f"{filler_name} {tl} ranging between {fp_min}% - {fp_max}%",
                    f"{filler_name} {tl} in the range of {fp_min} - {fp_max}%",
                    f"{filler_name} {tl} range {fp_min} to {fp_max}%",

                    # weightage
                    f"{filler_name} {fp_min} - {fp_max}",
                    f"{filler_name} {fp_min} to {fp_max}",
                    f"{filler_name} ranging between {fp_min} - {fp_max}",
                    f"{filler_name} in the range of {fp_min} - {fp_max}",
                    f"{filler_name} range {fp_min} to {fp_max}",

                    f"{filler_name} {tl} of {fp_min} - {fp_max}",
                    f"{filler_name} {tl} {fp_min} - {fp_max}",
                    f"{filler_name} {tl} {fp_min} to {fp_max}",
                    f"{filler_name} {tl} ranging between {fp_min} - {fp_max}",
                    f"{filler_name} {tl} in the range of {fp_min} - {fp_max}",
                    f"{filler_name} {tl} range {fp_min} to {fp_max}",
                ]

                query = [
                    f"{filler_name} {fp_new} with range {diff}%",
                    f"{filler_name} {fp_new}% with range {diff}%",
                    f"{filler_name} {fp_new} with {diff}% range",
                    f"{filler_name} {fp_new}% with {diff}% range",
                
                    f"{filler_name} {fp_new} with +/-{diff}% range",
                    f"{filler_name} {fp_new} with range +/-{diff}%",
                    f"{filler_name} {fp_new} with +/-{diff}",
                    f"{filler_name} {fp_new}% with +/-{diff}% range",
                    f"{filler_name} {fp_new}% with range +/-{diff}%",
                    f"{filler_name} {fp_new}% with +/-{diff}%",
                    
                    f"{filler_name} {tl} {fp_new} with range {diff}%",
                    f"{filler_name} {tl} of {fp_new} with range {diff}%",
                    f"{filler_name} {tl} {fp_new} with {diff}% range",
                    f"{filler_name} {tl} of {fp_new} with {diff}% range",

                    f"{filler_name} {tl} of {fp_new} with +/-{diff}% range",
                    f"{filler_name} {tl} of {fp_new} with range +/-{diff}%",
                    f"{filler_name} {tl} of {fp_new} with +/-{diff}",
                    f"{filler_name} {tl} of {fp_new} with +/-{diff}%",

                    f"{filler_name} {fp_new} with range {diff}",
                    f"{filler_name} {fp_new} with {diff} range",
                
                    f"{filler_name} {fp_new} with +/-{diff} range",
                    f"{filler_name} {fp_new} with range +/-{diff}",
                    f"{filler_name} {fp_new} with +/-{diff}",
                    
                    f"{filler_name} {tl} {fp_new} with range {diff}",
                    f"{filler_name} {tl} of {fp_new} with range {diff}",
                    f"{filler_name} {tl} {fp_new} with {diff} range",
                    f"{filler_name} {tl} of {fp_new} with {diff} range",

                    f"{filler_name} {tl} of {fp_new} with +/-{diff} range",
                    f"{filler_name} {tl} of {fp_new} with range +/-{diff}",
                    f"{filler_name} {tl} of {fp_new} with +/-{diff}",
                ]
            else:
                if not filler_details[1]['total_load']['min']:
                    query = [
                        f"{filler_name} max {fp}",
                        f"{filler_name} upto {fp}",
                        f"{filler_name} max {fp}%",
                        f"{filler_name} upto {fp}%",
                        f"{filler_name} less than {fp}",
                        f"{filler_name} less than {fp}%",
                        f"{filler_name} <{fp}",
                        f"{filler_name} <{fp}%",

                        f"{filler_name} {tl} max {fp}",
                        f"{filler_name} {tl} upto {fp}",
                        f"{filler_name} {tl} max {fp}%",
                        f"{filler_name} {tl} upto {fp}%",
                        f"{filler_name} {tl} less than {fp}",
                        f"{filler_name} {tl} less than {fp}%",
                        f"{filler_name} {tl} <{fp}",
                        f"{filler_name} {tl} <{fp}%",

                        f"{tl} max {fp} {filler_name}",
                        f"{tl} upto {fp} {filler_name}",
                        f"{tl} max {fp}% {filler_name}",
                        f"{tl} upto {fp}% {filler_name}",
                        f"{tl} less than {fp} {filler_name}",
                        f"{tl} less than {fp}% {filler_name}",
                        f"{tl} <{fp} {filler_name}",
                        f"{tl} <{fp}% {filler_name}",

                        # weightage
                        f"{filler_name} max {fp}",
                        f"{filler_name} upto {fp}",

                        f"{filler_name} less than {fp}",
                        f"{filler_name} <{fp}",

                        f"{filler_name} {tl} max {fp}",
                        f"{filler_name} {tl} upto {fp}",

                        f"{filler_name} {tl} less than {fp}",
                        f"{filler_name} {tl} <{fp}",

                        f"{tl} max {fp} {filler_name}",
                        f"{tl} upto {fp} {filler_name}",

                        f"{tl} less than {fp} {filler_name}",
                        f"{tl} <{fp} {filler_name}",
                    ]
                elif not filler_details[1]['total_load']['max']:
                    query = [
                        f"{filler_name} min {fp}",
                        f"{filler_name} at least {fp}",
                        f"{filler_name} min {fp}%",
                        f"{filler_name} at least {fp}%",
                        f"{filler_name} greater than {fp}",
                        f"{filler_name} greater than {fp}%",
                        f"{filler_name} >{fp}",
                        f"{filler_name} >{fp}%",

                        f"{filler_name} {tl} min {fp}",
                        f"{filler_name} {tl} at least {fp}",
                        f"{filler_name} {tl} min {fp}%",
                        f"{filler_name} {tl} at least {fp}%",
                        f"{filler_name} {tl} greater than {fp}",
                        f"{filler_name} {tl} greater than {fp}%",
                        f"{filler_name} {tl} >{fp}",
                        f"{filler_name} {tl} >{fp}%",

                        f"{tl} min {fp} {filler_name}",
                        f"{tl} at least {fp} {filler_name}",
                        f"{tl} min {fp}% {filler_name}",
                        f"{tl} at least {fp}% {filler_name}",
                        f"{tl} greater than {fp} {filler_name}",
                        f"{tl} greater than {fp}% {filler_name}",
                        f"{tl} >{fp} {filler_name}",
                        f"{tl} >{fp}% {filler_name}",

                        # weightage
                        f"{filler_name} min {fp}",
                        f"{filler_name} at least {fp}",
                        f"{filler_name} greater than {fp}",
                        f"{filler_name} >{fp}",

                        f"{filler_name} {tl} min {fp}",
                        f"{filler_name} {tl} at least {fp}",
                        f"{filler_name} {tl} greater than {fp}",
                        f"{filler_name} {tl} >{fp}",

                        f"{tl} min {fp} {filler_name}",
                        f"{tl} at least {fp} {filler_name}",
                        f"{tl} greater than {fp} {filler_name}",
                        f"{tl} >{fp} {filler_name}",
                    ]
                else:
                    query = [
                        f"{filler_name} {fp}",
                        f"{filler_name}, {fp}",
                        f"{filler_name} {fp}%",
                        f"{filler_name}, {fp}%",
                        f"{filler_name} - {fp}",
                        f"{filler_name} - {fp}%",
                        f"{filler_name} of {fp}",
                        f"{filler_name} of {fp}%",
                        f"{filler_name} with {tl} {fp}",
                        f"{filler_name} with {tl} {fp}%",
                        f"{tl} of {fp}% with {filler_name}",
                        f"{tl} of {fp} with {filler_name}",
                        f"{fp} {tl} with {filler_name}",
                        f"{fp}% {tl} with {filler_name}",

                        # weightage
                        f"{filler_name} {fp}",
                        f"{filler_name}, {fp}",

                        f"{filler_name} - {fp}",
                        f"{filler_name} of {fp}",

                        f"{filler_name} with {tl} {fp}",
                        f"{tl} of {fp} with {filler_name}",
                        f"{fp} {tl} with {filler_name}",
                    ]
                    if len(filler_name) <5:
                        # duplicated for more weightage
                        additional = [
                            f"{filler_name}{fp}",
                            f"{filler_name}{fp}%",
                            f"{fp}{filler_name}",
                            f"{fp}%{filler_name}",
                            f"{filler_name}{fp}",
                            f"{filler_name}{fp}%",
                            f"{fp}{filler_name}",
                            f"{fp}%{filler_name}",
                            f"{filler_name}{fp}",
                            f"{filler_name}{fp}%",
                            f"{fp}{filler_name}",
                            f"{fp}%{filler_name}",

                            # weightage
                            f"{filler_name}{fp}",
                            f"{fp}{filler_name}",

                            f"{fp}{filler_name}",
                            f"{filler_name}{fp}",
                        ]
                        query.extend(additional)
                
        elif not filler_details[1]['total_load']['value']:
             query = [filler_name]


    if is_filler_range:
        is_without_value = random.choice([False, False, True, True, True])
        if is_without_value:
            filler_details[1]['total_load']['value'] = filler_details[1]['total_load']['min']
            query = random.choice(query_range_without_value)
        else:
            query = random.choice(query)
    else:
        query = random.choice(query)
        
    remove_all = {'filler_name': ['all']}
    filler_details = [i for i in filler_details if i != remove_all]       
    return filler_details, query

In [42]:
for i in range(1, 100):
    print(get_random_filler(), "\n")

([{'filler_name': ['glass fiber']}, {'total_load': {'value': 73, 'min': 68, 'max': 78}}], 'gf 73%') 

([{'total_load': {'value': 88, 'min': 88, 'max': None}}], 'load >88%') 

([{'filler_name': ['unfilled']}, {'total_load': {'value': None, 'min': None, 'max': None}}], 'without fill') 

([{'filler_name': ['carbon powder', 'long carbon fiber']}, {'total_load': {'value': 55, 'min': 50, 'max': 60}}], 'carbon powder 55%long carbon fiber') 

([{'filler_name': ['natural organic fiber']}, {'total_load': {'value': 45, 'min': None, 'max': 45}}], 'natural organic fiber max 45%') 

([{'filler_name': ['continuous carbon fiber']}, {'total_load': {'value': 56, 'min': None, 'max': 56}}], 'continuous carbon fiber filler load <56') 

([{'filler_name': ['carbon']}, {'total_load': {'value': 46, 'min': 46, 'max': None}}], 'carbon load >46') 

([{'total_load': {'value': 71, 'min': None, 'max': 71}}], 'load max 71%') 

([{'filler_name': ['mineral']}, {'total_load': {'value': 77, 'min': None, 'max': 77}}], 'td

### Brands

In [43]:
all_brands

['ecomid',
 'pip',
 'mcm',
 'ge',
 'tc',
 'ol',
 'tx',
 'lft',
 'celstran lft',
 'zenite',
 'thermx',
 'ze',
 'fri',
 'sp',
 've',
 'hyt',
 'cnl',
 'fortron',
 'santoprene',
 'fo',
 'opp',
 'mid',
 'pf',
 'lp',
 'oc',
 'cl',
 'fa',
 'frianyl',
 'zyt',
 'celstran',
 'celanex',
 'cs',
 'hostaform',
 'celcon',
 'po',
 'ff',
 'ot',
 'kepital',
 'pultrusion',
 'hf',
 'va',
 'cp',
 'im',
 'cn',
 'coolpoly',
 'rynite',
 'cra',
 'vectra',
 'crastin',
 'bf',
 'hytrel',
 'lfrt',
 'vamc',
 'celanyl',
 'blue ridge',
 'kep',
 'neo',
 'lftr',
 'pb',
 'htr',
 'fp',
 'zytel',
 'br',
 'am',
 'gur',
 'cs lft',
 'stp',
 'ta',
 'ryn',
 'cx']

In [44]:
for b in OOS_Data['brands']:
    if b not in all_brands:
        all_brands.append(b)
        
all_brands = [x for x in all_brands if x not in ['at', 'lt', 'min', 'sf', 'sr', 'tp', 'im', 'ny', 'htr']]
all_brands

['ecomid',
 'pip',
 'mcm',
 'ge',
 'tc',
 'ol',
 'tx',
 'lft',
 'celstran lft',
 'zenite',
 'thermx',
 'ze',
 'fri',
 'sp',
 've',
 'hyt',
 'cnl',
 'fortron',
 'santoprene',
 'fo',
 'opp',
 'mid',
 'pf',
 'lp',
 'oc',
 'cl',
 'fa',
 'frianyl',
 'zyt',
 'celstran',
 'celanex',
 'cs',
 'hostaform',
 'celcon',
 'po',
 'ff',
 'ot',
 'kepital',
 'pultrusion',
 'hf',
 'va',
 'cp',
 'cn',
 'coolpoly',
 'rynite',
 'cra',
 'vectra',
 'crastin',
 'bf',
 'hytrel',
 'lfrt',
 'vamc',
 'celanyl',
 'blue ridge',
 'kep',
 'neo',
 'lftr',
 'pb',
 'fp',
 'zytel',
 'br',
 'am',
 'gur',
 'cs lft',
 'stp',
 'ta',
 'ryn',
 'cx',
 'forflex',
 'kepamid',
 'abistir',
 'impet',
 'blueridge',
 'blendfor',
 'omnilon',
 'nylfor',
 'ateva',
 'selar',
 'factor',
 'tecnoprene',
 'neolast',
 'omnicarb',
 'sofpur',
 'micromax',
 'tarnoform',
 'nilamid',
 'celapex',
 'omnitech',
 'pibifor',
 'vamac',
 'kepex',
 'pipelon',
 'vitaldose',
 'forprene',
 'cecopoly',
 'clarifoil',
 'compel',
 'talcoprene',
 'litepol',
 'sikam

In [45]:
brand_syns = synonym_df2[synonym_df2['TYPE']=='brand'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
brand_syns

{'abistir': ['abistir'],
 'amcel': ['amcel', 'am'],
 'ateva': ['ateva'],
 'bexloy': ['bexloy'],
 'blendfor': ['blendfor', 'bf'],
 'blueridge': ['blueridge', 'br', 'blue ridge'],
 'cecopoly': ['cecopoly'],
 'celanex': ['celanex', 'cx'],
 'celanyl': ['celanyl', 'cnl'],
 'celapex': ['celapex', 'cl'],
 'celcon': ['celcon', 'cn'],
 'celstran': ['celstran',
  'cs',
  'lft',
  'lfrt',
  'cs lft',
  'celstran lft',
  'pultrusion',
  'lftr'],
 'clarifoil': ['clarifoil'],
 'compel': ['compel'],
 'coolpoly': ['coolpoly', 'cp'],
 'crastin': ['crastin', 'cra'],
 'ecomid': ['ecomid'],
 'elvamide': ['elvamide', 'mid'],
 'factor': ['factor', 'fa'],
 'forflex': ['forflex', 'ff'],
 'forprene': ['forprene', 'fp'],
 'fortron': ['fortron', 'fo'],
 'frianyl': ['frianyl', 'fri'],
 'geolast': ['geolast', 'ge'],
 'gur': ['gur'],
 'hostaform': ['hostaform', 'hf'],
 'hytrel': ['hytrel', 'hyt'],
 'impet': ['impet'],
 'kepamid': ['kepamid'],
 'kepex': ['kepex'],
 'kepital': ['kepital', 'kep'],
 'keploy': ['keploy'

In [46]:
for i in all_brands:
    skip = True
    for j in brand_syns:
        if i in brand_syns[j]:
            skip = False
    if skip:
        print(i)
        brand_syns[i] = [i]

selar


In [47]:
brand_syn_mapping = {}
for b_meaning, b_syns in brand_syns.items():
    for syn in b_syns:
        brand_syn_mapping[syn] = b_meaning

brand_syn_mapping

{'abistir': 'abistir',
 'amcel': 'amcel',
 'am': 'amcel',
 'ateva': 'ateva',
 'bexloy': 'bexloy',
 'blendfor': 'blendfor',
 'bf': 'blendfor',
 'blueridge': 'blueridge',
 'br': 'blueridge',
 'blue ridge': 'blueridge',
 'cecopoly': 'cecopoly',
 'celanex': 'celanex',
 'cx': 'celanex',
 'celanyl': 'celanyl',
 'cnl': 'celanyl',
 'celapex': 'celapex',
 'cl': 'celapex',
 'celcon': 'celcon',
 'cn': 'celcon',
 'celstran': 'celstran',
 'cs': 'celstran',
 'lft': 'celstran',
 'lfrt': 'celstran',
 'cs lft': 'celstran',
 'celstran lft': 'celstran',
 'pultrusion': 'celstran',
 'lftr': 'celstran',
 'clarifoil': 'clarifoil',
 'compel': 'compel',
 'coolpoly': 'coolpoly',
 'cp': 'coolpoly',
 'crastin': 'crastin',
 'cra': 'crastin',
 'ecomid': 'ecomid',
 'elvamide': 'elvamide',
 'mid': 'elvamide',
 'factor': 'factor',
 'fa': 'factor',
 'forflex': 'forflex',
 'ff': 'forflex',
 'forprene': 'forprene',
 'fp': 'forprene',
 'fortron': 'fortron',
 'fo': 'fortron',
 'frianyl': 'frianyl',
 'fri': 'frianyl',
 'geo

In [48]:
for i in brand_syn_mapping:
    if i not in all_brands:
        print(i)
        print(f"add the brand {i} to all_brands list")

bexloy
add the brand bexloy to all_brands list
geolast
add the brand geolast to all_brands list


In [49]:
all_brands_without_syns = []
for i in all_brands:
    if i in brand_syn_mapping and brand_syn_mapping[i]==i:
        all_brands_without_syns.append(i)
    elif i not in brand_syn_mapping:
        all_brands_without_syns.append(i)

all_brands_without_syns

['ecomid',
 'zenite',
 'thermx',
 'fortron',
 'santoprene',
 'frianyl',
 'celstran',
 'celanex',
 'hostaform',
 'celcon',
 'kepital',
 'coolpoly',
 'rynite',
 'vectra',
 'crastin',
 'hytrel',
 'celanyl',
 'zytel',
 'gur',
 'forflex',
 'kepamid',
 'abistir',
 'impet',
 'blueridge',
 'blendfor',
 'omnilon',
 'nylfor',
 'ateva',
 'selar',
 'factor',
 'tecnoprene',
 'neolast',
 'omnicarb',
 'sofpur',
 'micromax',
 'tarnoform',
 'nilamid',
 'celapex',
 'omnitech',
 'pibifor',
 'vamac',
 'kepex',
 'pipelon',
 'vitaldose',
 'forprene',
 'cecopoly',
 'clarifoil',
 'compel',
 'talcoprene',
 'litepol',
 'sikamid',
 'laprene',
 'elvamide',
 'keploy',
 'vandar',
 'stirofor',
 'amcel',
 'polifor',
 'omnipro',
 'sofprene',
 'maximid',
 'minlon',
 'pibiter']

### Grades

In [50]:
len(all_gradenames)

4228

In [51]:
for g in OOS_Data['grades']:
    if g not in all_gradenames:
        all_gradenames.append(g)

len(all_gradenames)

18148

### Polymers

In [52]:
all_polymers = [item.lower() for item in unique_values['Polymer']]

In [53]:
print(len(all_polymers))
for p in OOS_Data['polymers']:
    if p not in all_polymers:
        all_polymers.append(p)

print(len(all_polymers))

24
42


In [54]:
polymer_syns = synonym_df2[synonym_df2['TYPE']=='polymer'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
polymer_syns

{'abs': ['abs', 'acrylonitrile-butadiene-styrene'],
 'aem': ['aem', 'ethylene methylacrylate'],
 'asa': ['asa'],
 'evac': ['evac', 'ethylene-vinyl-acetate', 'eva'],
 'lcp': ['lcp', 'liquid crystal polymer', 'liquid - crystalline polymer'],
 'lcpa': ['lcpa', 'long chain nylon', 'long chain pa'],
 'pa*': ['pa*', 'polyamide', 'pa', 'nylon'],
 'pa1010': ['pa1010',
  'polyamide 1010',
  'nylon 1010',
  'pa 1010',
  'polyamide 10/10',
  'nylon 10/10',
  'pa10/10',
  'pa 10/10',
  'polyamide 10,10',
  'nylon 10,10',
  'pa 10,10',
  'pa10,10',
  'polyamide 10-10',
  'nylon 10-10',
  'pa 10-10',
  'pa10-10'],
 'pa12': ['pa12', 'polyamide 12', 'nylon 12', 'pa 12'],
 'pa6': ['pa6', 'nylon 6', 'pa 6', 'polyamide 6'],
 'pa610': ['pa610',
  'polyamide 610',
  'nylon 610',
  'pa 610',
  'nylon 6-10',
  'polyamide 6-10',
  'pa 6-10',
  'pa6-10',
  'nylon 6/10',
  'polyamide 6/10',
  'pa 6/10',
  'pa6/10',
  'nylon 6,10',
  'polyamide 6,10',
  'pa 6,10',
  'pa6,10'],
 'pa612': ['pa612',
  'polyamide 61

In [55]:
polymer_syn_mapping = {}
for poly_meaning, poly_syns in polymer_syns.items():
    for syn in poly_syns:
        polymer_syn_mapping[syn] = poly_meaning

polymer_syn_mapping

{'abs': 'abs',
 'acrylonitrile-butadiene-styrene': 'abs',
 'aem': 'aem',
 'ethylene methylacrylate': 'aem',
 'asa': 'asa',
 'evac': 'evac',
 'ethylene-vinyl-acetate': 'evac',
 'eva': 'evac',
 'lcp': 'lcp',
 'liquid crystal polymer': 'lcp',
 'liquid - crystalline polymer': 'lcp',
 'lcpa': 'lcpa',
 'long chain nylon': 'lcpa',
 'long chain pa': 'lcpa',
 'pa*': 'pa*',
 'polyamide': 'pa*',
 'pa': 'pa*',
 'nylon': 'pa*',
 'pa1010': 'pa1010',
 'polyamide 1010': 'pa1010',
 'nylon 1010': 'pa1010',
 'pa 1010': 'pa1010',
 'polyamide 10/10': 'pa1010',
 'nylon 10/10': 'pa1010',
 'pa10/10': 'pa1010',
 'pa 10/10': 'pa1010',
 'polyamide 10,10': 'pa1010',
 'nylon 10,10': 'pa1010',
 'pa 10,10': 'pa1010',
 'pa10,10': 'pa1010',
 'polyamide 10-10': 'pa1010',
 'nylon 10-10': 'pa1010',
 'pa 10-10': 'pa1010',
 'pa10-10': 'pa1010',
 'pa12': 'pa12',
 'polyamide 12': 'pa12',
 'nylon 12': 'pa12',
 'pa 12': 'pa12',
 'pa6': 'pa6',
 'nylon 6': 'pa6',
 'pa 6': 'pa6',
 'polyamide 6': 'pa6',
 'pa610': 'pa610',
 'polyam

In [56]:
for p in polymer_syn_mapping:
    if p not in all_polymers:
        all_polymers.append(p)

In [57]:
for i in polymer_syn_mapping:
    if i not in all_polymers:
        print(f"add polymer '{i}' to all_polymers")

In [58]:
all_polymers_without_syns = []
for i in all_polymers:
    if i in polymer_syn_mapping and polymer_syn_mapping[i]==i:
        all_polymers_without_syns.append(i)
    elif i not in polymer_syn_mapping:
        all_polymers_without_syns.append(i)

all_polymers_without_syns

['pbt',
 'pet',
 'pa66',
 'pa6',
 'pa*',
 'pom',
 'pa666',
 'pe-hd',
 'pp',
 'pps',
 'tpu',
 'pc',
 'abs',
 'lcp',
 'pe-uhmw',
 'pe-hmw',
 'tpc',
 'tpv',
 'pct',
 'pa612',
 'pa6t/xt',
 'pa6t/66',
 'pa66/6t',
 'pa610',
 'evac',
 'pvdf',
 'smah',
 'pa6t/6i',
 'ps',
 'pa12',
 'paste',
 'ptt',
 'pe-ld',
 'pa1010',
 'sebs',
 'peek',
 'pamxd6',
 'asa',
 'sbs',
 'tpo',
 'aem',
 'lcpa',
 'pa6t',
 'polyether',
 'ppa',
 'ptfe',
 'tpe']

### Clean: Processing, Feature, and Delivery Form

In [59]:
processing_types = []
product_functions = []
delivery_forms = []
for item in unique_values['Feature']:
    if item['Feature Type']=='Processing':
        processing_types.append(item['FEATURE'].lower().replace("mould", "mold"))
    if item['Feature Type']=='Product Categories' and item['FEATURE'].lower() not in ['auto spec approved']:
        product_functions.append(item['FEATURE'].lower())
    if item['Feature Type']=='Delivery Form':
        delivery_forms.append(item['FEATURE'].lower())

In [60]:
print([i for i in OLD_UNIQUE_VALUES["FEATURE"] if len(i)<5])

['bd', 'bio', 'bm', 'cast', 'ebm', 'emi', 'esd', 'fda', 'film', 'fr', 'heat', 'hhr', 'hr', 'ibm', 'im', 'lds', 'lm', 'lt', 'lw', 'mmm', 'mt', 'nh', 'nhfr', 'pcr', 'pir', 'rec', 'rm', 'tape', 'uv']


### Processing

In [61]:
processing_syns = synonym_df2[synonym_df2['TYPE']=='processing'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
processing_syns

{'blow molding': ['blow molding',
  'blowmolding',
  'bm',
  'blow moulding',
  'blow moldable',
  'blow molded'],
 'calendering': ['calendering',
  'calandering',
  'calendered',
  'calendr',
  'calenderable'],
 'casting': ['casting', 'castable'],
 'coatable': ['coatable', 'coating', 'polymeric coating', 'spin coating'],
 'coextrusion': ['coextrusion'],
 'compression molding': ['compression molding',
  'compression moulding',
  'matched-die moulding',
  'matched-die molding',
  'matched die molding',
  'matched die moulding',
  'cold compression',
  'hot compression',
  'compmold',
  'compression moldable',
  'compression mouldable'],
 'extrusion - hose': ['extrusion - hose', 'hose extrusion'],
 'extrusion - ram': ['extrusion - ram', 'ram extrusion'],
 'extrusion - small tubing': ['extrusion - small tubing'],
 'extrusion - wire and cable': ['extrusion - wire and cable',
  'cable extrusion',
  'wire extrusion',
  'extrusion wire coating',
  'extrusion cable coating',
  'wire coating',


In [62]:
processing_syn_mapping = {}
for p_meaning, p_syns in processing_syns.items():
    for syn in p_syns:
        processing_syn_mapping[syn] = p_meaning

processing_syn_mapping

{'blow molding': 'blow molding',
 'blowmolding': 'blow molding',
 'bm': 'blow molding',
 'blow moulding': 'blow molding',
 'blow moldable': 'blow molding',
 'blow molded': 'blow molding',
 'calendering': 'calendering',
 'calandering': 'calendering',
 'calendered': 'calendering',
 'calendr': 'calendering',
 'calenderable': 'calendering',
 'casting': 'casting',
 'castable': 'casting',
 'coatable': 'coatable',
 'coating': 'coatable',
 'polymeric coating': 'coatable',
 'spin coating': 'coatable',
 'coextrusion': 'coextrusion',
 'compression molding': 'compression molding',
 'compression moulding': 'compression molding',
 'matched-die moulding': 'compression molding',
 'matched-die molding': 'compression molding',
 'matched die molding': 'compression molding',
 'matched die moulding': 'compression molding',
 'cold compression': 'compression molding',
 'hot compression': 'compression molding',
 'compmold': 'compression molding',
 'compression moldable': 'compression molding',
 'compression m

In [63]:
print(len(processing_types))
for i in processing_syn_mapping:
    if i not in processing_types:
        processing_types.append(i)

print(len(processing_types))

26
127


In [64]:
all_processing_without_syns = []
for i in processing_types:
    if i in processing_syn_mapping and processing_syn_mapping[i]==i:
        all_processing_without_syns.append(i)
    elif i not in processing_syn_mapping:
        all_processing_without_syns.append(i)

all_processing_without_syns

['injection molding',
 'other extrusion',
 'film extrusion',
 'profile extrusion',
 'sheet extrusion',
 'multi injection molding',
 'coextrusion',
 'compression molding',
 'blow molding',
 'calendering',
 'selective reinforcement',
 'thermoforming',
 'transfer molding',
 'coatable',
 'casting',
 'fiber spinning / gel spinning',
 'extrusion blow molding',
 'rotational molding',
 'foam processing',
 'extrusion - wire and cable',
 'extrusion - small tubing',
 'injection blow molding',
 'extrusion - hose',
 'extrusion - ram',
 'melt blowing / nonwovens']

### Delivery Form

In [65]:
delivery_forms

['pellets', 'granules', 'tape', 'powder', 'micropowder']

In [66]:
delivery_form_syns = synonym_df2[synonym_df2['TYPE']=='delivery form'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
delivery_form_syns

{'granules': ['granules'],
 'micropowder': ['micro powder', 'micro-powder'],
 'pellets': ['pellets', 'pelets', 'pellet'],
 'powder': ['powder'],
 'tape': ['tape', 'cfr', 'tp']}

In [67]:
delivery_syn_mapping = {}
for d_meaning, d_syns in delivery_form_syns.items():
    for syn in d_syns:
        delivery_syn_mapping[syn] = d_meaning

delivery_syn_mapping

{'granules': 'granules',
 'micro powder': 'micropowder',
 'micro-powder': 'micropowder',
 'pellets': 'pellets',
 'pelets': 'pellets',
 'pellet': 'pellets',
 'powder': 'powder',
 'tape': 'tape',
 'cfr': 'tape',
 'tp': 'tape'}

In [68]:
print(len(delivery_forms))
for i in delivery_syn_mapping:
    if i not in delivery_forms:
        delivery_forms.append(i)

print(len(delivery_forms))

5
11


In [69]:
all_delivery_without_syns = []
for i in delivery_forms:
    if i in delivery_syn_mapping and delivery_syn_mapping[i]==i:
        all_delivery_without_syns.append(i)
    elif i not in delivery_syn_mapping:
        all_delivery_without_syns.append(i)

all_delivery_without_syns

['pellets', 'granules', 'tape', 'powder', 'micropowder']

### Feature

In [70]:
feature_syn_meaning_mapped = {
    'impact modified': 'high impact or impact modified',
    'u.v. stabilized': 'u.v. stabilized or stable to weather',
    'heat stabilized': 'heat stabilized or stable to heat',
    'light stabilized': 'light stabilized or stable to light',
}

In [71]:
feature_df = synonym_df2[synonym_df2['TYPE']=='feature'][['DEFINED_NAME', 'SYNONYMS']]
feature_df['DEFINED_NAME'] = feature_df['DEFINED_NAME'].replace(to_replace=feature_syn_meaning_mapped)
feature_syns = feature_df.groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
feature_syns

{'anti-static': ['anti-static',
  'static resistant',
  'static',
  'static resistance',
  'antistatic',
  'avoid static',
  'avoid static build',
  'avoid static build up'],
 'bio-content': ['bio-content', 'bio-based', 'bio', 'biobased'],
 'carbon capture': ['carbon capture',
  'reduce carbon capture',
  'reduced carbon footprint',
  'eco-c',
  'carbon footprint',
  'iso 14067',
  'carbon capture and utilization',
  'ccu',
  'reduce carbon capture',
  'reduced carbon footprint'],
 'chemical resistant': ['chemical resistant',
  'acid resistant',
  'acid resistance',
  'chem resistance',
  'chem resistant',
  'anti chemical',
  'anti acid',
  'anti-chemical',
  'anti-acid',
  'chemical resistance'],
 'flame retardant': ['flame retardant',
  'flameretardent',
  'fr',
  'flame retarding agent',
  'flamret',
  'flamretag',
  'flame resistant',
  'flame resistance',
  'halogenated fr',
  'halogenated',
  'halogenated flame retardant'],
 'heat stabilized or stable to heat': ['heat stabilized

In [72]:
feature_syn_mapping = {}
for f_meaning, f_syns in feature_syns.items():
    for syn in f_syns:
        feature_syn_mapping[syn] = f_meaning

feature_syn_mapping

{'anti-static': 'anti-static',
 'static resistant': 'anti-static',
 'static': 'anti-static',
 'static resistance': 'anti-static',
 'antistatic': 'anti-static',
 'avoid static': 'anti-static',
 'avoid static build': 'anti-static',
 'avoid static build up': 'anti-static',
 'bio-content': 'bio-content',
 'bio-based': 'bio-content',
 'bio': 'bio-content',
 'biobased': 'bio-content',
 'carbon capture': 'carbon capture',
 'reduce carbon capture': 'carbon capture',
 'reduced carbon footprint': 'carbon capture',
 'eco-c': 'carbon capture',
 'carbon footprint': 'carbon capture',
 'iso 14067': 'carbon capture',
 'carbon capture and utilization': 'carbon capture',
 'ccu': 'carbon capture',
 'chemical resistant': 'chemical resistant',
 'acid resistant': 'chemical resistant',
 'acid resistance': 'chemical resistant',
 'chem resistance': 'chemical resistant',
 'chem resistant': 'chemical resistant',
 'anti chemical': 'chemical resistant',
 'anti acid': 'chemical resistant',
 'anti-chemical': 'chemic

In [73]:
print(len(product_functions))
for i in feature_syn_mapping:
    if i not in product_functions:
        product_functions.append(i)

print(len(product_functions))

41
278


In [74]:
all_features_without_syns = []
for i in product_functions:
    if i in feature_syn_mapping and feature_syn_mapping[i]==i:
        all_features_without_syns.append(i)
    elif i not in feature_syn_mapping:
        all_features_without_syns.append(i)

all_features_without_syns

['high viscosity',
 'lubricants',
 'release agent',
 'high flow',
 'hydrolysis resistant',
 'low wear / low friction',
 'high impact or impact modified',
 'flame retardant',
 'u.v. stabilized or stable to weather',
 'heat stabilized or stable to heat',
 'high gloss',
 'recycled content',
 'sustainable',
 'medical/healthcare',
 'bio-content',
 'nucleated',
 'laser markable',
 'specialty appearance',
 'low warpage',
 'non-halogenated/red phosphorous free flame retardant',
 'static dissipative',
 'anti-static',
 'increased electrical conductivity',
 'chemical resistant',
 'low halide content',
 'increased thermal conductivity',
 'light stabilized or stable to light',
 'improved creep',
 'reduced gloss',
 'carbon capture',
 'low emissions',
 'light weight',
 'lead-free soldering resistant',
 'laser weldable',
 'thermal shock resistant',
 'platable',
 'hydrophilic',
 'improved weld line',
 'ultrasonic weldable',
 'laser direct structurable',
 'plasticizer',
 'hr']

In [75]:
for i in processing_types:
    if i not in processing_syn_mapping:
        print(1, i)


for i in delivery_forms:
    if i not in delivery_syn_mapping:
        print(2, i)

# medical & pharma - drug delivery devices: no synonyms
for i in product_functions:
    if i not in feature_syn_mapping:
        print(3, i)

2 micropowder


### Properties

In [76]:
prop_df = synonym_df2[synonym_df2['TYPE']=='property'][['DEFINED_NAME', 'SYNONYMS']]
prop_df['DEFINED_NAME'] = prop_df['DEFINED_NAME'].apply(lambda x: x.replace(" -  (-)", "").replace(" (-)", "").rstrip(" -") if x else x)
prop_df

,DEFINED_NAME,SYNONYMS
881,average molecular weight margolies' equation (...,average molecular weight
882,average molecular weight margolies' equation (...,average molar mass
883,average molecular weight margolies' equation (...,average molecular mass
884,average molecular weight margolies' equation (...,avermw
885,average molecular weight margolies' equation (...,molecular weight
...,...,...
1432,water absorption sim. to iso 62 2mm (%),moisture absorption equilibrium 23°c/50% r.h.
1433,wear by sandslurry method (based on gur 4120=100),wear by sandslurry method
1484,molding shrinkage,mold shrinkage (md/td)
1489,melt mass-flow rate iso 1133 (g/10min),mi


In [77]:
print(prop_df[prop_df['DEFINED_NAME']=='fiber areal weight (g/m²)'])

prop_df['DEFINED_NAME'] = prop_df['DEFINED_NAME'].replace('fiber areal weight - (g/m²)', 'fiber areal weight (g/m²)')
prop_df['DEFINED_NAME'] = prop_df['DEFINED_NAME'].replace('tape areal weight - (g/m²)', 'tape areal weight (g/m²)')

print(prop_df[prop_df['DEFINED_NAME']=='fiber areal weight (g/m²)'])

                   DEFINED_NAME            SYNONYMS
1045  fiber areal weight (g/m²)  fiber areal weight
                   DEFINED_NAME            SYNONYMS
1045  fiber areal weight (g/m²)  fiber areal weight


In [78]:
property_syns = prop_df.groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
property_syns

{"average molecular weight margolies' equation (g/mol)": ['average molecular weight',
  'average molar mass',
  'average molecular mass',
  'avermw',
  'molecular weight',
  'mw'],
 'average particle size laser scattering d50 (µm)': ['average particle size laser scattering d50 (µm)',
  'average particle size',
  'avg. particle size',
  'avpartsize',
  'particle size',
  'particle size d50'],
 'ball indentation hardness iso 2039-1 h 358/30 (mpa)': ['ball indentation hardness',
  'ball indention hardness'],
 'ball pressure test iec 60695-10-2 (°c)': ['ball pressure test'],
 'bulk density iso 60 (kg/m³)': ['bulk density',
  'apparent (bulk) density',
  'apparent density',
  'bulkdens'],
 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)': ['charpy double notched impact strength',
  'charpy impact strength (double notched)',
  'charpy impact strength (double notch)',
  'double notch charpy impact strength',
  'double notched charpy impact strength',
  'double notch charpy',
 

In [79]:
list(property_syns)

["average molecular weight margolies' equation (g/mol)",
 'average particle size laser scattering d50 (µm)',
 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
 'ball pressure test iec 60695-10-2 (°c)',
 'bulk density iso 60 (kg/m³)',
 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
 'charpy impact strength',
 'charpy notched impact strength',
 'coefficient of linear thermal expansion (clte)',
 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
 'compression set',
 'compression set iso 815 23°c (%)',
 'compression set iso 815 23°c, 24h (%)',
 'compressive modulus iso 604 (mpa)',
 'compressive strength iso 604 (mpa)',
 'compressive stress at 1% strain iso 604 (mpa)',
 'continuous service temperature iec 60216-1 (°c)',
 'density iso 1183 (kg/m³)',
 'dissipation factor',
 'dissipation factor iec 62631-2-1 100hz (e-4)',
 'dissipation factor iec 62631-2-1 1m

In [80]:
all_property_names = []
for i in unique_values['Property']:
    all_property_names.append(i['PROPERTY_NAME_UI'].lower())

all_property_names2 = []
for i in unique_values['Property']:
    all_property_names2.append(i['PROPERTY_NAME'].lower())
    
for i in list(property_syns):
    if i not in all_property_names:
        # print(i)
        if i not in all_property_names2:
            print(i)
            # pass

shore hardness
thermal conductivity of melt iso 22007-2 (w/(m k))


In [81]:
prop_not_mapped = [
    'shore hardness',
    'thermal conductivity of melt iso 22007-2 (w/(m k))',
]

In [82]:
prop_syn_meaning_mapped = {
    'shore hardness': 'shore hardness',
    'thermal conductivity of melt iso 22007-2 (w/(m k))': 'thermal conductivity iso 22007-2 flow (w/(m k))',
    'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
}

In [83]:
prop_df['DEFINED_NAME'] = prop_df['DEFINED_NAME'].replace(to_replace=prop_syn_meaning_mapped)
property_syns = prop_df.groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
property_syns

{"average molecular weight margolies' equation (g/mol)": ['average molecular weight',
  'average molar mass',
  'average molecular mass',
  'avermw',
  'molecular weight',
  'mw'],
 'average particle size laser scattering d50 (µm)': ['average particle size laser scattering d50 (µm)',
  'average particle size',
  'avg. particle size',
  'avpartsize',
  'particle size',
  'particle size d50'],
 'ball indentation hardness iso 2039-1 h 358/30 (mpa)': ['ball indentation hardness',
  'ball indention hardness'],
 'ball pressure test iec 60695-10-2 (°c)': ['ball pressure test'],
 'bulk density iso 60 (kg/m³)': ['bulk density',
  'apparent (bulk) density',
  'apparent density',
  'bulkdens'],
 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)': ['charpy double notched impact strength',
  'charpy impact strength (double notched)',
  'charpy impact strength (double notch)',
  'double notch charpy impact strength',
  'double notched charpy impact strength',
  'double notch charpy',
 

In [84]:
all_property_names = []
for i in unique_values['Property']:
    all_property_names.append(i['PROPERTY_NAME_UI'].lower())

all_property_names2 = []
for i in unique_values['Property']:
    all_property_names2.append(i['PROPERTY_NAME'].lower())
    
for i in list(property_syns):
    if i not in all_property_names:
        # print(i)
        if i not in all_property_names2:
            print(i)

shore hardness


In [85]:
property_syn_meaning = sorted(list(property_syns), key=len, reverse=True)

# property name mapped to meaning column in synonym table
prop_mapping = {}
for item in unique_values['Property']:
    match_found = False

    if not match_found:
        for i in property_syn_meaning:
            if item['PROPERTY_NAME'].lower() == i or item['PROPERTY_NAME_UI'].lower() == i:
                match_found = True
                meaning = i
                # print(meaning)
    
            if match_found:
                break
                
    if not match_found:          
        for i in property_syn_meaning:
            if i.startswith(item['PROPERTY_NAME'].lower()):
                match_found = True
                meaning = i
                # print(i, item['PROPERTY_NAME'].lower())
        
            if match_found:
                break
        
    if match_found:
        # print(meaning)
        prop_mapping[item['PROPERTY_NAME'].lower()] = meaning.replace(" -  (-)", "").replace(" (-)", "").replace(" -", "")
    else:
        print("no match for:", item['PROPERTY_NAME'].lower())
        prop_mapping[item['PROPERTY_NAME'].lower()] = item['PROPERTY_NAME_UI'].lower()

prop_mapping   

no match for: stress at break
no match for: stress at 10% elongation


{'shore a hardness': 'shore a hardness iso 48-4 / iso 868 15s',
 'electric strength': 'electric strength iec 60243-1 (kv/mm)',
 'intrinsic viscosity': 'intrinsic viscosity iso 307, 1628',
 'vicat softening temperature': 'vicat softening temperature iso 306 50°c/h 10n (°c)',
 'relative permittivity': 'relative permittivity iec 62631-2-1 60hz',
 'tensile strain at failure': 'tensile strain at failure astm d 3039 m tape 0° (%)',
 'wear by sandslurry method (based on gur 4120=100)': 'wear by sandslurry method (based on gur 4120=100)',
 'tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
 'izod notched impact strength': 'izod notched impact strength',
 'stress at break': 'stress at break iso 527-1/-2 (mpa)',
 'tensile strain at break': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
 'puncture energy': 'puncture energy',
 'tensile creep modulus': 'tensile creep modulus iso 899-1 1000h (mpa)',
 'fiber volume content': 'fiber volume conten

In [86]:
prop_mapping['thermal conductivity'] = 'thermal conductivity'
prop_mapping['coefficient of linear thermal expansion (clte)'] = 'coefficient of linear thermal expansion (clte)'
prop_mapping['tensile strain at break'] = 'tensile strain at break'
prop_mapping['tensile creep modulus'] = 'tensile creep modulus'
prop_mapping['dissipation factor'] = 'dissipation factor'
prop_mapping['tensile stress at break'] = 'tensile stress at break'
prop_mapping['molding shrinkage'] = 'molding shrinkage'
prop_mapping['effective thermal diffusivity'] = 'effective thermal diffusivity'
prop_mapping['temperature of deflection under load'] = 'temperature of deflection under load'
prop_mapping['flexural modulus'] = 'flexural modulus'
prop_mapping['hardness, rockwell'] = 'hardness, rockwell'
prop_mapping['relative permittivity'] = 'relative permittivity'
prop_mapping['vicat softening temperature'] = 'vicat softening temperature'

prop_mapping   

{'shore a hardness': 'shore a hardness iso 48-4 / iso 868 15s',
 'electric strength': 'electric strength iec 60243-1 (kv/mm)',
 'intrinsic viscosity': 'intrinsic viscosity iso 307, 1628',
 'vicat softening temperature': 'vicat softening temperature',
 'relative permittivity': 'relative permittivity',
 'tensile strain at failure': 'tensile strain at failure astm d 3039 m tape 0° (%)',
 'wear by sandslurry method (based on gur 4120=100)': 'wear by sandslurry method (based on gur 4120=100)',
 'tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
 'izod notched impact strength': 'izod notched impact strength',
 'stress at break': 'stress at break iso 527-1/-2 (mpa)',
 'tensile strain at break': 'tensile strain at break',
 'puncture energy': 'puncture energy',
 'tensile creep modulus': 'tensile creep modulus',
 'fiber volume content': 'fiber volume content iso 11667 (%)',
 'izod impact strength': 'izod impact strength',
 'oxygen index': 'oxygen

In [87]:
property_and_values =[]
for item in unique_values['Property']:
    if item['UNIT_OF_MEAS_SI']:
        property_and_values.append((prop_mapping[item['PROPERTY_NAME'].lower()], item['Min_Value'], item['Max_Value'], item['UNIT_OF_MEAS_SI'].lower().strip(), item['PROPERTY_NAME'].lower()))
    else: 
        property_and_values.append((prop_mapping[item['PROPERTY_NAME'].lower()], item['Min_Value'], item['Max_Value'], "", item['PROPERTY_NAME'].lower()))

In [88]:
# ('wear by sandslurry method (based on gur 4120=100)', 80, 380, ''), >>>> ???

In [89]:
property_and_values

[('shore a hardness iso 48-4 / iso 868 15s', 91, 91, '', 'shore a hardness'),
 ('electric strength iec 60243-1 (kv/mm)',
  2.4,
  59,
  'kv/mm',
  'electric strength'),
 ('intrinsic viscosity iso 307, 1628', 0.63, 3400, '', 'intrinsic viscosity'),
 ('vicat softening temperature', 60, 290, '°c', 'vicat softening temperature'),
 ('relative permittivity', 2.7, 32, '', 'relative permittivity'),
 ('shore a hardness iso 48-4 / iso 868 15s', 28, 98, '', 'shore a hardness'),
 ('tensile strain at failure astm d 3039 m tape 0° (%)',
  1.76,
  3,
  '%',
  'tensile strain at failure'),
 ('wear by sandslurry method (based on gur 4120=100)',
  80,
  380,
  '',
  'wear by sandslurry method (based on gur 4120=100)'),
 ('tensile stress at 100% elongation iso 37 perpendicular (mpa)',
  0.7,
  9.9,
  'mpa',
  'tensile stress at 100% elongation'),
 ('izod notched impact strength',
  2,
  64,
  'kj/m²',
  'izod notched impact strength'),
 ('stress at break iso 527-1/-2 (mpa)', 9, 50, 'mpa', 'stress at brea

In [90]:
property_and_values += [
    ('tensile modulus astm d 3039 m tape 0° (mpa)', 25.7, 101, 'gpa', 'tape tens mod'), 
    ('tensile modulus astm d 3039 m tape 0° (mpa)', 25.7, 101, 'gpa', 'tensile mod tape'), 
    ('tensile modulus astm d 3039 m tape 0° (mpa)', 25.7, 101, 'gpa', 'tape tensile modulus'), 
    ('tensile modulus astm d 3039 m tape 0° (mpa)', 25.7, 101, 'gpa', 'tensile modulus tape'), 
    ('tensile modulus astm d 3039 m tape 0° (mpa)', 25.7, 101, 'gpa', 'tensile tape'), 
    
    ('flexural modulus astm d 790 tape 0° (mpa)', 26.9, 105, 'gpa', 'tape flex mod'), 
    ('flexural modulus astm d 790 tape 0° (mpa)', 26.9, 105, 'gpa', 'flex mod tape'), 
    ('flexural modulus astm d 790 tape 0° (mpa)', 26.9, 105, 'gpa', 'tape flexural modulus'), 
    ('flexural modulus astm d 790 tape 0° (mpa)', 26.9, 105, 'gpa', 'flexural modulus tape'), 
    
    ('flexural strength astm d 790 tape 0° (mpa)', 465, 1220, 'mpa', 'tape flex strength'), 
    ('flexural strength astm d 790 tape 0° (mpa)', 465, 1220, 'mpa', 'flex strength tape'), 
    ('flexural strength astm d 790 tape 0° (mpa)', 465, 1220, 'mpa', 'tape flexural strength'), 
    ('flexural strength astm d 790 tape 0° (mpa)', 465, 1220, 'mpa', 'flexural strength tape')]
property_and_values

[('shore a hardness iso 48-4 / iso 868 15s', 91, 91, '', 'shore a hardness'),
 ('electric strength iec 60243-1 (kv/mm)',
  2.4,
  59,
  'kv/mm',
  'electric strength'),
 ('intrinsic viscosity iso 307, 1628', 0.63, 3400, '', 'intrinsic viscosity'),
 ('vicat softening temperature', 60, 290, '°c', 'vicat softening temperature'),
 ('relative permittivity', 2.7, 32, '', 'relative permittivity'),
 ('shore a hardness iso 48-4 / iso 868 15s', 28, 98, '', 'shore a hardness'),
 ('tensile strain at failure astm d 3039 m tape 0° (%)',
  1.76,
  3,
  '%',
  'tensile strain at failure'),
 ('wear by sandslurry method (based on gur 4120=100)',
  80,
  380,
  '',
  'wear by sandslurry method (based on gur 4120=100)'),
 ('tensile stress at 100% elongation iso 37 perpendicular (mpa)',
  0.7,
  9.9,
  'mpa',
  'tensile stress at 100% elongation'),
 ('izod notched impact strength',
  2,
  64,
  'kj/m²',
  'izod notched impact strength'),
 ('stress at break iso 527-1/-2 (mpa)', 9, 50, 'mpa', 'stress at brea

In [91]:
RANGE_MODIFIERS = [
    
    # 'excellent', 'superior', 'exceptional', 'best', 'outstanding', 'very good', 'maximum', 'very high', 'great', 'high', 'highest', 'higher', 
    # 'less', 'least', 'weak', 'lower', 'lowest', 'inferior', 'limited', 'low', 'typical', 
    # 'good', 'fair', 'medium', 'moderate', 'normal', 'ordinary', 'average', 'standard'

    'high', 'highest', 'excellent', 'strong',
    'lowest', 'low', 'weak',
    'medium', 'moderate', 'average', 'standard', 'good',
]
RANGE_MODIFIERS

['high',
 'highest',
 'excellent',
 'strong',
 'lowest',
 'low',
 'weak',
 'medium',
 'moderate',
 'average',
 'standard',
 'good']

In [92]:
set(synonym_df2['TYPE'])

{'auto cert',
 'brand',
 'delivery form',
 'feature',
 'filler',
 'others',
 'polymer',
 'processing',
 'property',
 'ul property'}

In [93]:
list(property_syns)

["average molecular weight margolies' equation (g/mol)",
 'average particle size laser scattering d50 (µm)',
 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
 'ball pressure test iec 60695-10-2 (°c)',
 'bulk density iso 60 (kg/m³)',
 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
 'charpy impact strength',
 'charpy notched impact strength',
 'coefficient of linear thermal expansion (clte)',
 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
 'compression set',
 'compression set iso 815 23°c (%)',
 'compression set iso 815 23°c, 24h (%)',
 'compressive modulus iso 604 (mpa)',
 'compressive strength iso 604 (mpa)',
 'compressive stress at 1% strain iso 604 (mpa)',
 'continuous service temperature iec 60216-1 (°c)',
 'density iso 1183 (kg/m³)',
 'dissipation factor',
 'dissipation factor iec 62631-2-1 100hz (e-4)',
 'dissipation factor iec 62631-2-1 1m

In [94]:
syn_property_and_values = []
for i in list(property_syns):       
    matched = False
    for j in unique_values['Property']:
        if i==j['PROPERTY_NAME_UI'].lower() and not matched:
            matched = True
            # print(1, i, " | ", j['PROPERTY_NAME_UI'])
            # print(1, (j['PROPERTY_NAME_UI'].lower(), j['Min_Value'], j['Max_Value'], j['UNIT_OF_MEAS_SI'], j['PROPERTY_NAME_UI'].lower()))
            if j['UNIT_OF_MEAS_SI']:
                syn_property_and_values.append((j['PROPERTY_NAME_UI'].lower(), j['Min_Value'], j['Max_Value'], j['UNIT_OF_MEAS_SI'], j['PROPERTY_NAME_UI'].lower()))
            else:
                syn_property_and_values.append((j['PROPERTY_NAME_UI'].lower(), j['Min_Value'], j['Max_Value'], "", j['PROPERTY_NAME_UI'].lower()))
            
            for p_syn in property_syns[i]:
                if p_syn:
                    if j['UNIT_OF_MEAS_SI']:
                        syn_property_and_values.append((j['PROPERTY_NAME_UI'].lower(), j['Min_Value'], j['Max_Value'], j['UNIT_OF_MEAS_SI'], p_syn))
                    else:
                        syn_property_and_values.append((j['PROPERTY_NAME_UI'].lower(), j['Min_Value'], j['Max_Value'], "", p_syn))

            break
            

    if matched:
        # print(i)
        pass
    else:
        # print(i)
        matched_prop = process.extract(i, all_property_names2)[0][0]
        matched2 = False
        for j in property_and_values:
            if matched_prop==j[0] and not matched2:
                matched2 = True
                # print(2, i, " | ", j[0])
                # print(2, (i, j[1], j[2], j[3], i))
                syn_property_and_values.append((i, j[1], j[2], j[3], i))               
                for p_syn in property_syns[i]:
                    if p_syn:
                        # print((i, j[1], j[2], j[3], p_syn))
                        syn_property_and_values.append((i, j[1], j[2], j[3], p_syn))

                break

        if not matched2:
            matched_prop = process.extract(i, all_property_names2)[0][0]
            for j in property_and_values:
                if matched_prop in j[0] and not matched2:
                    matched2 = True
                    # print(3, i, " | ", j[0])
                    # print(2, (i, j[1], j[2], j[3], i))
                    syn_property_and_values.append((i, j[1], j[2], j[3], i))               
                    for p_syn in property_syns[i]:
                        if p_syn:
                            # print((i, j[1], j[2], j[3], p_syn))
                            syn_property_and_values.append((i, j[1], j[2], j[3], p_syn))
    
                    break
        
        if not matched2:
            print(i)
            pass
            
syn_property_and_values

[("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  "average molecular weight margolies' equation (g/mol)"),
 ("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  'average molecular weight'),
 ("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  'average molar mass'),
 ("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  'average molecular mass'),
 ("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  'avermw'),
 ("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  'molecular weight'),
 ("average molecular weight margolies' equation (g/mol)",
  240000,
  10200000,
  'g/mol',
  'mw'),
 ('average particle size laser scattering d50 (µm)',
  20,
  450,
  'µm',
  'average particle size laser scattering d50 (µm)'),
 ('average particle size laser scattering d50 

In [95]:
property_and_values

[('shore a hardness iso 48-4 / iso 868 15s', 91, 91, '', 'shore a hardness'),
 ('electric strength iec 60243-1 (kv/mm)',
  2.4,
  59,
  'kv/mm',
  'electric strength'),
 ('intrinsic viscosity iso 307, 1628', 0.63, 3400, '', 'intrinsic viscosity'),
 ('vicat softening temperature', 60, 290, '°c', 'vicat softening temperature'),
 ('relative permittivity', 2.7, 32, '', 'relative permittivity'),
 ('shore a hardness iso 48-4 / iso 868 15s', 28, 98, '', 'shore a hardness'),
 ('tensile strain at failure astm d 3039 m tape 0° (%)',
  1.76,
  3,
  '%',
  'tensile strain at failure'),
 ('wear by sandslurry method (based on gur 4120=100)',
  80,
  380,
  '',
  'wear by sandslurry method (based on gur 4120=100)'),
 ('tensile stress at 100% elongation iso 37 perpendicular (mpa)',
  0.7,
  9.9,
  'mpa',
  'tensile stress at 100% elongation'),
 ('izod notched impact strength',
  2,
  64,
  'kj/m²',
  'izod notched impact strength'),
 ('stress at break iso 527-1/-2 (mpa)', 9, 50, 'mpa', 'stress at brea

In [96]:
print(len(property_and_values))
for i in syn_property_and_values:
    if i not in property_and_values:
        property_and_values.append(i)

print(len(property_and_values))

property_and_values

118
716


[('shore a hardness iso 48-4 / iso 868 15s', 91, 91, '', 'shore a hardness'),
 ('electric strength iec 60243-1 (kv/mm)',
  2.4,
  59,
  'kv/mm',
  'electric strength'),
 ('intrinsic viscosity iso 307, 1628', 0.63, 3400, '', 'intrinsic viscosity'),
 ('vicat softening temperature', 60, 290, '°c', 'vicat softening temperature'),
 ('relative permittivity', 2.7, 32, '', 'relative permittivity'),
 ('shore a hardness iso 48-4 / iso 868 15s', 28, 98, '', 'shore a hardness'),
 ('tensile strain at failure astm d 3039 m tape 0° (%)',
  1.76,
  3,
  '%',
  'tensile strain at failure'),
 ('wear by sandslurry method (based on gur 4120=100)',
  80,
  380,
  '',
  'wear by sandslurry method (based on gur 4120=100)'),
 ('tensile stress at 100% elongation iso 37 perpendicular (mpa)',
  0.7,
  9.9,
  'mpa',
  'tensile stress at 100% elongation'),
 ('izod notched impact strength',
  2,
  64,
  'kj/m²',
  'izod notched impact strength'),
 ('stress at break iso 527-1/-2 (mpa)', 9, 50, 'mpa', 'stress at brea

In [97]:
property_and_values

[('shore a hardness iso 48-4 / iso 868 15s', 91, 91, '', 'shore a hardness'),
 ('electric strength iec 60243-1 (kv/mm)',
  2.4,
  59,
  'kv/mm',
  'electric strength'),
 ('intrinsic viscosity iso 307, 1628', 0.63, 3400, '', 'intrinsic viscosity'),
 ('vicat softening temperature', 60, 290, '°c', 'vicat softening temperature'),
 ('relative permittivity', 2.7, 32, '', 'relative permittivity'),
 ('shore a hardness iso 48-4 / iso 868 15s', 28, 98, '', 'shore a hardness'),
 ('tensile strain at failure astm d 3039 m tape 0° (%)',
  1.76,
  3,
  '%',
  'tensile strain at failure'),
 ('wear by sandslurry method (based on gur 4120=100)',
  80,
  380,
  '',
  'wear by sandslurry method (based on gur 4120=100)'),
 ('tensile stress at 100% elongation iso 37 perpendicular (mpa)',
  0.7,
  9.9,
  'mpa',
  'tensile stress at 100% elongation'),
 ('izod notched impact strength',
  2,
  64,
  'kj/m²',
  'izod notched impact strength'),
 ('stress at break iso 527-1/-2 (mpa)', 9, 50, 'mpa', 'stress at brea

In [98]:
all_property_names3 = []
prop_abrv = []
for i in property_and_values:
    all_property_names3.append(i[-1])
    if len(i[-1]) <= 4:
        prop_abrv.append(i[-1])

set(all_property_names3)

{'100hz df',
 '100hz dk',
 '1mhz df',
 '1mhz dk',
 '60hz dk',
 'a',
 'ac',
 'ac dielectric strength',
 'ac electristrength',
 'apparent (bulk) density',
 'apparent density',
 'average molar mass',
 'average molecular mass',
 'average molecular weight',
 "average molecular weight margolies' equation (g/mol)",
 'average particle size',
 'average particle size laser scattering d50 (µm)',
 'avermw',
 'avg. particle size',
 'avpartsize',
 'ball indentation hardness',
 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
 'ball indention hardness',
 'ball pressure test',
 'ball pressure test iec 60695-10-2 (°c)',
 'bend strength',
 'break',
 'break elongation 5 mm/min',
 'break elongation 50 mm/min',
 'break elongation 50mm/min',
 'break elongation 5mm/min',
 'break strain 5 mm/min',
 'break strain 50 mm/min',
 'break strain 50mm/min',
 'break strain 5mm/min',
 'break strain, break elongation',
 'break stress',
 'break stress at 50mm/min',
 'break stress at 5mm/min',
 'bulk density',
 'bul

In [99]:
print(prop_abrv)

['mw', 'clte', 'cut', 'df', 'ac', 'flex', 'tg', 'η', 'mfr', 'mfi', 'flow', 'mi', 'mvr', 'mv', 'mp', 'loi', 'dk', 'sh a', 'a', 'soft', 'sh d', 'd', 'sr', 'dtul', 'hdt', 'emod', 'vr']


In [100]:
all_filler_syns = []
for i in filler_syns:
    all_filler_syns.extend(filler_syns[i])

# prop_abrv = [
#     'bpt', 'bsa', 'bsb', 'bse', 'bsl', 'bspa', 'clte', 'cte', 'dc', 'df', 'dk', 'dr', 'dres', 'ds', 'dtul', 'emod', 'eoc', 'eov',
#     'flow', 'fs', 'hdt', 'izod', 'loi', 'lsa', 'lsb', 'lse', 'lsl', 'mfi', 'mfr', 'mrc', 'mv', 'mvr', 
#     'mw', 'sg', 'sr', 'tg', 'wsa', 'wsb', 'wse', 'wsl', 'wvtr', 'fm', 'fs', 'ts', 'sha'
# ]


print(len(prop_abrv))

# prop_abrv = [i for i in prop_abrv if i not in all_ulp_syns]
prop_abrv = [i for i in prop_abrv if i not in all_brands]
prop_abrv = [i for i in prop_abrv if i not in all_filler_syns]

print(len(prop_abrv))

prop_abrv_mapping = {}
for i in prop_df[prop_df['SYNONYMS'].isin(prop_abrv)][['DEFINED_NAME', 'SYNONYMS']].to_dict('split')['data']:
    matched_prop = process.extract(i[0], all_property_names3)[0][0]
    # print(i[1], i[0], matched_prop)
    prop_abrv_mapping[i[1]] = matched_prop
    
prop_abrv_mapping   

27
27


{'mw': "average molecular weight margolies' equation (g/mol)",
 'clte': 'coefficient of linear thermal expansion (clte)',
 'cut': 'continuous service temperature iec 60216-1 (°c)',
 'df': 'dissipation factor',
 'ac': 'electric strength iec 60243-1 (kv/mm)',
 'flex': 'flexural modulus',
 'tg': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
 'η': 'intrinsic viscosity iso 307, 1628',
 'mfr': 'melt mass-flow rate iso 1133 (g/10min)',
 'mfi': 'melt mass-flow rate iso 1133 (g/10min)',
 'flow': 'melt mass-flow rate iso 1133 (g/10min)',
 'mvr': 'melt volume-flow rate iso 1133 (cm³/10min)',
 'mv': 'melt volume-flow rate iso 1133 (cm³/10min)',
 'mp': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
 'loi': 'oxygen index iso 4589-1/-2 (%)',
 'dk': 'relative permittivity',
 'sh a': 'shore a hardness iso 48-4 / iso 868 15s',
 'a': 'shore a hardness iso 48-4 / iso 868 15s',
 'soft': 'shore a hardness iso 48-4 / iso 868 15s',
 'sh d': 'shore d hardness iso 48-4 / iso 868 15s',
 'd': 

In [101]:
a = list(feature_syn_mapping) + list(brand_syn_mapping) + list(polymer_syn_mapping) + list(processing_syn_mapping) + list(delivery_syn_mapping)
for i in all_property_names3:
    if i in a:
        print(i)


# all_property_names3

In [102]:
# for i in ['mw', 'cut', 'ac', 'tg', 'η', 'mfr', 'mfi', 'flow', 'mvr', 'mv', 'mp', 'loi', 'sh a', 'a', 'soft', 'sh a', 'a', 'soft', 'sh d', 'd', 'hard', 'sr', 'dtul', 'hdt', 'dtul', 'hdt', 'dtul', 'hdt', 'vr']:
#     if i not in list(prop_abrv_mapping):
#         print(i)

# print(len(set(prop_abrv)), len(prop_abrv_mapping))

In [103]:
prop_abrv_mapping

{'mw': "average molecular weight margolies' equation (g/mol)",
 'clte': 'coefficient of linear thermal expansion (clte)',
 'cut': 'continuous service temperature iec 60216-1 (°c)',
 'df': 'dissipation factor',
 'ac': 'electric strength iec 60243-1 (kv/mm)',
 'flex': 'flexural modulus',
 'tg': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
 'η': 'intrinsic viscosity iso 307, 1628',
 'mfr': 'melt mass-flow rate iso 1133 (g/10min)',
 'mfi': 'melt mass-flow rate iso 1133 (g/10min)',
 'flow': 'melt mass-flow rate iso 1133 (g/10min)',
 'mvr': 'melt volume-flow rate iso 1133 (cm³/10min)',
 'mv': 'melt volume-flow rate iso 1133 (cm³/10min)',
 'mp': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
 'loi': 'oxygen index iso 4589-1/-2 (%)',
 'dk': 'relative permittivity',
 'sh a': 'shore a hardness iso 48-4 / iso 868 15s',
 'a': 'shore a hardness iso 48-4 / iso 868 15s',
 'soft': 'shore a hardness iso 48-4 / iso 868 15s',
 'sh d': 'shore d hardness iso 48-4 / iso 868 15s',
 'd': 

In [104]:
property_and_values

[('shore a hardness iso 48-4 / iso 868 15s', 91, 91, '', 'shore a hardness'),
 ('electric strength iec 60243-1 (kv/mm)',
  2.4,
  59,
  'kv/mm',
  'electric strength'),
 ('intrinsic viscosity iso 307, 1628', 0.63, 3400, '', 'intrinsic viscosity'),
 ('vicat softening temperature', 60, 290, '°c', 'vicat softening temperature'),
 ('relative permittivity', 2.7, 32, '', 'relative permittivity'),
 ('shore a hardness iso 48-4 / iso 868 15s', 28, 98, '', 'shore a hardness'),
 ('tensile strain at failure astm d 3039 m tape 0° (%)',
  1.76,
  3,
  '%',
  'tensile strain at failure'),
 ('wear by sandslurry method (based on gur 4120=100)',
  80,
  380,
  '',
  'wear by sandslurry method (based on gur 4120=100)'),
 ('tensile stress at 100% elongation iso 37 perpendicular (mpa)',
  0.7,
  9.9,
  'mpa',
  'tensile stress at 100% elongation'),
 ('izod notched impact strength',
  2,
  64,
  'kj/m²',
  'izod notched impact strength'),
 ('stress at break iso 527-1/-2 (mpa)', 9, 50, 'mpa', 'stress at brea

In [105]:
all_units = []
for i in property_and_values:
    if i[3].lower() not in all_units:
        all_units.append(i[3].lower())
        
all_units

['',
 'kv/mm',
 '°c',
 '%',
 'mpa',
 'kj/m²',
 'j',
 'm²/s',
 'ohm',
 'mm',
 'kg/m³',
 'e-4',
 'g/mol',
 'g/m²',
 'cm³/10min',
 'w/(m k)',
 'n',
 'ohm.m',
 'j/(kg k)',
 'cm³/g',
 'µm',
 'g/10min',
 'kn/m',
 'e-6/k',
 'gpa']

In [106]:
unit_syns = {
    'mpa':  ['megapascal', 'kilopascal', 'kpa', 'pascal'], 
    '%': ['percent', 'percentage'], 
    'kj/m²': ['kj/m2', 'kilojoule per square meter', 'joule per square meter', 'j/m²', 'j/m2'], 
    '°c': ['degrees celsius', 'celsius', 'c', 'centigrade', 'fahrenheit', '°f', 'f', 'kelvin', 'k'], 
    'kv/mm': ['kilovolts per millimeter', 'volts per mil', 'v/mm'], 
    'gpa': ['gigapascal', 'gpa'], 
    'kg/m³': ['kilograms per cubic meter', 'grams per cubic centimeter', 'g/cm³'], 
}

for k in unit_syns:
    unit_syns[k] = [x.lower().strip() for x in unit_syns[k]]
    
for i in unit_syns:
    found = False
    if i not in all_units:
        for j in unit_syns[i]:
            if j in all_units:
                found = True
    else:
        found = True
    
    if not found:
        print(i)

In [107]:
def get_random_percent(v):
    range_percent = random.choice([5, 10, 15, 20, 25, 30])
    min_value = v*(1+ (-range_percent/100))
    max_value = v*(1+ (range_percent/100))
    return range_percent, min_value, max_value

get_random_percent(random.randint(10, 1000))

(5, 438.9, 485.1)

In [108]:
def get_random_range(value):
    # Generate random min and max
    start = float(value*0.5)
    end = float(value*1.5)
    min_value = round(random.uniform(start, value*0.9), 2)
    max_value = round(random.uniform(value*1.1, end), 2)
    if not -10 < min_value < 10:
        min_value = int(min_value)
        max_value = int(max_value)
        
    return min_value, max_value

In [109]:
temp = []
for i in property_and_values:
    if not i[-1].replace(" ", "") in temp:
        temp.append(i[-1].replace(" ", ""))
		
temp

['shoreahardness',
 'electricstrength',
 'intrinsicviscosity',
 'vicatsofteningtemperature',
 'relativepermittivity',
 'tensilestrainatfailure',
 'wearbysandslurrymethod(basedongur4120=100)',
 'tensilestressat100%elongation',
 'izodnotchedimpactstrength',
 'stressatbreak',
 'tensilestrainatbreak',
 'punctureenergy',
 'tensilecreepmodulus',
 'fibervolumecontent',
 'izodimpactstrength',
 'oxygenindex',
 'shoredhardness',
 'effectivethermaldiffusivity',
 'surfaceresistivity',
 'tensilestrainatyield',
 'flexuralstrength',
 'ballindentationhardness',
 'flexuralmodulus',
 'tensilestressatyield',
 'tapethickness',
 'density',
 'dissipationfactor',
 'tensilestressat50%strain',
 'tensilestressat100%strain',
 'averagemolecularweight',
 'stressat10%elongation',
 'continuousservicetemperature',
 'tensilenotchedimpactstrength',
 'tensilestressatbreak',
 'waterabsorption',
 'compressionset',
 'fiberarealweight',
 'charpyimpactstrength',
 'moldingshrinkage',
 'temperatureofdeflectionunderload',
 'cha

#### For Abbreviations

In [110]:
def get_property_abb():
    abrv = random.choice(list(prop_abrv_mapping))
    abrv_meaning = prop_abrv_mapping[abrv]
    value = None
    unit = ""
    exponential = ''
    for i in property_and_values:
        if abrv_meaning == i[0] or abrv_meaning == i[-1]:
            try:    
                value = random.randrange(i[1],i[2])
            except: 
                value = round(random.uniform(i[1], i[2]),2)

            if value > 89999999:
                # print("*"*20, "{:.2e}".format(value))
                float_value = value
                value = "{:.2e}".format(value)
                exponential = 'positive'

            while value==0.0:
                value = round(random.uniform(i[1], i[2]),9)
                # print("$"*20, f"value was 0.0, picking another random number {i[1]} to {i[2]}", value)
                # print("#"*20, "{:.8f}".format(value))
                if -0.99 < value < 0.99:
                    float_value = value
                    value = "{:.2e}".format(value)
                    # value = "{:.8f}".format(value)
                    exponential = 'negative'
                else:
                    value = round(value, 2)
                    
            unit = random.choice([i[3],f" {i[3]}", ""])
            if not unit.strip():
                unit=""
        
            break
            
    if not value:
        # print(f"could not find the value for {abrv_meaning}: {abrv}")
        value = random.randint(20, 200)

    queries = [       
        f"{abrv} of {value}{unit}",
        f"{abrv} {value}{unit}",
        f"{abrv} ({value}{unit})",
        f"{abrv} ={value}{unit}",
        f"{abrv} = {value}{unit}",
        f"{abrv} - {value}{unit}",
        f"{abrv}: {value}{unit}",
        f"{value}{unit} {abrv}",
    ]

    query = random.choice(queries)
    
    if not " " in abrv and not exponential:
        query2 = random.choice([
            f"{abrv}{value}",
            f"{abrv}-{value}",
            f"{value}{abrv}",
            f"{abrv}={value}",
        ])
        is_unit = random.choice([True, True, True, False, False])
        if not is_unit:
            query = query2
            unit = ''
        
    if exponential == 'positive':
        # print("#"*25, 'positive')
        if float_value < 0:
            property_details = {'property_name': abrv_meaning, 'modifier': {'value': "{:.2e}".format(round(float_value, 2)), 'min': "{:.2e}".format(round(float_value*1.1, 2)), 'max': "{:.2e}".format(round(float_value*0.9, 2)), 'unit': unit.strip()}, 'property_type': 'property'}
        else:
            property_details = {'property_name': abrv_meaning, 'modifier': {'value': "{:.2e}".format(round(float_value, 2)), 'min': "{:.2e}".format(round(float_value*0.9, 2)), 'max': "{:.2e}".format(round(float_value*1.1, 2)), 'unit': unit.strip()}, 'property_type': 'property'}
    elif exponential == 'negative':
        print("*"*25, 'negative exponential')
        if float_value < 0:
            property_details = {'property_name': abrv_meaning, 'modifier': {'value': "{:.2e}".format(round(float_value, 9)), 'min': "{:.2e}".format(round(float_value*1.1, 9)), 'max': "{:.2e}".format(round(float_value*0.9, 9)), 'unit': unit.strip()}, 'property_type': 'property'}
        else:
            property_details = {'property_name': abrv_meaning, 'modifier': {'value': "{:.2e}".format(round(float_value, 9)), 'min': "{:.2e}".format(round(float_value*0.9, 9)), 'max': "{:.2e}".format(round(float_value*1.1, 9)), 'unit': unit.strip()}, 'property_type': 'property'}
        print(property_details, query)
    else:
        if value < 0:
            property_details = {'property_name': abrv_meaning, 'modifier': {'value': str(round(value, 2)), 'min': str(round(value*1.1, 2)), 'max': str(round(value*0.9, 2)), 'unit': unit.strip()}, 'property_type': 'property'}
        else:
            property_details = {'property_name': abrv_meaning, 'modifier': {'value': str(round(value, 2)), 'min': str(round(value*0.9, 2)), 'max': str(round(value*1.1, 2)), 'unit': unit.strip()}, 'property_type': 'property'}
    return property_details, query

In [111]:
for i in range(20):
    print(get_property_abb(), "\n")

({'property_name': 'melt mass-flow rate iso 1133 (g/10min)', 'modifier': {'value': '118.77', 'min': '106.89', 'max': '130.65', 'unit': ''}, 'property_type': 'property'}, 'flow 118.77') 

({'property_name': 'shore a hardness iso 48-4 / iso 868 15s', 'modifier': {'value': '91.0', 'min': '81.9', 'max': '100.1', 'unit': ''}, 'property_type': 'property'}, 'soft-91.0') 

({'property_name': 'melt volume-flow rate iso 1133 (cm³/10min)', 'modifier': {'value': '9.54', 'min': '8.59', 'max': '10.49', 'unit': ''}, 'property_type': 'property'}, 'mv 9.54') 

({'property_name': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)', 'modifier': {'value': '-53', 'min': '-58.3', 'max': '-47.7', 'unit': ''}, 'property_type': 'property'}, 'tg--53') 

({'property_name': 'melt mass-flow rate iso 1133 (g/10min)', 'modifier': {'value': '195.14', 'min': '175.63', 'max': '214.65', 'unit': ''}, 'property_type': 'property'}, 'flow195.14') 

({'property_name': 'intrinsic viscosity iso 307, 1628', 'modifier': 

C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\1632742568.py:10: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(i[1],i[2])


In [112]:
prop_syn_mapping = {}
for i in property_and_values:
    prop_syn_mapping[i[-1]] = i[0]

prop_syn_mapping

{'shore a hardness': 'shore a hardness iso 48-4 / iso 868 15s',
 'electric strength': 'electric strength iec 60243-1 (kv/mm)',
 'intrinsic viscosity': 'intrinsic viscosity iso 307, 1628',
 'vicat softening temperature': 'vicat softening temperature',
 'relative permittivity': 'relative permittivity',
 'tensile strain at failure': 'tensile strain at failure astm d 3039 m tape 0° (%)',
 'wear by sandslurry method (based on gur 4120=100)': 'wear by sandslurry method (based on gur 4120=100)',
 'tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
 'izod notched impact strength': 'izod notched impact strength',
 'stress at break': 'stress at break iso 527-1/-2 (mpa)',
 'tensile strain at break': 'tensile strain at break',
 'puncture energy': 'puncture energy',
 'tensile creep modulus': 'tensile creep modulus',
 'fiber volume content': 'fiber volume content iso 11667 (%)',
 'izod impact strength': 'izod impact strength',
 'oxygen index': 'oxygen

In [113]:
prop_abrv_mapping
for k, v in prop_abrv_mapping.items():
    prop_syn_mapping[k] = v
    print(k, ":", v)

mw : average molecular weight margolies' equation (g/mol)
clte : coefficient of linear thermal expansion (clte)
cut : continuous service temperature iec 60216-1 (°c)
df : dissipation factor
ac : electric strength iec 60243-1 (kv/mm)
flex : flexural modulus
tg : glass transition temperature iso 11357-1/-3 10°c/min (°c)
η : intrinsic viscosity iso 307, 1628
mfr : melt mass-flow rate iso 1133 (g/10min)
mfi : melt mass-flow rate iso 1133 (g/10min)
flow : melt mass-flow rate iso 1133 (g/10min)
mvr : melt volume-flow rate iso 1133 (cm³/10min)
mv : melt volume-flow rate iso 1133 (cm³/10min)
mp : melting temperature iso 11357-1/-3 10°c/min (°c)
loi : oxygen index iso 4589-1/-2 (%)
dk : relative permittivity
sh a : shore a hardness iso 48-4 / iso 868 15s
a : shore a hardness iso 48-4 / iso 868 15s
soft : shore a hardness iso 48-4 / iso 868 15s
sh d : shore d hardness iso 48-4 / iso 868 15s
d : shore d hardness iso 48-4 / iso 868 15s
dtul : temperature of deflection under load
hdt : temperature 

In [114]:
prop_syn_mapping

{'shore a hardness': 'shore a hardness iso 48-4 / iso 868 15s',
 'electric strength': 'electric strength iec 60243-1 (kv/mm)',
 'intrinsic viscosity': 'intrinsic viscosity iso 307, 1628',
 'vicat softening temperature': 'vicat softening temperature',
 'relative permittivity': 'relative permittivity',
 'tensile strain at failure': 'tensile strain at failure astm d 3039 m tape 0° (%)',
 'wear by sandslurry method (based on gur 4120=100)': 'wear by sandslurry method (based on gur 4120=100)',
 'tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
 'izod notched impact strength': 'izod notched impact strength',
 'stress at break': 'stress at break iso 527-1/-2 (mpa)',
 'tensile strain at break': 'tensile strain at break',
 'puncture energy': 'puncture energy',
 'tensile creep modulus': 'tensile creep modulus',
 'fiber volume content': 'fiber volume content iso 11667 (%)',
 'izod impact strength': 'izod impact strength',
 'oxygen index': 'oxygen

#### For Property

In [115]:
def get_property():
    min_value = None
    max_value = None
    unit = ''
    exponential = ''
    p_metadata = random.choice(property_and_values)
    property_name = p_metadata[-1].lower()
    property_name_meaning = p_metadata[0].lower()
    range_type = random.choice(['value', 'value', 'value', 'min_value', 'max_value', 'value_range', 'range_without_value', 'range_percent'])
    property_details = {'property_name': None, 'modifier': {'value': None, 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}
    single_space_pattern = '(\s+)'
    try:    
        value = random.randrange(p_metadata[1],p_metadata[2])
    except: 
        value = round(random.uniform(p_metadata[1], p_metadata[2]),2)
        
    if p_metadata[3] != '°c':
        value = random.choice([value, value, value, value, value, value, value, value, value, value, value, value, value, value, random.randrange(100000, 1000000000)])

    if value > 899999:
        # print("*"*20, "{:.2e}".format(value))
        float_value = value
        value = "{:.2e}".format(value)
        exponential = 'positive'

    while value==0.0:
        value = round(random.uniform(p_metadata[1], p_metadata[2]), 9)
        print("$"*20, f"value was 0.0 for property range {p_metadata[1]} to {p_metadata[2]}, picking another random number: {value}")
        # print("#"*20, "{:.8f}".format(value))
        if -0.99999999999 < value < 0.99999999999:
            float_value = value
            value = "{:.2e}".format(value)
            # value = "{:.8f}".format(value)
            exponential = 'negative'
        else:
            value = round(value, 2)
            
    unit = random.choice([p_metadata[3],f" {p_metadata[3]}", ""])
    if not unit.strip():
        unit=''
            
    if not value:
        # print(f"could not find the value for {property_name_meaning}: {property_name}")
        value = random.randint(20, 200)
        
    is_unit_syn = random.choice([False, False, True])
    if unit and is_unit_syn and unit in unit_syns :
        # print("unit synonym")
        unit = random.choice(unit_syns[unit])
        
    if range_type=='min_value':
        try:
            if value < 0:
                max_value, min_value = get_random_range(float(value))
            else:
                min_value, max_value = get_random_range(float(value))
        except:
            min_value, max_value = get_random_range(random.randint(20, 2000))
            
        value = None
        max_value = None
        queries = [
            f"{property_name} of min {min_value}{unit}",
            f"{property_name} of minimum {min_value}{unit}",
            f"minimum of {property_name} {min_value}{unit}",
            f"{property_name} min {min_value}{unit}",
            f"{property_name} minimum {min_value}{unit}",           
            f"minimum {property_name} {min_value}{unit}",
            
            f"{property_name} more than {min_value}{unit}",
            f"{property_name} greater than {min_value}{unit}",
            f"{property_name} greater than {min_value}{unit}",
            f"{property_name} more than or eaual to {min_value}{unit}",
            f"{property_name} greater than or equal to {min_value}{unit}",
            f"{property_name} greater than or equal to {min_value}{unit}",
            f"{property_name} >{min_value}{unit}",
            f"{property_name} > {min_value}{unit}",
            f"{property_name} >={min_value}{unit}",
            f"{property_name} >= {min_value}{unit}",
        ]
        query = random.choice(queries)
    elif range_type=='max_value':
        try:
            if value < 0:
                max_value, min_value = get_random_range(float(value))
            else:
                min_value, max_value = get_random_range(float(value))
        except:
            min_value, max_value = get_random_range(random.randint(20, 2000))
            
        value = None
        min_value = None
        queries = [
            f"{property_name} of max {max_value}{unit}",
            f"{property_name} of maximum {max_value}{unit}",
            f"maximum of {property_name} {max_value}{unit}",
            f"{property_name} max {max_value}{unit}",
            f"{property_name} maximum {max_value}{unit}",
            f"maximum {property_name} {max_value}{unit}",
            
            f"{property_name} lesser than {max_value}{unit}",
            f"{property_name} less than {max_value}{unit}",
            f"{property_name} lesser than or equal to {max_value}{unit}",
            f"{property_name} less than or equal to {max_value}{unit}",
            f"{property_name} <{max_value}{unit}",
            f"{property_name} < {max_value}{unit}",
            f"{property_name} <={max_value}{unit}",
            f"{property_name} <= {max_value}{unit}",
        ]
        query = random.choice(queries)   
    else:
        queries = [
            f"{property_name} {value}{unit}",
            f"{property_name} {value}{unit}",

            f"{property_name} of {value}{unit}",
            f"{property_name} = {value}{unit}",
            f"{property_name} - {value}{unit}",
            f"{property_name}: {value}{unit}",
            f"{property_name} ({value}{unit})",
            f"{value}{unit} {property_name}",

            f"{property_name} {value}{unit}",
            f"{property_name} {value}{unit}",

            f"{property_name} of {value}{unit}",
            f"{property_name} = {value}{unit}",
            f"{property_name} - {value}{unit}",
            f"{property_name}: {value}{unit}",
            f"{property_name} ({value}{unit})",
            f"{value}{unit} {property_name}",

            f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} = {value}{unit}",
            f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} ={value}{unit}",

        ]

        if range_type == 'value':
            queries += [
                f"{property_name} of approximately {value}{unit}",
                f"{property_name} of approx {value}{unit}",
                f"{property_name} close to {value}{unit}",
                f"{property_name} around {value}{unit}",
            ]

        query = random.choice(queries)
        if random.choice(range(0, 10)) == 1 and property_name:
            query = f"{property_name.replace(' ', '')}={value}{unit}"
    #         print(f"cant remove space for {property_name}")

        if not " " in property_name and not exponential and value > 0:
            query2 = random.choice([
                f"{property_name}{value}",
                f"{property_name}-{value}",
                f"{value}{property_name}",
                f"{property_name}={value}",
            ])
            is_unit = random.choice([True, True, True, False, False])
            if not is_unit:
                query = query2
                unit = ''

        if not exponential and range_type != 'value' and value > 0:
            min_value, max_value = get_random_range(float(value))
            query3 = random.choice([       
                f"{property_name} of {value}{unit}",
                f"{property_name} {value}{unit}",
                f"{property_name} ({value}{unit})",
                f"{property_name} ={value}{unit}",
                f"{property_name} = {value}{unit}",
                f"{property_name} - {value}{unit}",
                f"{property_name}: {value}{unit}",
            ])

            if range_type == 'value_range': 
                range_query = random.choice([
                    f" from {min_value} to {max_value}",
                    f" from {min_value} - {max_value}",
                    f" from {min_value}-{max_value}",
                    f" between {min_value} to {max_value}",
                    f" between {min_value} - {max_value}",
                    f" between {min_value}-{max_value}",
                    f" between {min_value} and {max_value}",
                    f" in the range of {min_value} to {max_value}",
                    f" in the range of {min_value} - {max_value}",
                    f" in the range of {min_value}-{max_value}",
                ])
                query = query3 + range_query
            elif range_type == 'range_without_value':
                range_query = random.choice([
                    f" from {min_value} to {max_value}",
                    f" from {min_value} - {max_value}",
                    f" from {min_value}-{max_value}",
                    f" {min_value}-{max_value}",
                    f" {min_value} - {max_value}",
                    f" {min_value} to {max_value}",
                    f" between {min_value} to {max_value}",
                    f" between {min_value} - {max_value}",
                    f" between {min_value}-{max_value}",
                    f" between {min_value} and {max_value}",
                    f" in the range of {min_value} to {max_value}",
                    f" in the range of {min_value} - {max_value}",
                    f" in the range of {min_value}-{max_value}",
                ])
                query = f"{property_name}" + range_query + f"{unit}"

            elif range_type == 'range_percent':
                percent, min_value, max_value = get_random_percent(value)
                range_percent_query = random.choice([
                    f" with {percent}% range",
                    f" with +/-{percent}% range",
                    f" of +/-{percent}% range",
                    f" of -/+{percent}% range",
                    f" around {percent}% variation",
                    f" around {percent}% range",
                    f" around {percent}% margin",
    #                 f" close to a {percent}% variation",
    #                 f" close to a {percent}% range",
    #                 f" close to a {percent}% margin",
                    f" in the vicinity of {percent}%",
                    f" in the range of {percent}%",
                    f" within a range of {percent}%",
                    f" within {percent}% range",
                    f" within {percent}% margin",
                    ])

                query = query3 + range_percent_query
            else:
                print("@"*40, range_type)
        else:
            range_type = 'value'
        
    property_details['property_name'] = property_name_meaning
  
    property_details['modifier']['unit'] = unit.strip().lower()
    if not property_details['modifier']['unit']:
        property_details['modifier']['unit'] = None

    # no range queries for exponential queries
    if range_type == 'min_value':
        property_details['modifier']['min'] =  str(round(min_value, 2))
        property_details['modifier']['max'] =  None
        property_details['modifier']['value'] = property_details['modifier']['min']

    elif range_type == 'max_value':
        property_details['modifier']['max'] = str(round(max_value, 2))
        property_details['modifier']['min'] =  None
        property_details['modifier']['value'] = property_details['modifier']['max']

    elif range_type in ['value_range', 'range_percent']:
        property_details['modifier']['value'] = str(round(value, 2))
        property_details['modifier']['min'] = str(round(min_value, 2))
        property_details['modifier']['max'] = str(round(max_value, 2))
        
    elif range_type == 'range_without_value':
        property_details['modifier']['min'] = str(round(min_value, 2))
        property_details['modifier']['max'] = str(round(max_value, 2))
        property_details['modifier']['value'] = property_details['modifier']['min'] 
    else:
        if value:
            if exponential == 'positive':
                # print("#"*25, 'positive')
                if float_value < 0:
                    property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 2))
                    property_details['modifier']['max'] = "{:.2e}".format(round(float_value*0.9, 2))
                    property_details['modifier']['min'] = "{:.2e}".format(round(float_value*1.1, 2))
                else:
                    property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 2))
                    property_details['modifier']['min'] = "{:.2e}".format(round(float_value*0.9, 2))
                    property_details['modifier']['max'] = "{:.2e}".format(round(float_value*1.1, 2))
                    
            elif exponential == 'negative':
                print("*"*25, 'negative exponential')
                if float_value < 0:
                    property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 9))
                    property_details['modifier']['max'] = "{:.2e}".format(round(float_value*0.9, 9))
                    property_details['modifier']['min'] = "{:.2e}".format(round(float_value*1.1, 9))
                else:
                    property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 9))
                    property_details['modifier']['min'] = "{:.2e}".format(round(float_value*0.9, 9))
                    property_details['modifier']['max'] = "{:.2e}".format(round(float_value*1.1, 9))
                print(property_details)
                
            else:
                if value < 0:
                    property_details['modifier']['value'] = str(round(value, 2))
                    property_details['modifier']['max'] = str(round(value*0.9, 2))
                    property_details['modifier']['min'] = str(round(value*1.1, 2))
                else:
                    property_details['modifier']['value'] = str(round(value, 2))
                    property_details['modifier']['min'] = str(round(value*0.9, 2))
                    property_details['modifier']['max'] = str(round(value*1.1, 2))
        else:
            property_details['modifier']['value'] = None
        
    query = re.sub("(\s+)", " ", query.strip().lower())

    return property_details, query.replace("young s", "young's")

In [116]:
for i in range(1, 20):
    print(get_property(), "\n")

({'property_name': 'melt volume-flow rate iso 1133 (cm³/10min)', 'modifier': {'value': '97', 'min': '97', 'max': '227', 'unit': None}, 'property_type': 'property'}, 'mv in the range of 97 to 227') 

({'property_name': 'melt volume-flow rate iso 1133 (cm³/10min)', 'modifier': {'value': '135', 'min': '135', 'max': None, 'unit': 'cm³/10min'}, 'property_type': 'property'}, 'melt volume rate greater than 135cm³/10min') 

({'property_name': 'melting temperature iso 11357-1/-3 10°c/min (°c)', 'modifier': {'value': '319', 'min': '287.1', 'max': '350.9', 'unit': None}, 'property_type': 'property'}, 'dsc melting temp 319') 

({'property_name': 'tensile strain at break iso 527-1/-2 50mm/min (%)', 'modifier': {'value': '85.05', 'min': '68.04', 'max': '102.06', 'unit': None}, 'property_type': 'property'}, 'elongation break 50 mm/min - 85.05 within 20% margin') 

({'property_name': 'dissipation factor iec 62631-2-1 1mhz (e-4)', 'modifier': {'value': '1332', 'min': '1087', 'max': '1587', 'unit': None

C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\164086352.py:13: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(p_metadata[1],p_metadata[2])


#### For Range Modifiers

In [117]:
def get_property_range_modifier(cm=False):
    property_details = []
    range_modifier = random.choice(RANGE_MODIFIERS)
    single_space_pattern = '(\s+)'
    is_abrv = random.choice([False, False, False, False, True])
    
    p_metadata = random.choice(property_and_values)
    while p_metadata[-1] in prop_abrv_mapping:
        p_metadata = random.choice(property_and_values)
        
    property_name = p_metadata[-1].lower()
    property_name_meaning = p_metadata[0].lower()

    p_metadata2 = random.choice(property_and_values)
    property_name2 = p_metadata2[-1].lower()
    property_name_meaning2 = p_metadata2[0].lower()
    
    while p_metadata2[-1] in prop_abrv_mapping:
        p_metadata2 = random.choice(property_and_values)
        property_name2 = p_metadata2[-1].lower()
        property_name_meaning2 = p_metadata2[0].lower()
        
    if is_abrv:
        abrv = random.choice(list(prop_abrv_mapping))
        while len(abrv)<2:
            abrv = random.choice(list(prop_abrv_mapping))
            
        property_name_meaning = prop_abrv_mapping[abrv]

        abrv2 = random.choice(list(prop_abrv_mapping))
        property_name_meaning2 = prop_abrv_mapping[abrv2]

        while property_name_meaning2 == property_name_meaning:
            abrv2 = random.choice(list(prop_abrv_mapping))
            property_name_meaning2 = prop_abrv_mapping[abrv2]
    
    if cm:
    
        property_details.extend([
            {"property_name": property_name_meaning, "modifier": {"value": str(range_modifier), "min": None, "max": None, "unit": ""}, "property_type": "property"},
            {"property_name": property_name_meaning2, "modifier": {"value": str(range_modifier), "min": None, "max": None, "unit": ""}, "property_type": "property"},
        ])
                
        if is_abrv:
            query = random.choice([
                f"{range_modifier} {abrv} and {abrv2}",
                f"{range_modifier} {abrv} and {abrv2}",
                f"{range_modifier} {abrv} or {abrv2}",    
            ])
        else:
            query = random.choice([
                f"{range_modifier} {property_name} and {property_name2}",
                f"{range_modifier} {property_name} and {property_name2}",
                f"{range_modifier} {property_name} or {property_name2}",    
            ])

    else:
        property_details.extend([
            {"property_name": property_name_meaning, "modifier": {"value": str(range_modifier), "min": None, "max": None, "unit": ""}, "property_type": "property"},
        ])
        
        if is_abrv:
            query = random.choice([
                f"{abrv} {range_modifier}",
                f"{abrv}: {range_modifier}",
                f"{abrv} - {range_modifier}",
                f"{abrv} = {range_modifier}",
                f"{abrv} ({range_modifier})",
        
                f"{range_modifier} {abrv}",
                f"{range_modifier} - {abrv}",    
            ])
        else:
            query = random.choice([
                f"{property_name} {range_modifier}",
                f"{property_name}: {range_modifier}",
                f"{property_name} - {range_modifier}",
                f"{property_name} = {range_modifier}",
                f"{property_name} ({range_modifier})",
        
                f"{range_modifier} {property_name}",
                f"{range_modifier} - {property_name}",
                
                f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} = {range_modifier}",
                f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} ={range_modifier}",
    
            ])
            
            if random.choice(range(0, 12)) == 1:
                query = f"{property_name.replace(' ', '')}={range_modifier}"

    ignore_prop_range_mod = [
        {'property_name': 'impact', 'modifier': {'value': 'high', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'},
        {'property_name': 'impact', 'modifier': {'value': 'highest', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'},
    ]
    
    for i in property_details:
        if i in ignore_prop_range_mod:
            return get_property_range_modifier(cm)
                    
    return property_details, query.replace("young s", "young's")

In [118]:
get_property_range_modifier(True)

([{'property_name': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
   'modifier': {'value': 'low', 'min': None, 'max': None, 'unit': ''},
   'property_type': 'property'},
  {'property_name': 'shore a hardness iso 48-4 / iso 868 15s',
   'modifier': {'value': 'low', 'min': None, 'max': None, 'unit': ''},
   'property_type': 'property'}],
 'low mp and soft')

In [119]:
get_property_range_modifier()

([{'property_name': 'dissipation factor iec 62631-2-1 100hz (e-4)',
   'modifier': {'value': 'high', 'min': None, 'max': None, 'unit': ''},
   'property_type': 'property'}],
 'tangent delta, 100hz - high')

In [120]:
for i in range(1, 20):
    print(get_property_range_modifier(True), "\n")
    
for i in range(1, 20):
    print(get_property_range_modifier(), "\n")

([{'property_name': 'average particle size laser scattering d50 (µm)', 'modifier': {'value': 'moderate', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}, {'property_name': 'charpy impact strength', 'modifier': {'value': 'moderate', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}], 'moderate average particle size and charpy unnotched impact') 

([{'property_name': "average molecular weight margolies' equation (g/mol)", 'modifier': {'value': 'lowest', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}, {'property_name': 'melt mass-flow rate iso 1133 (g/10min)', 'modifier': {'value': 'lowest', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}], 'lowest mw and mi') 

([{'property_name': 'temperature of deflection under load', 'modifier': {'value': 'medium', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}, {'property_name': 'shore d hardness iso 48-4 / iso 868 15s', 'modifier': {'value': 'medium',

In [121]:
for i in range(1, 20):
    print(get_property(), "\n")

for i in range(1, 20):
    print(get_property_range_modifier(), "\n")

({'property_name': 'flexural stress at 3.5% iso 178 (mpa)', 'modifier': {'value': '180.34', 'min': '162.31', 'max': '198.37', 'unit': None}, 'property_type': 'property'}, 'flexural stress at 3.5% iso 178 (mpa) close to 180.34') 

({'property_name': 'izod impact strength', 'modifier': {'value': '267', 'min': '240.3', 'max': '293.7', 'unit': None}, 'property_type': 'property'}, 'izod unnotch impact 267') 

({'property_name': 'shore d hardness iso 48-4 / iso 868 15s', 'modifier': {'value': '14', 'min': '14', 'max': None, 'unit': None}, 'property_type': 'property'}, 'd greater than 14') 

({'property_name': 'vicat softening temperature', 'modifier': {'value': '144', 'min': '144', 'max': None, 'unit': None}, 'property_type': 'property'}, 'vicat softing temp of minimum 144') 

({'property_name': 'tensile stress at 100% strain iso 527-1/-2 (mpa)', 'modifier': {'value': '5', 'min': '4.5', 'max': '5.5', 'unit': 'mpa'}, 'property_type': 'property'}, 'tensile stress at 100% strain = 5mpa') 

({'p

C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\164086352.py:13: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(p_metadata[1],p_metadata[2])


### UL Properties

In [122]:
ul_cert = unique_values["UL"]

In [123]:
ul_properties = []
for item in ul_cert:
    if item['UL_PROPERTY']!="Minimum Thickness (mm)":
#         print(item)
#         print("\n")
        # if item["UNIT_OF_MEAS_SI"]:
        #     ul_properties.append((item["UL_PROPERTY"].lower(), item["Min_Value"], item["Max_Value"], item["Categorical_Values"], item["UNIT_OF_MEAS_SI"].lower(), item["UL_PROPERTY"].lower()))
        # else:
        #     ul_properties.append((item["UL_PROPERTY"].lower(), item["Min_Value"], item["Max_Value"], item["Categorical_Values"], "", item["UL_PROPERTY"].lower()))    
        if item["UNIT_OF_MEAS_SI"] and item["Categorical_Values"]:
            ul_properties.append((item["UL_PROPERTY"].lower(), item["Min_Value"], item["Max_Value"], [x.lower() for x in item["Categorical_Values"]], item["UNIT_OF_MEAS_SI"].lower(), item["UL_PROPERTY"].lower()))
        elif item["Categorical_Values"] and not item["UNIT_OF_MEAS_SI"]:
            ul_properties.append((item["UL_PROPERTY"].lower(), item["Min_Value"], item["Max_Value"], [x.lower() for x in item["Categorical_Values"]], "", item["UL_PROPERTY"].lower()))    
        elif item["UNIT_OF_MEAS_SI"] and not item["Categorical_Values"]:
            ul_properties.append((item["UL_PROPERTY"].lower(), item["Min_Value"], item["Max_Value"], None, item["UNIT_OF_MEAS_SI"].lower(), item["UL_PROPERTY"].lower()))    
        else:
            ul_properties.append((item["UL_PROPERTY"].lower(), item["Min_Value"], item["Max_Value"], None, "", item["UL_PROPERTY"].lower()))    

In [124]:
ul_properties

[('flexural stress', 74, 157, None, 'mpa', 'flexural stress'),
 ('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('comparative tracking index (cti)',
  125,
  600,
  None,
  'v',
  'comparative tracking index (cti)'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('surface resistivity',
  10000000000,
  100000000000,
  None,
  'ohms',
  'surface resistivity'),
 ('volume resistivity', 1, 1e+22, None, 'ohms·cm', 'volume resistivity'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc 

In [125]:
remove_ulp = []
for i in ul_properties:
    if 'comparative tracking index' in i[0].lower() and 'v'==i[4].lower():
        remove_ulp.append(i)
    elif i[0].lower() in ['outdoor suitability', 'detergent resistance', 'flexural stress', 'volume resistivity', 'surface resistivity']:
        remove_ulp.append(i)
        
ul_properties.append(('detergent resistance', None, None, ['f3','f4'], '', 'detergent resistance'))
ul_properties.append(('outdoor suitability', None, None, ['f1','f2'], '', 'outdoor suitability'))

ul_properties = [i for i in ul_properties if i not in remove_ulp]
ul_properties

[('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc resistance',
  None,
  None,
  ['plc 0 (420 seconds or longer)',
   'plc 1 (360 - 419 seconds)',
   'plc 2 (300 - 359 seconds)',
   'plc 3 (240 - 299 seconds)',
   'plc 4 (180 - 239 seconds)',
   'plc 5 (120 - 179 seconds)',
   'plc 6 (60 - 119 seconds)',
   'plc 7 (less than 60 seconds)'],
  '',
  'arc resistance'),
 ('comparative tracking 

In [126]:
ul_properties

[('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc resistance',
  None,
  None,
  ['plc 0 (420 seconds or longer)',
   'plc 1 (360 - 419 seconds)',
   'plc 2 (300 - 359 seconds)',
   'plc 3 (240 - 299 seconds)',
   'plc 4 (180 - 239 seconds)',
   'plc 5 (120 - 179 seconds)',
   'plc 6 (60 - 119 seconds)',
   'plc 7 (less than 60 seconds)'],
  '',
  'arc resistance'),
 ('comparative tracking 

In [127]:
for ul in ul_properties:
    if ul[3]:
        values = []
        for v in ul[3]:
            values.append(v.split(" (")[0])
        values = [i for i in values if i not in ul[3]]
        if values:
            ul[3].extend(values)

ul_properties

[('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)',
   'plc 0',
   'plc 1',
   'plc 2',
   'plc 3',
   'plc 4'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc resistance',
  None,
  None,
  ['plc 0 (420 seconds or longer)',
   'plc 1 (360 - 419 seconds)',
   'plc 2 (300 - 359 seconds)',
   'plc 3 (240 - 299 seconds)',
   'plc 4 (180 - 239 seconds)',
   'plc 5 (120 - 179 seconds)',
   'plc 6 (60 - 119 seconds)',
   'plc 7 (less than 60 sec

In [128]:
ul_prop_names = []
for ul in ul_properties:
    ul_prop_names.append(ul[0].lower())

ul_prop_names

['tensile impact strength',
 'dielectric strength',
 'ball pressure test',
 'dimensional change',
 'inclined-plane tracking',
 'high voltage arc tracking rate (hvtr)',
 'arc resistance',
 'comparative tracking index (cti)',
 'rohs 2011/65/eu material',
 'non-halogenated material',
 'non-chlorine & non-bromine material',
 'detergent resistance',
 'outdoor suitability']

In [129]:
extra_uls = []
for ulp in ul_list_name_value:
    if not ul_list_name_value[ulp]['has_thickness']:
        if ul_list_name_value[ulp]['values']:
            for syn in ul_list_name_value[ulp]['synonyms']:
                extra_uls.append((ulp, None, None, ul_list_name_value[ulp]['values'], '', syn))
        elif ulp.lower() in ul_prop_names:
            for ul in ul_properties:
                if ul[0] == ulp:
                    for syn in ul_list_name_value[ulp]['synonyms']:
                        extra_uls.append((ulp, ul[1], ul[2], ul[3], ul[4], syn)) 
        else:
            print("\n", ulp, False, "currently not adding")


extra_uls.extend(
    [
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'comparative tracking index (cti)'),
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'comparative tracking'),
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'comparative tracking index'),
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'cti'),
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'comparative tracking rating'),
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'comparative tracking index rating'),
    ('comparative tracking index (cti)', 100, 650, None, 'v', 'comp tracking'),
]
)

for i in extra_uls:
    if i not in ul_properties:
        ul_properties.append(i)
        
ul_properties

[('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)',
   'plc 0',
   'plc 1',
   'plc 2',
   'plc 3',
   'plc 4'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc resistance',
  None,
  None,
  ['plc 0 (420 seconds or longer)',
   'plc 1 (360 - 419 seconds)',
   'plc 2 (300 - 359 seconds)',
   'plc 3 (240 - 299 seconds)',
   'plc 4 (180 - 239 seconds)',
   'plc 5 (120 - 179 seconds)',
   'plc 6 (60 - 119 seconds)',
   'plc 7 (less than 60 sec

In [130]:
for i in ul_properties:
    if not i[0] or not i[-1]:
        print(i, "\n")

In [131]:
ul_property_syns = synonym_df2[synonym_df2['TYPE']=='ul property'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
ul_property_syns

{'arc resistance': ['arc resistance'],
 'comparative tracking index (cti)': ['comparative tracking index (cti)',
  'comparative tracking index',
  'cti',
  'iec tracking index',
  'tracking index'],
 'detergent resistance': ['detergent resistance', 'detergence resistance'],
 'dielectric strength': ['dielectric strength',
  'electrically insulative',
  'electrical insulation',
  'dielectric breakdown strength'],
 'dimensional change': ['dimensional change'],
 'flame rating': ['flame rating', 'ul listed', 'flame class rating'],
 'flammability classification': ['flammability classification',
  'flame classification',
  'flameclassification',
  'flame class'],
 'glow wire flammability index': ['glow wire flammability index',
  'gwfi',
  'glow wire flammability',
  'glow wire flame',
  'glow wire flame index'],
 'glow wire ignition temperature': ['flow wire ignition temperature', 'gwit'],
 'high voltage arc tracking rate (hvtr)': ['high voltage arc tracking rate (hvtr)',
  'high voltage arc

In [132]:
for i in list(ul_property_syns):
    match_ul_prop = process.extract(i, list(ul_list_name_value))
    if match_ul_prop[0][1] < 88:
        print("Not adding: ", i, match_ul_prop[0], "\n")
    else:
        # print(i, match_ul_prop[0], "\n")
        if ul_property_syns[i] != [None]:
            ul_list_name_value[match_ul_prop[0][0]]['synonyms'].extend(ul_property_syns[i])
            ul_list_name_value[match_ul_prop[0][0]]['synonyms'] = list(set(ul_list_name_value[match_ul_prop[0][0]]['synonyms']))
        else:
            print(ul_property_syns[i])

In [133]:
for i in list(ul_property_syns):
    if i not in list(ul_list_name_value):
        print(1, i, "\n")
        
    
# for j in list(ul_list_name_value):
#     if j not in list(ul_property_syns):
#         print(2, j, "\n")

In [134]:
list(ul_list_name_value)

['arc resistance',
 'ball pressure test',
 'comparative tracking index (cti)',
 'dielectric strength',
 'dimensional change',
 'flame rating',
 'flammability classification',
 'glow wire flammability index',
 'glow wire ignition temperature',
 'glow wire',
 'high voltage arc tracking rate (hvtr)',
 'high-current arc ignition (hai)',
 'hot wire ignition (hwi)',
 'inclined-plane tracking',
 'non-halogenated material',
 'outdoor suitability',
 'detergent resistance',
 'relative thermal index - electrical (rti elec) (°c)',
 'relative thermal index - mechanical impact (rti imp) (°c)',
 'relative thermal index - mechanical strength (rti str) (°c)',
 'relative thermal index',
 'rohs 2011/65/eu material',
 'tensile impact strength']

In [135]:
ul_prop_syn_mapping = {}
ul_sub_prop_syn_mapping = {}

for ulp in ul_list_name_value:
    if ul_list_name_value[ulp]['has_thickness']:
        for syn in ul_list_name_value[ulp]['synonyms']:
            ul_sub_prop_syn_mapping[syn] = ulp
    else:
        for syn in ul_list_name_value[ulp]['synonyms']:
            ul_prop_syn_mapping[syn] = ulp

In [136]:
ul_prop_syn_mapping

{'high voltage arc': 'arc resistance',
 'high voltage arc resist ignition': 'arc resistance',
 'arcres': 'arc resistance',
 'high voltage arc resistant to ignition': 'arc resistance',
 'high voltage arc resistance to ignition': 'arc resistance',
 'ar': 'arc resistance',
 'arc r': 'arc resistance',
 'high voltage arc resistant ignition': 'arc resistance',
 'hvar': 'arc resistance',
 'arc resist': 'arc resistance',
 'high voltage arc resistivity to ignition': 'arc resistance',
 'arc res': 'arc resistance',
 'arc resistant': 'arc resistance',
 'arc resistance': 'arc resistance',
 'high voltage arc resistivity ignition': 'arc resistance',
 'high voltage arc resistance ignition': 'arc resistance',
 'high voltage arc resist to ignition': 'arc resistance',
 'arc resistivity': 'arc resistance',
 'ball pressure test': 'ball pressure test',
 'ball pressure': 'ball pressure test',
 'bpt': 'ball pressure test',
 'ball test': 'ball pressure test',
 'comparative tracking index rating': 'comparative 

In [137]:
list(ul_list_name_value)

['arc resistance',
 'ball pressure test',
 'comparative tracking index (cti)',
 'dielectric strength',
 'dimensional change',
 'flame rating',
 'flammability classification',
 'glow wire flammability index',
 'glow wire ignition temperature',
 'glow wire',
 'high voltage arc tracking rate (hvtr)',
 'high-current arc ignition (hai)',
 'hot wire ignition (hwi)',
 'inclined-plane tracking',
 'non-halogenated material',
 'outdoor suitability',
 'detergent resistance',
 'relative thermal index - electrical (rti elec) (°c)',
 'relative thermal index - mechanical impact (rti imp) (°c)',
 'relative thermal index - mechanical strength (rti str) (°c)',
 'relative thermal index',
 'rohs 2011/65/eu material',
 'tensile impact strength']

In [138]:
a = []
for i in ul_properties:
    a.append(i[0])

a = list(set(a))

for i in a:
    if not i in list(set(ul_prop_syn_mapping.values())):
        print(i)

non-chlorine & non-bromine material


In [139]:
ul_properties

[('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)',
   'plc 0',
   'plc 1',
   'plc 2',
   'plc 3',
   'plc 4'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc resistance',
  None,
  None,
  ['plc 0 (420 seconds or longer)',
   'plc 1 (360 - 419 seconds)',
   'plc 2 (300 - 359 seconds)',
   'plc 3 (240 - 299 seconds)',
   'plc 4 (180 - 239 seconds)',
   'plc 5 (120 - 179 seconds)',
   'plc 6 (60 - 119 seconds)',
   'plc 7 (less than 60 sec

In [140]:
ul_properties_merge_values = {}
for i in ul_properties:
    if i[3]:
        if i[0] in ul_properties_merge_values:
            ul_properties_merge_values[i[0]] += [x for x in i[3] if x not in ul_properties_merge_values[i[0]]]
        else:
            ul_properties_merge_values[i[0]] = i[3]
            
        # print(i)

ul_properties_merge_values

{'high voltage arc tracking rate (hvtr)': ['plc 0 (0 - 10 mm/min.)',
  'plc 1 (10.1 - 25.4 mm/min.)',
  'plc 2 (25.5 - 80 mm/min.)',
  'plc 3 (80.1 - 150 mm/min.)',
  'plc 4 (greater than 150 mm/min.)',
  'plc 0',
  'plc 1',
  'plc 2',
  'plc 3',
  'plc 4',
  'plc0',
  'plc1',
  'plc2',
  'plc3',
  'plc4',
  'plc-0',
  'plc-1',
  'plc-2',
  'plc-3',
  'plc-4'],
 'arc resistance': ['plc 0 (420 seconds or longer)',
  'plc 1 (360 - 419 seconds)',
  'plc 2 (300 - 359 seconds)',
  'plc 3 (240 - 299 seconds)',
  'plc 4 (180 - 239 seconds)',
  'plc 5 (120 - 179 seconds)',
  'plc 6 (60 - 119 seconds)',
  'plc 7 (less than 60 seconds)',
  'plc 0',
  'plc 1',
  'plc 2',
  'plc 3',
  'plc 4',
  'plc 5',
  'plc 6',
  'plc 7',
  'plc0',
  'plc1',
  'plc2',
  'plc3',
  'plc4',
  'plc5',
  'plc6',
  'plc7',
  'plc-0',
  'plc-1',
  'plc-2',
  'plc-3',
  'plc-4',
  'plc-5',
  'plc-6',
  'plc-7'],
 'comparative tracking index (cti)': ['plc 0 (600 v and greater)',
  'plc 1 (400 v - 599 v)',
  'plc 2 (250

In [141]:
len(ul_properties[0])

6

In [142]:
ul_properties2 = []
temp = []
for i in ul_properties:
    if i[0] in ul_properties_merge_values and i[0]==i[-1] and i[0] not in temp:
        temp.append(i[0])
        ul_properties2.append((i[0], i[1], i[2], ul_properties_merge_values[i[0]], i[4], i[5]))
    if i[0] in ul_properties_merge_values and i[0]==i[-1] and i[0] in temp:
        pass
    else:
        if i not in ul_properties2:
             ul_properties2.append(i)

ul_properties2

[('tensile impact strength',
  761,
  761,
  None,
  'kj/m²',
  'tensile impact strength'),
 ('dielectric strength', 8, 55, None, 'kv/mm', 'dielectric strength'),
 ('ball pressure test', 125, 245, None, '', 'ball pressure test'),
 ('dimensional change', 0, 1.8, None, '%', 'dimensional change'),
 ('inclined-plane tracking', 1, 2.5, None, 'kv', 'inclined-plane tracking'),
 ('high voltage arc tracking rate (hvtr)',
  None,
  None,
  ['plc 0 (0 - 10 mm/min.)',
   'plc 1 (10.1 - 25.4 mm/min.)',
   'plc 2 (25.5 - 80 mm/min.)',
   'plc 3 (80.1 - 150 mm/min.)',
   'plc 4 (greater than 150 mm/min.)',
   'plc 0',
   'plc 1',
   'plc 2',
   'plc 3',
   'plc 4',
   'plc0',
   'plc1',
   'plc2',
   'plc3',
   'plc4',
   'plc-0',
   'plc-1',
   'plc-2',
   'plc-3',
   'plc-4'],
  '',
  'high voltage arc tracking rate (hvtr)'),
 ('arc resistance',
  None,
  None,
  ['plc 0 (420 seconds or longer)',
   'plc 1 (360 - 419 seconds)',
   'plc 2 (300 - 359 seconds)',
   'plc 3 (240 - 299 seconds)',
   'plc

In [143]:
len(ul_properties), len(ul_properties2)

(107, 102)

In [144]:
temp = []
duplicates = []
for i in ul_properties2:
    if i[-1] not in temp:
        temp.append(i[-1])
    else:
        duplicates.append(i[-1])
        print(i[-1])

comparative tracking
comparative tracking index
cti
comparative tracking rating
comparative tracking index rating
comp tracking


In [145]:
ul_properties3 = []
for i in ul_properties2:
    if i not in ul_properties3:
        ul_properties3.append(i)
        if i[-1] in duplicates:
            print(i)
    else:
        print(i)

print(len(ul_properties2), len(ul_properties3))

('comparative tracking index (cti)', None, None, ['plc 0', 'plc 1', 'plc 2', 'plc 3', 'plc 4', 'plc 5', 'plc0', 'plc1', 'plc2', 'plc3', 'plc4', 'plc5', 'plc-0', 'plc-1', 'plc-2', 'plc-3', 'plc-4', 'plc-5'], '', 'comparative tracking')
('comparative tracking index (cti)', None, None, ['plc 0', 'plc 1', 'plc 2', 'plc 3', 'plc 4', 'plc 5', 'plc0', 'plc1', 'plc2', 'plc3', 'plc4', 'plc5', 'plc-0', 'plc-1', 'plc-2', 'plc-3', 'plc-4', 'plc-5'], '', 'comparative tracking index')
('comparative tracking index (cti)', None, None, ['plc 0', 'plc 1', 'plc 2', 'plc 3', 'plc 4', 'plc 5', 'plc0', 'plc1', 'plc2', 'plc3', 'plc4', 'plc5', 'plc-0', 'plc-1', 'plc-2', 'plc-3', 'plc-4', 'plc-5'], '', 'cti')
('comparative tracking index (cti)', None, None, ['plc 0', 'plc 1', 'plc 2', 'plc 3', 'plc 4', 'plc 5', 'plc0', 'plc1', 'plc2', 'plc3', 'plc4', 'plc5', 'plc-0', 'plc-1', 'plc-2', 'plc-3', 'plc-4', 'plc-5'], '', 'comparative tracking rating')
('comparative tracking index (cti)', None, None, ['plc 0', 'plc 

In [146]:
ul_properties = [x for x in ul_properties2]

In [147]:
def get_ul_property():
    p_metadata = random.choice(ul_properties)
    property_name = p_metadata[-1].lower()

    if property_name in ul_prop_syn_mapping:
        property_name_meaning = ul_prop_syn_mapping[property_name]
    else:
        property_name_meaning = property_name
    
    try:
        min_value = None
        max_value = None
        unit = ''
        exponential = ''
        
        range_type = random.choice(['value', 'value', 'value', 'min_value', 'max_value', 'value_range', 'range_without_value', 'range_percent'])

        property_details = {'property_name': None, 'modifier': {'value': None, 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_property'}
        single_space_pattern = '(\s+)'
        try:    
            value = random.randrange(p_metadata[1],p_metadata[2])
        except: 
            value = round(random.uniform(p_metadata[1], p_metadata[2]),2)
            
        if p_metadata[4] != '°c':
            value = random.choice([value, value, value, value, value, value, value, value, value, value, value, value, value, value, random.randrange(100000, 1000000000)])

        if value > 899999:
            # print("*"*20, "{:.2e}".format(value))
            float_value = value
            value = "{:.2e}".format(value)
            exponential = 'positive'

        while value==0.0:
            value = round(random.uniform(p_metadata[1], p_metadata[2]), 9)
            print("$"*20, f"value was 0.0 for property range {p_metadata[1]} to {p_metadata[2]}, picking another random number: {value}")
            # print("#"*20, "{:.8f}".format(value))
            if -0.99999999999 < value < 0.99999999999:
                float_value = value
                value = "{:.2e}".format(value)
                # value = "{:.8f}".format(value)
                exponential = 'negative'
            else:
                value = round(value, 2)
                
        unit = random.choice([p_metadata[4],f" {p_metadata[4]}", ""])
        if not unit.strip():
            unit=''
                
        if not value:
            # print(f"could not find the value for {property_name_meaning}: {property_name}")
            value = random.randint(20, 200)
            
        is_unit_syn = random.choice([False, False, True])
        if unit and is_unit_syn and unit in unit_syns :
            # print("unit synonym")
            unit = random.choice(unit_syns[unit])
            
        if range_type=='min_value':
            try:
                if value < 0:
                    max_value, min_value = get_random_range(float(value))
                else:
                    min_value, max_value = get_random_range(float(value))
            except:
                min_value, max_value = get_random_range(random.randint(20, 2000))
                
            value = None
            max_value = None
            queries = [
                f"{property_name} of min {min_value}{unit}",
                f"{property_name} of minimum {min_value}{unit}",
                f"minimum of {property_name} {min_value}{unit}",
                f"{property_name} min {min_value}{unit}",
                f"{property_name} minimum {min_value}{unit}",           
                f"minimum {property_name} {min_value}{unit}",
                
                f"{property_name} more than {min_value}{unit}",
                f"{property_name} greater than {min_value}{unit}",
                f"{property_name} greater than {min_value}{unit}",
                f"{property_name} more than or eaual to {min_value}{unit}",
                f"{property_name} greater than or equal to {min_value}{unit}",
                f"{property_name} greater than or equal to {min_value}{unit}",
                f"{property_name} >{min_value}{unit}",
                f"{property_name} > {min_value}{unit}",
                f"{property_name} >={min_value}{unit}",
                f"{property_name} >= {min_value}{unit}",
            ]
            query = random.choice(queries)
        elif range_type=='max_value':
            try:
                if value < 0:
                    max_value, min_value = get_random_range(float(value))
                else:
                    min_value, max_value = get_random_range(float(value))
            except:
                min_value, max_value = get_random_range(random.randint(20, 2000))
                
            value = None
            min_value = None
            queries = [
                f"{property_name} of max {max_value}{unit}",
                f"{property_name} of maximum {max_value}{unit}",
                f"maximum of {property_name} {max_value}{unit}",
                f"{property_name} max {max_value}{unit}",
                f"{property_name} maximum {max_value}{unit}",
                f"maximum {property_name} {max_value}{unit}",
                
                f"{property_name} lesser than {max_value}{unit}",
                f"{property_name} less than {max_value}{unit}",
                f"{property_name} lesser than or equal to {max_value}{unit}",
                f"{property_name} less than or equal to {max_value}{unit}",
                f"{property_name} <{max_value}{unit}",
                f"{property_name} < {max_value}{unit}",
                f"{property_name} <={max_value}{unit}",
                f"{property_name} <= {max_value}{unit}",
            ]
            query = random.choice(queries)   
        else:
            queries = [
                f"{property_name} {value}{unit}",
                f"{property_name} {value}{unit}",

                f"{property_name} of {value}{unit}",
                f"{property_name} = {value}{unit}",
                f"{property_name} - {value}{unit}",
                f"{property_name}: {value}{unit}",
                f"{property_name} ({value}{unit})",
                f"{value}{unit} {property_name}",

                f"{property_name} {value}{unit}",
                f"{property_name} {value}{unit}",

                f"{property_name} of {value}{unit}",
                f"{property_name} = {value}{unit}",
                f"{property_name} - {value}{unit}",
                f"{property_name}: {value}{unit}",
                f"{property_name} ({value}{unit})",
                f"{value}{unit} {property_name}",

                f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} = {value}{unit}",
                f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} ={value}{unit}",

            ]

            if range_type == 'value':
                queries += [
                    f"{property_name} of approximately {value}{unit}",
                    f"{property_name} of approx {value}{unit}",
                    f"{property_name} close to {value}{unit}",
                    f"{property_name} around {value}{unit}",
                ]

            query = random.choice(queries)
            if random.choice(range(0, 10)) == 1 and property_name:
                query = f"{property_name.replace(' ', '')}={value}{unit}"
        #         print(f"cant remove space for {property_name}")

            if not " " in property_name and not exponential and value > 0:
                query2 = random.choice([
                    f"{property_name}{value}",
                    f"{property_name}-{value}",
                    f"{value}{property_name}",
                    f"{property_name}={value}",
                ])
                is_unit = random.choice([True, True, True, False, False])
                if not is_unit:
                    query = query2
                    unit = ''

            if not exponential and range_type != 'value' and value > 0:
                min_value, max_value = get_random_range(float(value))
                query3 = random.choice([       
                    f"{property_name} of {value}{unit}",
                    f"{property_name} {value}{unit}",
                    f"{property_name} ({value}{unit})",
                    f"{property_name} ={value}{unit}",
                    f"{property_name} = {value}{unit}",
                    f"{property_name} - {value}{unit}",
                    f"{property_name}: {value}{unit}",
                ])

                if range_type == 'value_range': 
                    range_query = random.choice([
                        f" from {min_value} to {max_value}",
                        f" from {min_value} - {max_value}",
                        f" from {min_value}-{max_value}",
                        f" between {min_value} to {max_value}",
                        f" between {min_value} - {max_value}",
                        f" between {min_value}-{max_value}",
                        f" between {min_value} and {max_value}",
                        f" in the range of {min_value} to {max_value}",
                        f" in the range of {min_value} - {max_value}",
                        f" in the range of {min_value}-{max_value}",
                    ])
                    query = query3 + range_query
                elif range_type == 'range_without_value':
                    range_query = random.choice([
                        f" from {min_value} to {max_value}",
                        f" from {min_value} - {max_value}",
                        f" from {min_value}-{max_value}",
                        f" {min_value}-{max_value}",
                        f" {min_value} - {max_value}",
                        f" {min_value} to {max_value}",
                        f" between {min_value} to {max_value}",
                        f" between {min_value} - {max_value}",
                        f" between {min_value}-{max_value}",
                        f" between {min_value} and {max_value}",
                        f" in the range of {min_value} to {max_value}",
                        f" in the range of {min_value} - {max_value}",
                        f" in the range of {min_value}-{max_value}",
                    ])
                    query = f"{property_name}" + range_query + f"{unit}"

                elif range_type == 'range_percent':
                    percent, min_value, max_value = get_random_percent(value)
                    range_percent_query = random.choice([
                        f" with {percent}% range",
                        f" with +/-{percent}% range",
                        f" of +/-{percent}% range",
                        f" of -/+{percent}% range",
                        f" around {percent}% variation",
                        f" around {percent}% range",
                        f" around {percent}% margin",
        #                 f" close to a {percent}% variation",
        #                 f" close to a {percent}% range",
        #                 f" close to a {percent}% margin",
                        f" in the vicinity of {percent}%",
                        f" in the range of {percent}%",
                        f" within a range of {percent}%",
                        f" within {percent}% range",
                        f" within {percent}% margin",
                        ])

                    query = query3 + range_percent_query
                else:
                    print("@"*40, range_type)
            else:
                range_type = 'value'
            
        property_details['property_name'] = property_name_meaning
    
        property_details['modifier']['unit'] = unit.strip().lower()
        if not property_details['modifier']['unit']:
            property_details['modifier']['unit'] = None

        # no range queries for exponential queries
        if range_type == 'min_value':
            property_details['modifier']['min'] =  str(round(min_value, 2))
            property_details['modifier']['max'] =  None
            property_details['modifier']['value'] = property_details['modifier']['min']

        elif range_type == 'max_value':
            property_details['modifier']['max'] = str(round(max_value, 2))
            property_details['modifier']['min'] =  None
            property_details['modifier']['value'] = property_details['modifier']['max']

        elif range_type in ['value_range', 'range_percent']:
            property_details['modifier']['value'] = str(round(value, 2))
            property_details['modifier']['min'] = str(round(min_value, 2))
            property_details['modifier']['max'] = str(round(max_value, 2))
            
        elif range_type == 'range_without_value':
            property_details['modifier']['min'] = str(round(min_value, 2))
            property_details['modifier']['max'] = str(round(max_value, 2))
            property_details['modifier']['value'] = property_details['modifier']['min'] 
        else:
            if value:
                if exponential == 'positive':
                    # print("#"*25, 'positive')
                    if float_value < 0:
                        property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 2))
                        property_details['modifier']['max'] = "{:.2e}".format(round(float_value*0.9, 2))
                        property_details['modifier']['min'] = "{:.2e}".format(round(float_value*1.1, 2))
                    else:
                        property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 2))
                        property_details['modifier']['min'] = "{:.2e}".format(round(float_value*0.9, 2))
                        property_details['modifier']['max'] = "{:.2e}".format(round(float_value*1.1, 2))
                        
                elif exponential == 'negative':
                    print("*"*25, 'negative exponential')
                    if float_value < 0:
                        property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 9))
                        property_details['modifier']['max'] = "{:.2e}".format(round(float_value*0.9, 9))
                        property_details['modifier']['min'] = "{:.2e}".format(round(float_value*1.1, 9))
                    else:
                        property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 9))
                        property_details['modifier']['min'] = "{:.2e}".format(round(float_value*0.9, 9))
                        property_details['modifier']['max'] = "{:.2e}".format(round(float_value*1.1, 9))
                    print(property_details)
                    
                else:
                    if value < 0:
                        property_details['modifier']['value'] = str(round(value, 2))
                        property_details['modifier']['max'] = str(round(value*0.9, 2))
                        property_details['modifier']['min'] = str(round(value*1.1, 2))
                    else:
                        property_details['modifier']['value'] = str(round(value, 2))
                        property_details['modifier']['min'] = str(round(value*0.9, 2))
                        property_details['modifier']['max'] = str(round(value*1.1, 2))
            else:
                property_details['modifier']['value'] = None
    except:
        values = p_metadata[3]
        unit = "" 
        if type(values)==list:
            value = random.choice(values)
            queries = [
                f"{property_name}-{value}{unit}",
                
                ####
                f"{property_name} {value}{unit}",
                f"{property_name} {value}{unit}",
                
                f"{property_name} of {value}{unit}",
                f"{property_name} = {value}{unit}",
                f"{property_name}={value}{unit}",
                f"{property_name} - {value}{unit}",
                
                f"{property_name}: {value}{unit}",
                f"{property_name} ({value}{unit})",
                f"{value}{unit} {property_name}",

                f"{property_name.replace(' ', '')} {value}{unit}",
                f"{property_name.replace(' ', '')} of {value}{unit}",

                ####
                f"{property_name} {value}{unit}",
                f"{property_name} {value}{unit}",
                
                f"{property_name} of {value}{unit}",
                f"{property_name} = {value}{unit}",
                f"{property_name}={value}{unit}",
                f"{property_name} - {value}{unit}",
                
                f"{property_name}: {value}{unit}",
                f"{property_name} ({value}{unit})",
                f"{value}{unit} {property_name}",

                f"{property_name.replace(' ', '')} {value}{unit}",
                f"{property_name.replace(' ', '')} of {value}{unit}",
            ]
            query = random.choice(queries)
            property_details = {"property_name": property_name_meaning, "modifier": {"value": value, "min": None, "max": None, "unit": unit.strip()}, "property_type": "ul_property"}

        
    query = re.sub("(\s+)", " ", query.strip().lower())

    return property_details, query

In [148]:
for i in range(1, 25):
    print(get_ul_property(), "\n")

({'property_name': 'ball pressure test', 'modifier': {'value': '1.96e+08', 'min': '1.76e+08', 'max': '2.15e+08', 'unit': None}, 'property_type': 'ul_property'}, 'ball pressure test around 1.96e+08') 

({'property_name': 'outdoor suitability', 'modifier': {'value': 'f-2', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_property'}, 'suitability = f-2') 

({'property_name': 'detergent resistance', 'modifier': {'value': 'f4', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_property'}, 'detergent resis f4') 

({'property_name': 'outdoor suitability', 'modifier': {'value': 'f-2', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_property'}, 'outdoor suitability - f-2') 

({'property_name': 'arc resistance', 'modifier': {'value': 'plc 4', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_property'}, 'arc resistant of plc 4') 

({'property_name': 'dielectric strength', 'modifier': {'value': '28', 'min': '28', 'max': '50', 'unit': 'kilovolts per milli

C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\2640670372.py:21: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(p_metadata[1],p_metadata[2])


### UL Sub-Property

In [149]:
ul_sub_properties = []
for item in ul_cert:
    if item['UL_PROPERTY']=="Minimum Thickness (mm)":
        sub_items = item['SUB_PROPERTIES']
        for sub_item in sub_items:
            if "Categorical_Values" in  sub_item.keys():
                ul_sub_properties.append((sub_item["UL_SUB_PROPERTY"].lower().replace("–", "-"),None,None,[x.lower() for x in sub_item["Categorical_Values"] if x],None,sub_item["UL_SUB_PROPERTY"].lower().replace("–", "-")))
            else:
                if sub_item["UNIT_OF_MEAS_SI"]:
                    ul_sub_properties.append((sub_item["UL_SUB_PROPERTY"].lower().replace("–", "-"), sub_item["Min_Value"], sub_item["Max_Value"], None, sub_item["UNIT_OF_MEAS_SI"].lower(),sub_item["UL_SUB_PROPERTY"].lower().replace("–", "-")))
                else:
                    ul_sub_properties.append((sub_item["UL_SUB_PROPERTY"].lower().replace("–", "-"), sub_item["Min_Value"], sub_item["Max_Value"], None, '',sub_item["UL_SUB_PROPERTY"].lower().replace("–", "-")))

In [150]:
ul_sub_properties

[('relative thermal index - mechanical strength (rti str) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - mechanical strength (rti str) (°c)'),
 ('relative thermal index - mechanical impact (rti imp) (°c)',
  50,
  220,
  None,
  '°c',
  'relative thermal index - mechanical impact (rti imp) (°c)'),
 ('relative thermal index - electrical (rti elec) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - electrical (rti elec) (°c)'),
 ('glow wire ignition temperature',
  625,
  985,
  None,
  '°c',
  'glow wire ignition temperature'),
 ('glow wire flammability index',
  650,
  960,
  None,
  '°c',
  'glow wire flammability index'),
 ('flame rating',
  None,
  None,
  ['5va', '5vb', 'v-0', 'v-1', 'v-2', 'hb'],
  None,
  'flame rating'),
 ('hot wire ignition (hwi)',
  None,
  None,
  ['plc 0 (120 seconds and longer)',
   'plc 1 (60 through 119 seconds)',
   'plc 2 (30 through 59 seconds)',
   'plc 3 (15 through 29 seconds)',
   'plc 4 (7 through 14 seconds)',
   'pl

In [151]:
for ul_sub in ul_sub_properties:
    if ul_sub[3]:
        values = []
        for v in ul_sub[3]:
            values.append(v.split(" (")[0])
        values = [i for i in values if i not in ul_sub[3]]
        if values:
            ul_sub[3].extend(values)

ul_sub_properties

[('relative thermal index - mechanical strength (rti str) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - mechanical strength (rti str) (°c)'),
 ('relative thermal index - mechanical impact (rti imp) (°c)',
  50,
  220,
  None,
  '°c',
  'relative thermal index - mechanical impact (rti imp) (°c)'),
 ('relative thermal index - electrical (rti elec) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - electrical (rti elec) (°c)'),
 ('glow wire ignition temperature',
  625,
  985,
  None,
  '°c',
  'glow wire ignition temperature'),
 ('glow wire flammability index',
  650,
  960,
  None,
  '°c',
  'glow wire flammability index'),
 ('flame rating',
  None,
  None,
  ['5va', '5vb', 'v-0', 'v-1', 'v-2', 'hb'],
  None,
  'flame rating'),
 ('hot wire ignition (hwi)',
  None,
  None,
  ['plc 0 (120 seconds and longer)',
   'plc 1 (60 through 119 seconds)',
   'plc 2 (30 through 59 seconds)',
   'plc 3 (15 through 29 seconds)',
   'plc 4 (7 through 14 seconds)',
   'pl

In [152]:
ul_sub_prop_names = []
for ul_sub in ul_sub_properties:
    ul_sub_prop_names.append(ul_sub[0].lower())

ul_sub_prop_names

['relative thermal index - mechanical strength (rti str) (°c)',
 'relative thermal index - mechanical impact (rti imp) (°c)',
 'relative thermal index - electrical (rti elec) (°c)',
 'glow wire ignition temperature',
 'glow wire flammability index',
 'flame rating',
 'hot wire ignition (hwi)',
 'high-current arc ignition (hai)',
 'flammability classification']

In [153]:
ul_sub_properties

[('relative thermal index - mechanical strength (rti str) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - mechanical strength (rti str) (°c)'),
 ('relative thermal index - mechanical impact (rti imp) (°c)',
  50,
  220,
  None,
  '°c',
  'relative thermal index - mechanical impact (rti imp) (°c)'),
 ('relative thermal index - electrical (rti elec) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - electrical (rti elec) (°c)'),
 ('glow wire ignition temperature',
  625,
  985,
  None,
  '°c',
  'glow wire ignition temperature'),
 ('glow wire flammability index',
  650,
  960,
  None,
  '°c',
  'glow wire flammability index'),
 ('flame rating',
  None,
  None,
  ['5va', '5vb', 'v-0', 'v-1', 'v-2', 'hb'],
  None,
  'flame rating'),
 ('hot wire ignition (hwi)',
  None,
  None,
  ['plc 0 (120 seconds and longer)',
   'plc 1 (60 through 119 seconds)',
   'plc 2 (30 through 59 seconds)',
   'plc 3 (15 through 29 seconds)',
   'plc 4 (7 through 14 seconds)',
   'pl

In [154]:
ul_list_name_value.keys()

dict_keys(['arc resistance', 'ball pressure test', 'comparative tracking index (cti)', 'dielectric strength', 'dimensional change', 'flame rating', 'flammability classification', 'glow wire flammability index', 'glow wire ignition temperature', 'glow wire', 'high voltage arc tracking rate (hvtr)', 'high-current arc ignition (hai)', 'hot wire ignition (hwi)', 'inclined-plane tracking', 'non-halogenated material', 'outdoor suitability', 'detergent resistance', 'relative thermal index - electrical (rti elec) (°c)', 'relative thermal index - mechanical impact (rti imp) (°c)', 'relative thermal index - mechanical strength (rti str) (°c)', 'relative thermal index', 'rohs 2011/65/eu material', 'tensile impact strength'])

In [155]:
for i in ul_sub_properties:
    if i[0] == i[-1]:
        print(i[0], " | ", i[3], " | ", i[-1])

relative thermal index - mechanical strength (rti str) (°c)  |  None  |  relative thermal index - mechanical strength (rti str) (°c)
relative thermal index - mechanical impact (rti imp) (°c)  |  None  |  relative thermal index - mechanical impact (rti imp) (°c)
relative thermal index - electrical (rti elec) (°c)  |  None  |  relative thermal index - electrical (rti elec) (°c)
glow wire ignition temperature  |  None  |  glow wire ignition temperature
glow wire flammability index  |  None  |  glow wire flammability index
flame rating  |  ['5va', '5vb', 'v-0', 'v-1', 'v-2', 'hb']  |  flame rating
hot wire ignition (hwi)  |  ['plc 0 (120 seconds and longer)', 'plc 1 (60 through 119 seconds)', 'plc 2 (30 through 59 seconds)', 'plc 3 (15 through 29 seconds)', 'plc 4 (7 through 14 seconds)', 'plc 5 (less than 7 seconds)', 'plc 0', 'plc 1', 'plc 2', 'plc 3', 'plc 4', 'plc 5']  |  hot wire ignition (hwi)
high-current arc ignition (hai)  |  ['plc 0 (120 and greater)', 'plc 1 (60 - 119)', 'plc 2 

In [156]:
extra_ul_subs = []
for ulp in ul_list_name_value:
    if ul_list_name_value[ulp]['has_thickness']:
        if ul_list_name_value[ulp]['values']:
            for syn in ul_list_name_value[ulp]['synonyms']:
                extra_ul_subs.append((ulp, None, None, ul_list_name_value[ulp]['values'], None, syn))
        elif ulp.lower() in ul_sub_prop_names:
            for ul_sub in ul_sub_properties:
                if ul_sub[0] == ulp:
                    for syn in ul_list_name_value[ulp]['synonyms']:
                        extra_ul_subs.append((ulp, ul_sub[1], ul_sub[2], ul_sub[3], ul_sub[4], syn))
        else:
            print("\n", ulp, False, "should we add?")

extra_ul_subs.extend([
    ('relative thermal index', 50, 240, None, '°c', 'relative thermal index'), 
    ('glow wire', 600, 100, None, '°c', 'glow wire')
]
)

ul_sub_properties.extend(extra_ul_subs)
ul_sub_properties


 glow wire False should we add?

 relative thermal index False should we add?


[('relative thermal index - mechanical strength (rti str) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - mechanical strength (rti str) (°c)'),
 ('relative thermal index - mechanical impact (rti imp) (°c)',
  50,
  220,
  None,
  '°c',
  'relative thermal index - mechanical impact (rti imp) (°c)'),
 ('relative thermal index - electrical (rti elec) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - electrical (rti elec) (°c)'),
 ('glow wire ignition temperature',
  625,
  985,
  None,
  '°c',
  'glow wire ignition temperature'),
 ('glow wire flammability index',
  650,
  960,
  None,
  '°c',
  'glow wire flammability index'),
 ('flame rating',
  None,
  None,
  ['5va', '5vb', 'v-0', 'v-1', 'v-2', 'hb'],
  None,
  'flame rating'),
 ('hot wire ignition (hwi)',
  None,
  None,
  ['plc 0 (120 seconds and longer)',
   'plc 1 (60 through 119 seconds)',
   'plc 2 (30 through 59 seconds)',
   'plc 3 (15 through 29 seconds)',
   'plc 4 (7 through 14 seconds)',
   'pl

In [157]:
ul_sub_properties_merge_values = {}
for i in ul_sub_properties:
    if i[3]:
        if i[0] in ul_sub_properties_merge_values:
            ul_sub_properties_merge_values[i[0]] += [x for x in i[3] if x not in ul_sub_properties_merge_values[i[0]]]
        else:
            ul_sub_properties_merge_values[i[0]] = i[3]
            
        # print(i)

ul_sub_properties_merge_values

{'flame rating': ['5va',
  '5vb',
  'v-0',
  'v-1',
  'v-2',
  'hb',
  'vo',
  'v0',
  'v1',
  'v2',
  '5 va',
  '5 vb',
  'v 0',
  'v 1',
  'v 2',
  'v-o',
  '5-va',
  '5-vb'],
 'hot wire ignition (hwi)': ['plc 0 (120 seconds and longer)',
  'plc 1 (60 through 119 seconds)',
  'plc 2 (30 through 59 seconds)',
  'plc 3 (15 through 29 seconds)',
  'plc 4 (7 through 14 seconds)',
  'plc 5 (less than 7 seconds)',
  'plc 0',
  'plc 1',
  'plc 2',
  'plc 3',
  'plc 4',
  'plc 5',
  'plc0',
  'plc1',
  'plc2',
  'plc3',
  'plc4',
  'plc5',
  'plc-0',
  'plc-1',
  'plc-2',
  'plc-3',
  'plc-4',
  'plc-5'],
 'high-current arc ignition (hai)': ['plc 0 (120 and greater)',
  'plc 1 (60 - 119)',
  'plc 2 (30 - 59)',
  'plc 3 (15 - 29)',
  'plc 4 (less than 15)',
  'plc 0',
  'plc 1',
  'plc 2',
  'plc 3',
  'plc 4',
  'plc0',
  'plc1',
  'plc2',
  'plc3',
  'plc4',
  'plc-0',
  'plc-1',
  'plc-2',
  'plc-3',
  'plc-4'],
 'flammability classification': ['5va',
  '5vb',
  'v-0',
  'v-1',
  'v-2',
  

In [158]:
len(ul_sub_properties[0])

6

In [159]:
ul_sub_properties2 = []
temp = []
for i in ul_sub_properties:
    if i[0] in ul_sub_properties_merge_values and i[0]==i[-1] and i[0] not in temp:
        temp.append(i[0])
        ul_sub_properties2.append((i[0], i[1], i[2], ul_sub_properties_merge_values[i[0]], i[4], i[5]))
    if i[0] in ul_sub_properties_merge_values and i[0]==i[-1] and i[0] in temp:
        pass
    else:
        if i not in ul_sub_properties2: 
             ul_sub_properties2.append(i)
ul_sub_properties2

[('relative thermal index - mechanical strength (rti str) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - mechanical strength (rti str) (°c)'),
 ('relative thermal index - mechanical impact (rti imp) (°c)',
  50,
  220,
  None,
  '°c',
  'relative thermal index - mechanical impact (rti imp) (°c)'),
 ('relative thermal index - electrical (rti elec) (°c)',
  50,
  240,
  None,
  '°c',
  'relative thermal index - electrical (rti elec) (°c)'),
 ('glow wire ignition temperature',
  625,
  985,
  None,
  '°c',
  'glow wire ignition temperature'),
 ('glow wire flammability index',
  650,
  960,
  None,
  '°c',
  'glow wire flammability index'),
 ('flame rating',
  None,
  None,
  ['5va',
   '5vb',
   'v-0',
   'v-1',
   'v-2',
   'hb',
   'vo',
   'v0',
   'v1',
   'v2',
   '5 va',
   '5 vb',
   'v 0',
   'v 1',
   'v 2',
   'v-o',
   '5-va',
   '5-vb'],
  None,
  'flame rating'),
 ('hot wire ignition (hwi)',
  None,
  None,
  ['plc 0 (120 seconds and longer)',
   'plc 1 (60 t

In [160]:
len(ul_sub_properties), len(ul_sub_properties2)

(92, 88)

In [161]:
temp = []
duplicates = []
for i in ul_sub_properties2:
    if i[-1] not in temp:
        temp.append(i[-1])
    else:
        duplicates.append(i[-1])
        print(i[-1])

In [162]:
ul_sub_properties3 = []
for i in ul_sub_properties2:
    if i not in ul_sub_properties3:
        ul_sub_properties3.append(i)
        if i[-1] in duplicates:
            print(i)
    else:
        print(i)

print(len(ul_sub_properties2), len(ul_sub_properties3))

88 88


In [163]:
ul_sub_prop_syn_mapping

{'flame class rating': 'flame rating',
 'ul listed': 'flame rating',
 'flammability class rating': 'flame rating',
 'flame rating': 'flame rating',
 'flame rated': 'flame rating',
 'flame rate': 'flame rating',
 'flamerated': 'flame rating',
 'flame': 'flame rating',
 'flammability rating class': 'flame rating',
 'flame rating class': 'flame rating',
 'flamerate': 'flame rating',
 'flamerating': 'flame rating',
 'fr': 'flame rating',
 'flammability rating': 'flame rating',
 'flammability classification': 'flammability classification',
 'flammability class': 'flammability classification',
 'flame classification': 'flammability classification',
 'flame class': 'flammability classification',
 'flameclassification': 'flammability classification',
 'glow wire flammability index': 'glow wire flammability index',
 'glow-wire flammability': 'glow wire flammability index',
 'glow wire flammability': 'glow wire flammability index',
 'gwf index': 'glow wire flammability index',
 'glow wire flame'

In [164]:
def get_ul_sub_property(default=None):
    if default==None:
        default = random.choice([True, False])
        
    p_metadata = random.choice(ul_sub_properties)
    property_name = p_metadata[-1].lower()
    if property_name in ul_sub_prop_syn_mapping:
        property_name_meaning = ul_sub_prop_syn_mapping[property_name]
    else:
        property_name_meaning = property_name

    value_only_prop = {
        'flammability classification': ['hb-40', 'hb-75'], 
        'flame rating': ['5va', '5vb', 'hb', '5 va', '5 vb', '5-va', '5-vb', '5va', '5vb', 'hb', '5 va', '5 vb', '5-va', '5-vb', 'v-1', 'v-2', 'v1', 'v2', 'v 1', 'v 2',]
    }
    if property_name_meaning in value_only_prop:
        only_value = random.choice([True, False, False])
    else:
        only_value = False

    if default:
        try:    
            min_value = None
            max_value = None
            unit = ''
            exponential = ''
            
            range_type = random.choice(['value', 'value', 'value', 'min_value', 'max_value', 'value_range', 'range_without_value', 'range_percent'])
            
            sub_property_details = {'property_name': None, 'modifier': {'value': None, 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_sub_property'}
            single_space_pattern = '(\s+)'
            try:    
                value = random.randrange(p_metadata[1],p_metadata[2])
            except: 
                value = round(random.uniform(p_metadata[1], p_metadata[2]),2)
                
            if p_metadata[4] != '°c':
                value = random.choice([value, value, value, value, value, value, value, value, value, value, value, value, value, value, random.randrange(100000, 1000000000)])
            
            if value > 899999:
                # print("*"*20, "{:.2e}".format(value))
                float_value = value
                value = "{:.2e}".format(value)
                exponential = 'positive'
            
            while value==0.0:
                value = round(random.uniform(p_metadata[1], p_metadata[2]), 9)
                print("$"*20, f"value was 0.0 for property range {p_metadata[1]} to {p_metadata[2]}, picking another random number: {value}")
                # print("#"*20, "{:.8f}".format(value))
                if -0.99999999999 < value < 0.99999999999:
                    float_value = value
                    value = "{:.2e}".format(value)
                    # value = "{:.8f}".format(value)
                    exponential = 'negative'
                else:
                    value = round(value, 2)
                    
            unit = random.choice([p_metadata[4],f" {p_metadata[4]}", ""])
            if not unit.strip():
                unit=''
                    
            if not value:
                # print(f"could not find the value for {property_name_meaning}: {property_name}")
                value = random.randint(20, 200)
                
            is_unit_syn = random.choice([False, False, True])
            if unit and is_unit_syn and unit in unit_syns :
                # print("unit synonym")
                unit = random.choice(unit_syns[unit])
                
            if range_type=='min_value':
                try:
                    if value < 0:
                        max_value, min_value = get_random_range(float(value))
                    else:
                        min_value, max_value = get_random_range(float(value))
                except:
                    min_value, max_value = get_random_range(random.randint(20, 2000))
                    
                value = None
                max_value = None
                queries = [
                    f"{property_name} of min {min_value}{unit}",
                    f"{property_name} of minimum {min_value}{unit}",
                    f"minimum of {property_name} {min_value}{unit}",
                    f"{property_name} min {min_value}{unit}",
                    f"{property_name} minimum {min_value}{unit}",           
                    f"minimum {property_name} {min_value}{unit}",
                    
                    f"{property_name} more than {min_value}{unit}",
                    f"{property_name} greater than {min_value}{unit}",
                    f"{property_name} greater than {min_value}{unit}",
                    f"{property_name} more than or eaual to {min_value}{unit}",
                    f"{property_name} greater than or equal to {min_value}{unit}",
                    f"{property_name} greater than or equal to {min_value}{unit}",
                    f"{property_name} >{min_value}{unit}",
                    f"{property_name} > {min_value}{unit}",
                    f"{property_name} >={min_value}{unit}",
                    f"{property_name} >= {min_value}{unit}",
                ]
                query = random.choice(queries)
            elif range_type=='max_value':
                try:
                    if value < 0:
                        max_value, min_value = get_random_range(float(value))
                    else:
                        min_value, max_value = get_random_range(float(value))
                except:
                    min_value, max_value = get_random_range(random.randint(20, 2000))
                    
                value = None
                min_value = None
                queries = [
                    f"{property_name} of max {max_value}{unit}",
                    f"{property_name} of maximum {max_value}{unit}",
                    f"maximum of {property_name} {max_value}{unit}",
                    f"{property_name} max {max_value}{unit}",
                    f"{property_name} maximum {max_value}{unit}",
                    f"maximum {property_name} {max_value}{unit}",
                    
                    f"{property_name} lesser than {max_value}{unit}",
                    f"{property_name} less than {max_value}{unit}",
                    f"{property_name} lesser than or equal to {max_value}{unit}",
                    f"{property_name} less than or equal to {max_value}{unit}",
                    f"{property_name} <{max_value}{unit}",
                    f"{property_name} < {max_value}{unit}",
                    f"{property_name} <={max_value}{unit}",
                    f"{property_name} <= {max_value}{unit}",
                ]
                query = random.choice(queries)   
            else:
                queries = [
                    f"{property_name} {value}{unit}",
                    f"{property_name} {value}{unit}",
            
                    f"{property_name} of {value}{unit}",
                    f"{property_name} = {value}{unit}",
                    f"{property_name} - {value}{unit}",
                    f"{property_name}: {value}{unit}",
                    f"{property_name} ({value}{unit})",
                    f"{value}{unit} {property_name}",
            
                    f"{property_name} {value}{unit}",
                    f"{property_name} {value}{unit}",
            
                    f"{property_name} of {value}{unit}",
                    f"{property_name} = {value}{unit}",
                    f"{property_name} - {value}{unit}",
                    f"{property_name}: {value}{unit}",
                    f"{property_name} ({value}{unit})",
                    f"{value}{unit} {property_name}",
            
                    f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} = {value}{unit}",
                    f"{re.sub(single_space_pattern, ' ', re.sub('[^A-Za-z0-9]+', ' ', property_name).strip())} ={value}{unit}",
            
                ]
            
                if range_type == 'value':
                    queries += [
                        f"{property_name} of approximately {value}{unit}",
                        f"{property_name} of approx {value}{unit}",
                        f"{property_name} close to {value}{unit}",
                        f"{property_name} around {value}{unit}",
                    ]
            
                query = random.choice(queries)
                if random.choice(range(0, 10)) == 1 and property_name:
                    query = f"{property_name.replace(' ', '')}={value}{unit}"
            #         print(f"cant remove space for {property_name}")
            
                if not " " in property_name and not exponential and value > 0:
                    query2 = random.choice([
                        f"{property_name}{value}",
                        f"{property_name}-{value}",
                        f"{value}{property_name}",
                        f"{property_name}={value}",
                    ])
                    is_unit = random.choice([True, True, True, False, False])
                    if not is_unit:
                        query = query2
                        unit = ''
            
                if not exponential and range_type != 'value' and value > 0:
                    min_value, max_value = get_random_range(float(value))
                    query3 = random.choice([       
                        f"{property_name} of {value}{unit}",
                        f"{property_name} {value}{unit}",
                        f"{property_name} ({value}{unit})",
                        f"{property_name} ={value}{unit}",
                        f"{property_name} = {value}{unit}",
                        f"{property_name} - {value}{unit}",
                        f"{property_name}: {value}{unit}",
                    ])
            
                    if range_type == 'value_range': 
                        range_query = random.choice([
                            f" from {min_value} to {max_value}",
                            f" from {min_value} - {max_value}",
                            f" from {min_value}-{max_value}",
                            f" between {min_value} to {max_value}",
                            f" between {min_value} - {max_value}",
                            f" between {min_value}-{max_value}",
                            f" between {min_value} and {max_value}",
                            f" in the range of {min_value} to {max_value}",
                            f" in the range of {min_value} - {max_value}",
                            f" in the range of {min_value}-{max_value}",
                        ])
                        query = query3 + range_query
                    elif range_type == 'range_without_value':
                        range_query = random.choice([
                            f" from {min_value} to {max_value}",
                            f" from {min_value} - {max_value}",
                            f" from {min_value}-{max_value}",
                            f" {min_value}-{max_value}",
                            f" {min_value} - {max_value}",
                            f" {min_value} to {max_value}",
                            f" between {min_value} to {max_value}",
                            f" between {min_value} - {max_value}",
                            f" between {min_value}-{max_value}",
                            f" between {min_value} and {max_value}",
                            f" in the range of {min_value} to {max_value}",
                            f" in the range of {min_value} - {max_value}",
                            f" in the range of {min_value}-{max_value}",
                        ])
                        query = f"{property_name}" + range_query + f"{unit}"
            
                    elif range_type == 'range_percent':
                        percent, min_value, max_value = get_random_percent(value)
                        range_percent_query = random.choice([
                            f" with {percent}% range",
                            f" with +/-{percent}% range",
                            f" of +/-{percent}% range",
                            f" of -/+{percent}% range",
                            f" around {percent}% variation",
                            f" around {percent}% range",
                            f" around {percent}% margin",
            #                 f" close to a {percent}% variation",
            #                 f" close to a {percent}% range",
            #                 f" close to a {percent}% margin",
                            f" in the vicinity of {percent}%",
                            f" in the range of {percent}%",
                            f" within a range of {percent}%",
                            f" within {percent}% range",
                            f" within {percent}% margin",
                            ])
            
                        query = query3 + range_percent_query
                    else:
                        print("@"*40, range_type)
                else:
                    range_type = 'value'
                
            sub_property_details['property_name'] = property_name_meaning
            
            sub_property_details['modifier']['unit'] = unit.strip().lower()
            if not sub_property_details['modifier']['unit']:
                sub_property_details['modifier']['unit'] = None
            
            # no range queries for exponential queries
            if range_type == 'min_value':
                sub_property_details['modifier']['min'] =  str(round(min_value, 2))
                sub_property_details['modifier']['max'] =  None
                sub_property_details['modifier']['value'] = sub_property_details['modifier']['min']
            
            elif range_type == 'max_value':
                sub_property_details['modifier']['max'] = str(round(max_value, 2))
                sub_property_details['modifier']['min'] =  None
                sub_property_details['modifier']['value'] = sub_property_details['modifier']['max']
            
            elif range_type in ['value_range', 'range_percent']:
                sub_property_details['modifier']['value'] = str(round(value, 2))
                sub_property_details['modifier']['min'] = str(round(min_value, 2))
                sub_property_details['modifier']['max'] = str(round(max_value, 2))
                
            elif range_type == 'range_without_value':
                sub_property_details['modifier']['min'] = str(round(min_value, 2))
                sub_property_details['modifier']['max'] = str(round(max_value, 2))
                sub_property_details['modifier']['value'] = sub_property_details['modifier']['min'] 
            else:
                if value:
                    if exponential == 'positive':
                        # print("#"*25, 'positive')
                        if float_value < 0:
                            sub_property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 2))
                            sub_property_details['modifier']['max'] = "{:.2e}".format(round(float_value*0.9, 2))
                            sub_property_details['modifier']['min'] = "{:.2e}".format(round(float_value*1.1, 2))
                        else:
                            sub_property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 2))
                            sub_property_details['modifier']['min'] = "{:.2e}".format(round(float_value*0.9, 2))
                            sub_property_details['modifier']['max'] = "{:.2e}".format(round(float_value*1.1, 2))
                            
                    elif exponential == 'negative':
                        print("*"*25, 'negative exponential')
                        if float_value < 0:
                            sub_property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 9))
                            sub_property_details['modifier']['max'] = "{:.2e}".format(round(float_value*0.9, 9))
                            sub_property_details['modifier']['min'] = "{:.2e}".format(round(float_value*1.1, 9))
                        else:
                            sub_property_details['modifier']['value'] = "{:.2e}".format(round(float_value, 9))
                            sub_property_details['modifier']['min'] = "{:.2e}".format(round(float_value*0.9, 9))
                            sub_property_details['modifier']['max'] = "{:.2e}".format(round(float_value*1.1, 9))
                        print(sub_property_details)
                        
                    else:
                        if value < 0:
                            sub_property_details['modifier']['value'] = str(round(value, 2))
                            sub_property_details['modifier']['max'] = str(round(value*0.9, 2))
                            sub_property_details['modifier']['min'] = str(round(value*1.1, 2))
                        else:
                            sub_property_details['modifier']['value'] = str(round(value, 2))
                            sub_property_details['modifier']['min'] = str(round(value*0.9, 2))
                            sub_property_details['modifier']['max'] = str(round(value*1.1, 2))
                else:
                    sub_property_details['modifier']['value'] = None
    

            property_details = [sub_property_details, {"property_name": "minimum thickness (mm)", "modifier": {"value": '0.4', "min": None, "max": None, "unit": 'mm'}, "property_type": "ul_property"}]
        except:
            values = p_metadata[3]
            unit = "" 
            if type(values)==list:
                value = random.choice(values)
                if only_value and property_name_meaning in value_only_prop:
                    # print(f"adding only value for ul sub property {property_name_meaning}")
                    value = random.choice(value_only_prop[property_name_meaning])
                    query = str(value)
                    if value == 'vo':
                        value = 'v0'
                    elif value =='v-o':
                        value = 'v-0'
                    property_details = [{"property_name": property_name_meaning, "modifier": {"value": value, "min": None, "max": None, "unit": unit.strip()}, "property_type": "ul_sub_property"}, {"property_name": "minimum thickness (mm)", "modifier": {"value": '0.4', "min": None, "max": None, "unit": 'mm'}, "property_type": "ul_property"}]
                else:
                    queries = [
                        f"{property_name}-{value}{unit}",
                        
                        ####
                        f"{property_name} {value}{unit}",
                        f"{property_name} {value}{unit}",
                        
                        f"{property_name} of {value}{unit}",
                        f"{property_name} = {value}{unit}",
                        f"{property_name}={value}{unit}",
                        f"{property_name} - {value}{unit}",
                        
                        f"{property_name}: {value}{unit}",
                        f"{property_name} ({value}{unit})",
                        f"{value}{unit} {property_name}",
        
                        f"{property_name.replace(' ', '')} {value}{unit}",
                        f"{property_name.replace(' ', '')} of {value}{unit}",
        
                        ####
                        f"{property_name} {value}{unit}",
                        f"{property_name} {value}{unit}",
                        
                        f"{property_name} of {value}{unit}",
                        f"{property_name} = {value}{unit}",
                        f"{property_name}={value}{unit}",
                        f"{property_name} - {value}{unit}",
                        
                        f"{property_name}: {value}{unit}",
                        f"{property_name} ({value}{unit})",
                        f"{value}{unit} {property_name}",
        
                        f"{property_name.replace(' ', '')} {value}{unit}",
                        f"{property_name.replace(' ', '')} of {value}{unit}",
                    ]
                    query = random.choice(queries)
                    if value == 'vo':
                        value = 'v0'
                    elif value =='v-o':
                        value = 'v-0'
                    property_details = [{"property_name": property_name_meaning, "modifier": {"value": value, "min": None, "max": None, "unit": unit.strip()}, "property_type": "ul_sub_property"}, {"property_name": "minimum thickness (mm)", "modifier": {"value": '0.4', "min": None, "max": None, "unit": 'mm'}, "property_type": "ul_property"}]

    
    else:
        thickness_values = [0.1,0.3,0.5,0.6,0.7,0.8,0.9,1,1.2,1.3,1.5,1.6,2,2.2,3,3.1,3.5,3,5,6]
    
        try:
            try:    value = random.randrange(p_metadata[1],p_metadata[2])
            except: value = round(random.uniform(p_metadata[1], p_metadata[2]),2)
            unit = random.choice([p_metadata[4],f" {p_metadata[4]}","",""])
            if not unit:
                unit=""
            thickness = random.choice(thickness_values)
            queries = [
                f"{property_name} {value}{unit} at {thickness}mm",
                f"{property_name} at {thickness}mm {value}{unit}",
                f"{property_name} at {thickness}mm of {value}{unit}",
                f"{property_name} at {thickness} {value}{unit}",
                f"{property_name} at {thickness} of {value}{unit}",

                f"{property_name} @ {thickness} {value}{unit}",
                f"{property_name} @ {thickness} of {value}{unit}",
                
                f"{property_name} {thickness}mm of {value}{unit}",
                f"{property_name} {value}{unit}, {thickness}mm thickness",
                f"{property_name} {value}{unit} {thickness}mm thick.",
                f"min thickness {thickness}, {value}{unit} {property_name}",
                f"{value}{unit} {property_name} @ {thickness}mm",

                f"{property_name.replace(' ', '')} {value}{unit} at {thickness}mm",
                f"{value}{unit} {property_name.replace(' ', '')} @ {thickness}mm",
            ]
            query = random.choice(queries)
            property_details = [{"property_name": property_name_meaning, "modifier": {"value": value, "min": round(value-value*10/100,2), "max": round(value+value*10/100,2), "unit": unit.strip()}, "property_type": "ul_sub_property"}, {"property_name": "minimum thickness (mm)", "modifier": {"value": str(thickness), "min": None, "max": None, "unit": 'mm'}, "property_type": "ul_property"}]
        except:
            values = p_metadata[3]
            unit = "" 
            if type(values)==list:
                value = random.choice(values)
                thickness = random.choice(thickness_values)
                if only_value and property_name_meaning in value_only_prop:
                    # print(f"adding only value for ul sub property {property_name_meaning}")
                    value = random.choice(value_only_prop[property_name_meaning])
                    queries = [
                        f"{value}{unit} at {thickness}mm",
                        f"{thickness}mm {value}{unit}",
                        f"{thickness}mm {value}{unit}",
                        f"{thickness} {value}{unit}",
                        
                        f"{value}{unit}, {thickness}mm thickness",
                        f"{value}{unit} {thickness}mm thickness",
                        f"{value}{unit} {thickness}mm thick.",
                        f"min thickness {thickness}, {value}{unit}",
                        f"min thickness {thickness} {value}{unit}",
                        f"{value}{unit} @ {thickness}mm",
                    ]
                    query = random.choice(queries)
                    if value == 'vo':
                        value = 'v0'
                    elif value =='v-o':
                        value = 'v-0'
                    property_details = [{"property_name": property_name_meaning, "modifier": {"value": value, "min": None, "max": None, "unit": ''}, "property_type": "ul_sub_property"}, {"property_name": "minimum thickness (mm)", "modifier": {"value": str(thickness), "min": None, "max": None, "unit": 'mm'}, "property_type": "ul_property"}]
                else:
                    queries = [
                        f"{property_name} {value}{unit} at {thickness}mm",
                        f"{property_name} at {thickness}mm {value}{unit}",
                        f"{property_name} at {thickness}mm of {value}{unit}",
                        f"{property_name} at {thickness} {value}{unit}",
                        f"{property_name} at {thickness} of {value}{unit}",
        
                        f"{property_name} @ {thickness} {value}{unit}",
                        f"{property_name} @ {thickness} of {value}{unit}",
                        
                        f"{property_name} {thickness}mm of {value}{unit}",
                        f"{property_name} {value}{unit}, {thickness}mm thickness",
                        f"{property_name} {value}{unit} {thickness}mm thick.",
                        f"min thickness {thickness}, {value}{unit} {property_name}",
                        f"{value}{unit} {property_name} @ {thickness}mm",
                        
                        f"{property_name.replace(' ', '')} {value}{unit} at {thickness}mm",
                        f"{value}{unit} {property_name.replace(' ', '')} @ {thickness}mm",
                    ]
                    query = random.choice(queries)
                    if value == 'vo':
                        value = 'v0'
                    elif value =='v-o':
                        value = 'v-0'
                        
                    property_details = [{"property_name": property_name_meaning, "modifier": {"value": value, "min": None, "max": None, "unit": unit.strip()}, "property_type": "ul_sub_property"}, {"property_name": "minimum thickness (mm)", "modifier": {"value": str(thickness), "min": None, "max": None, "unit": 'mm'}, "property_type": "ul_property"}]

    
    return property_details, query

In [165]:
for i in range(1, 100):
    print(get_ul_sub_property(), "\n")

([{'property_name': 'relative thermal index - mechanical impact (rti imp) (°c)', 'modifier': {'value': 216, 'min': 194.4, 'max': 237.6, 'unit': ''}, 'property_type': 'ul_sub_property'}, {'property_name': 'minimum thickness (mm)', 'modifier': {'value': '0.7', 'min': None, 'max': None, 'unit': 'mm'}, 'property_type': 'ul_property'}], '216 relative thermal index impact @ 0.7mm') 

([{'property_name': 'glow wire flammability index', 'modifier': {'value': 658, 'min': 592.2, 'max': 723.8, 'unit': ''}, 'property_type': 'ul_sub_property'}, {'property_name': 'minimum thickness (mm)', 'modifier': {'value': '0.8', 'min': None, 'max': None, 'unit': 'mm'}, 'property_type': 'ul_property'}], 'gwfindex 658 at 0.8mm') 

([{'property_name': 'relative thermal index - electrical (rti elec) (°c)', 'modifier': {'value': '116', 'min': '86', 'max': '147', 'unit': None}, 'property_type': 'ul_sub_property'}, {'property_name': 'minimum thickness (mm)', 'modifier': {'value': '0.4', 'min': None, 'max': None, 'unit

#### Multi UL Sub-Properties

In [166]:
def get_ul_sub_properties():
    pdetails, q = [], ''
    pd1, q1 = get_ul_sub_property(True)
    pd2, q2 = get_ul_sub_property(False)
    
    pdetails = []
    if pd2[0]['property_name'] != pd1[0]['property_name'] == pd1[0]['property_name']!= 'minimum thickness (mm)':
        num = random.choice([1, 1, 2])
        if num==1:
            q = random.choice([
                f"{q1} {q2}",
                f"{q1}, {q2}",
                f"{q1} and {q2}",
            ])
            pdetails.append(pd1[0]) 
            pdetails.append(pd2[0]) 
        else:
            q = random.choice([
                f"{q2} {q1}",
                f"{q2}, {q1}",
                f"{q2} and {q1}",
            ])
            pdetails.append(pd2[0]) 
            pdetails.append(pd1[0]) 
            
        pdetails.append(pd2[1])
    else:
        print("got same properties")
        pdetails, q = get_ul_sub_properties()
        
    return pdetails, q

In [167]:
for i in range(1, 20):
    print(get_ul_sub_properties(), "\n")

([{'property_name': 'hot wire ignition (hwi)', 'modifier': {'value': 'plc-3', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_sub_property'}, {'property_name': 'glow wire flammability index', 'modifier': {'value': '896', 'min': None, 'max': '896', 'unit': None}, 'property_type': 'ul_sub_property'}, {'property_name': 'minimum thickness (mm)', 'modifier': {'value': '0.9', 'min': None, 'max': None, 'unit': 'mm'}, 'property_type': 'ul_property'}], 'hot-wire ignition at 0.9mm of plc-3, glow wire flammability index lesser than or equal to 896') 

([{'property_name': 'flame rating', 'modifier': {'value': 'v1', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_sub_property'}, {'property_name': 'high-current arc ignition (hai)', 'modifier': {'value': 'plc 4', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'ul_sub_property'}, {'property_name': 'minimum thickness (mm)', 'modifier': {'value': '2.2', 'min': None, 'max': None, 'unit': 'mm'}, 'property_type': 'ul_proper

### Certifications (excluding Auto Certification)

In [168]:
for item in unique_values['Certification']:
    if item['CERTIFICATION_TYPE']=='NSF':
        nsf_certifications = item['CERTIFICATIONS']
    if item['CERTIFICATION_TYPE']=='RAILWAY SPECIFICATION':
        railway_certifications = item['CERTIFICATIONS']
    if item['CERTIFICATION_TYPE']=='WATER APPROVAL':
        water_certifications = item['CERTIFICATIONS']

water_certifications_combinations = [
    ['NSF 61 at 23°C', 'NSF 61', '23'],
    ['NSF 61 at 82°C','NSF 61', '82'],
    ['NSF 61 at 60°C','NSF 61', '60'],
    ['WRAS 23°C','WRAS','23'],
    ['WRAS 60°C','WRAS','60'],
    ['KTW BWGL at 23°C','KTW BWGL', '23'],
    ['KTW BWGL at 60°C','KTW BWGL', '60'],
    ['WRAS 85°C','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23°C','ACS','23'],
    ['KTW BWGL at 85°C','KTW BWGL', '85'],
    ['ACS 85°C','ACS','85'],
    ['ACS 60°C','ACS','60'],
    ['NSF 61 at 23°C', 'NSF 61', '23'],
    ['NSF 61 at 82°C','NSF 61', '82'],
    ['NSF 61 at 60°C','NSF 61', '60'],
    ['WRAS 23°C','WRAS','23'],
    ['WRAS 60°C','WRAS','60'],
    ['KTW BWGL at 23°C','KTW BWGL', '23'],
    ['KTW BWGL at 60°C','KTW BWGL', '60'],
    ['WRAS 85°C','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23°C','ACS','23'],
    ['KTW BWGL at 85°C','KTW BWGL', '85'],
    ['ACS 85°C','ACS','85'],
    ['ACS 60°C','ACS','60'],
    ['NSF 61 at 23°C', 'NSF 61', '23'],
    ['NSF 61 at 82°C','NSF 61', '82'],
    ['NSF 61 at 60°C','NSF 61', '60'],
    ['WRAS 23°C','WRAS','23'],
    ['WRAS 60°C','WRAS','60'],
    ['KTW BWGL at 23°C','KTW BWGL', '23'],
    ['KTW BWGL at 60°C','KTW BWGL', '60'],
    ['WRAS 85°C','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23°C','ACS','23'],
    ['KTW BWGL at 85°C','KTW BWGL', '85'],
    ['ACS 85°C','ACS','85'],
    ['ACS 60°C','ACS','60'],    

    ['NSF 61 at 23 degrees', 'NSF 61', '23'],
    ['NSF 61 at 82 degrees','NSF 61', '82'],
    ['NSF 61 at 60 degrees','NSF 61', '60'],
    ['WRAS 23 degrees', 'WRAS', '23'],
    ['WRAS 60 degrees', 'WRAS', '60'],
    ['KTW BWGL at 23 degrees','KTW BWGL', '23'],
    ['KTW BWGL at 60 degrees','KTW BWGL', '60'],
    ['WRAS 85 degrees','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23 degrees','ACS','23'],
    ['KTW BWGL at 85 degrees','KTW BWGL', '85'],
    ['ACS 85 degrees','ACS','85'],
    ['ACS 60 degrees','ACS','60'],
    ['NSF 61 at 23 degree celsius', 'NSF 61', '23'],
    ['NSF 61 at 82 degree celsius','NSF 61', '82'],
    ['NSF 61 at 60 degree celsius','NSF 61', '60'],
    ['WRAS 23 degree celsius','WRAS','23'],
    ['WRAS 60 degree celsius','WRAS','60'],
    ['KTW BWGL at 23 degree celsius','KTW BWGL', '23'],
    ['KTW BWGL at 60 degree celsius','KTW BWGL', '60'],
    ['WRAS 85 degree celsius','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23 degree celsius','ACS','23'],
    ['KTW BWGL at 85 degree celsius','KTW BWGL', '85'],
    ['ACS 85 degree celsius','ACS','85'],
    ['ACS 60 degree celsius','ACS','60'],
    ['NSF 61 at 23 celsius', 'NSF 61', '23'],
    ['NSF 61 at 82 celsius','NSF 61', '82'],
    ['NSF 61 at 60 celsius','NSF 61', '60'],
    ['WRAS 23 celsius','WRAS','23'],
    ['WRAS 60 celsius','WRAS','60'],
    ['KTW BWGL at 23 celsius','KTW BWGL', '23'],
    ['KTW BWGL at 60 celsius','KTW BWGL', '60'],
    ['WRAS 85 celsius','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23 celsius','ACS','23'],
    ['KTW BWGL at 85 celsius','KTW BWGL', '85'],
    ['ACS 85 celsius','ACS','85'],
    ['ACS 60 celsius','ACS','60'],   
    ['NSF 61 at 23 deg celsius', 'NSF 61', '23'],
    ['NSF 61 at 82 deg celsius','NSF 61', '82'],
    ['NSF 61 at 60 deg celsius','NSF 61', '60'],
    ['WRAS 23 deg celsius','WRAS','23'],
    ['WRAS 60 deg celsius','WRAS','60'],
    ['KTW BWGL at 23 deg celsius','KTW BWGL', '23'],
    ['KTW BWGL at 60 deg celsius','KTW BWGL', '60'],
    ['WRAS 85 deg celsius','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23 deg celsius','ACS','23'],
    ['KTW BWGL at 85 deg celsius','KTW BWGL', '85'],
    ['ACS 85 deg celsius','ACS','85'],
    ['ACS 60 deg celsius','ACS','60'],
    ['NSF 61 at 23c', 'NSF 61', '23'],
    ['NSF 61 at 82c','NSF 61', '82'],
    ['NSF 61 at 60c','NSF 61', '60'],
    ['WRAS 23c','WRAS','23'],
    ['WRAS 60c','WRAS','60'],
    ['KTW BWGL at 23c','KTW BWGL', '23'],
    ['KTW BWGL at 60c','KTW BWGL', '60'],
    ['WRAS 85c','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23c','ACS','23'],
    ['KTW BWGL at 85c','KTW BWGL', '85'],
    ['ACS 85c','ACS','85'],
    ['ACS 60c','ACS','60'],
    ['NSF 61 at 23 deg c', 'NSF 61', '23'],
    ['NSF 61 at 82 deg c','NSF 61', '82'],
    ['NSF 61 at 60 deg c','NSF 61', '60'],
    ['WRAS 23 deg c','WRAS','23'],
    ['WRAS 60 deg c','WRAS','60'],
    ['KTW BWGL at 23 deg c','KTW BWGL', '23'],
    ['KTW BWGL at 60 deg c','KTW BWGL', '60'],
    ['WRAS 85 deg c','WRAS','85'],
    ['EN16421','EN16421','ALL'],
    ['ACS 23 deg c','ACS','23'],
    ['KTW BWGL at 85 deg c','KTW BWGL', '85'],
    ['ACS 85 deg c','ACS','85'],
    ['ACS 60 deg c','ACS','60'],

    ['WRAS', 'WRAS', 'ALL'],
    ['WRAS', 'WRAS', 'ALL'],
    ['ACS', 'ACS', 'ALL'],
    ['ACS', 'ACS', 'ALL'],
    ['KTW', 'KTW BWGL', 'ALL'],
    ['KTW', 'KTW BWGL', 'ALL'],
    ['KTW BWGL', 'KTW BWGL', 'ALL'],
    ['KTW BWGL', 'KTW BWGL', 'ALL'],
    ['BWGL', 'KTW BWGL', 'ALL'],
    ['BWGL', 'KTW BWGL', 'ALL'],

    ['hot water', 'all', 'hot'],
    ['hot-water', 'all', 'hot'],

    ['commerical-hot water', 'all', 'c-hot'],
    ['commerical-hot', 'all', 'c-hot'],
    ['c-hot water', 'all', 'c-hot'],
    ['c-hot', 'all', 'c-hot'], 
    
    ['domestic-hot water', 'all', 'd-hot'],
    ['domestic-hot', 'all', 'd-hot'],
    ['d-hot water', 'all', 'd-hot'],
    ['d-hot', 'all', 'd-hot'],

    ['cold water', 'all', 'cold'],
    ['c-water', 'all', 'cold'],
    

]
railway_certifications_combinations = [(i, i.split(" - ")[0], i.split(" - ")[1].split(" (")[0],"EN 45545-2") for i in railway_certifications]

In [169]:
railway_certifications_combinations

[('R23 - HL3 (EN 45545-2)', 'R23', 'HL3', 'EN 45545-2'),
 ('R24 - HL3 (EN 45545-2)', 'R24', 'HL3', 'EN 45545-2'),
 ('R22 - HL2 (EN 45545-2)', 'R22', 'HL2', 'EN 45545-2'),
 ('R26 - V0 (EN 45545-2)', 'R26', 'V0', 'EN 45545-2'),
 ('R22 - HL3 (EN 45545-2)', 'R22', 'HL3', 'EN 45545-2'),
 ('R24 - HL2 (EN 45545-2)', 'R24', 'HL2', 'EN 45545-2'),
 ('R23 - HL2 (EN 45545-2)', 'R23', 'HL2', 'EN 45545-2'),
 ('R22 - HL1 (EN 45545-2)', 'R22', 'HL1', 'EN 45545-2')]

In [170]:
nsf_certifications

['NSF 61', 'NSF 51', 'NSF 42', 'NSF 372']

### Auto Approvals

In [171]:
auto_certs = {}
# for cert_details in unique_values['Auto_Approval']:
for cert_details in unique_values_cert['Auto_Approval']: 
    if cert_details['OEM_Name'].lower() not in auto_certs:
        auto_certs[cert_details['OEM_Name'].lower()] = [x.lower() for x in cert_details['CERTIFICATIONS']]
    else:
        auto_certs[cert_details['OEM_Name'].lower()] += [x.lower() for x in cert_details['CERTIFICATIONS']]

auto_certs

{'baic': ['q-bjev 01.59', 'q-bjev 01.33', 'bas-491', 'bas-492'],
 'bmw': ['gs93017',
  'gs93016',
  'gs97014',
  'gs93016-pbt+asa-gf20',
  'gs93016-pbt+asa-gf30',
  'gs93016-pbt',
  'gs93016-pbt-gf15',
  'gs93016-pbt-gf20',
  'gs93016-pbt-gf30',
  'gs93016-pbt-gb20',
  'gs93016-pet-gf30',
  'gs93016-pet-gf45',
  'gs93016-pet-(mx20+gf15)',
  'gs93042',
  'gs93016-pa66',
  'gs93016-pa66-gf30',
  'gs93016-pa66-gf35',
  'gs93016-pa66-gf50',
  'gs93016-pa6-gf15',
  'gs93016-pa6-gf30',
  'gs93016-pa66-gf15',
  'gs93016-pa66-gf25'],
 'byd': ['byd-ty-d13f05-0106', 'byd-ty-d13f05-0043'],
 'bosch': ['n28 bn07-gf010',
  'n28 bn07-o001',
  'n28 bn07-gf014',
  'n28 bn07-gf032',
  'n28 bn08-gf009',
  'n28 bn08-gf013',
  'n28 bn07-gf028',
  'n28 bn22-x003',
  'n28 bn22-x009',
  'n28 bn22-o034',
  'n28 bn09-gf027',
  'n28 bn09-gf026',
  'n28 bn09-gf024',
  'n28 bn07-gf019',
  'n28 bn08-gf018',
  'n28 bn07-gf003',
  'n28 bn07-gf012',
  'n28 bn07-gf023',
  'n28 bn07-gf051',
  'n28 bn14-gf010',
  'n28 bn

In [172]:
remove_cert_with_oem = []
ignore_certs = [
    ("bosch", "gs93016-pa66"),
    ("nissan", "as26"),
    ("geely", "q/jly j7111001a-2016â‘¢"),
    ("geely", "q/jly j7110235b-2018â‘¡"),
    # ("stellantis-chrysler", "b62 0300"),
    # ("stellantis-fca group", "b62 0300"),
]

for oem in auto_certs:
    for x in auto_certs[oem]:
        if re.match(fr'^{re.escape(oem)}', x.strip()):
            new_cert = re.sub(fr'^{re.escape(oem)}', '', x.strip()).strip('- ')
            try:
                eval(new_cert)
            except:
                print(f"adding '{new_cert}' under '{oem}' oem")
                auto_certs[oem].append(new_cert)
                remove_cert_with_oem.append((oem, x))

ignore_certs += remove_cert_with_oem
for oem in auto_certs:
    temp = []
    for i in ignore_certs:
        if i[0] == oem:
            temp.append(i[1])

    auto_certs[oem] = [re.sub("(\s+)", " ", x.strip().lower()) for x in auto_certs[oem] if x not in temp]

ignore_certs

adding 'ty-d13f05-0106' under 'byd' oem
adding 'ty-d13f05-0043' under 'byd' oem
adding 'm3236' under 'man' oem
adding 'm3236 a7' under 'man' oem
adding 'sm.51.010-c6' under 'nio' oem
adding 'sm.51.010-c4' under 'nio' oem
adding 'sm.51.010' under 'nio' oem
adding 'sm.51.041' under 'nio' oem
adding 'sm.51.007-c2' under 'nio' oem
adding 'sm.51.003' under 'nio' oem


[('bosch', 'gs93016-pa66'),
 ('nissan', 'as26'),
 ('geely', 'q/jly j7111001a-2016â‘¢'),
 ('geely', 'q/jly j7110235b-2018â‘¡'),
 ('byd', 'byd-ty-d13f05-0106'),
 ('byd', 'byd-ty-d13f05-0043'),
 ('man', 'man m3236'),
 ('man', 'man m3236 a7'),
 ('nio', 'nio-sm.51.010-c6'),
 ('nio', 'nio-sm.51.010-c4'),
 ('nio', 'nio-sm.51.010'),
 ('nio', 'nio-sm.51.041'),
 ('nio', 'nio-sm.51.007-c2'),
 ('nio', 'nio-sm.51.003')]

In [173]:
# identify duplicates

for oem in auto_certs:
    for cert in auto_certs[oem]:
        for oem2 in auto_certs:
            if oem!=oem2 and cert in auto_certs[oem2]:
                print(f"'{cert}' present in '{oem}' and '{oem2}'")      

In [174]:
railway_certifications_combinations

[('R23 - HL3 (EN 45545-2)', 'R23', 'HL3', 'EN 45545-2'),
 ('R24 - HL3 (EN 45545-2)', 'R24', 'HL3', 'EN 45545-2'),
 ('R22 - HL2 (EN 45545-2)', 'R22', 'HL2', 'EN 45545-2'),
 ('R26 - V0 (EN 45545-2)', 'R26', 'V0', 'EN 45545-2'),
 ('R22 - HL3 (EN 45545-2)', 'R22', 'HL3', 'EN 45545-2'),
 ('R24 - HL2 (EN 45545-2)', 'R24', 'HL2', 'EN 45545-2'),
 ('R23 - HL2 (EN 45545-2)', 'R23', 'HL2', 'EN 45545-2'),
 ('R22 - HL1 (EN 45545-2)', 'R22', 'HL1', 'EN 45545-2')]

In [175]:
auto_certifications = []
for oem in auto_certs:
    for cert in auto_certs[oem]:
        auto_certifications.append((oem, cert))
        
    if len(auto_certs[oem]) < 5:
        for cert in auto_certs[oem]:
            auto_certifications.append((oem, cert))

        for cert in auto_certs[oem]:
            auto_certifications.append((oem, cert))

    # add again
    if len(auto_certs[oem]) < 3:
        for cert in auto_certs[oem]:
            auto_certifications.append((oem, cert))

        for cert in auto_certs[oem]:
            auto_certifications.append((oem, cert))

In [176]:
auto_certifications[:5]

[('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('baic', 'q-bjev 01.59')]

In [177]:
print(len(auto_certifications))

973


In [178]:
auto_syns = synonym_df2[synonym_df2['TYPE']=='auto cert'][['DEFINED_NAME', 'SYNONYMS']].groupby(by='DEFINED_NAME')['SYNONYMS'].apply(list).to_dict()
for i in auto_syns:
    if i not in auto_syns[i]:
        auto_syns[i].append(i)
        
auto_syns

{'mercedes-benz': ['mercedes-benz',
  'daimler',
  'daimler-benz',
  'daimler mercedes',
  'daimler mercedes-benz',
  'daimler mercedes-benz group',
  'mercedes',
  'mercedes-benz group (daimier)'],
 'vw group': ['vw group', 'vw', 'bentley']}

In [179]:
auto_syns_mapping = {}
for meaning, syns in auto_syns.items():
    for syn in syns:
        auto_syns_mapping[syn] = meaning

auto_syns_mapping

{'mercedes-benz': 'mercedes-benz',
 'daimler': 'mercedes-benz',
 'daimler-benz': 'mercedes-benz',
 'daimler mercedes': 'mercedes-benz',
 'daimler mercedes-benz': 'mercedes-benz',
 'daimler mercedes-benz group': 'mercedes-benz',
 'mercedes': 'mercedes-benz',
 'mercedes-benz group (daimier)': 'mercedes-benz',
 'vw group': 'vw group',
 'vw': 'vw group',
 'bentley': 'vw group'}

#### Auto Certifications

In [180]:
auto_cert = random.choice(auto_certifications)
auto_name = auto_cert[0]
auto_cert_name = auto_cert[1]
print(auto_name)
print(auto_cert_name)

stellantis-chrysler
cpn 2776 ms.50017


In [181]:
auto_certifications

[('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('bmw', 'gs93017'),
 ('bmw', 'gs93016'),
 ('bmw', 'gs97014'),
 ('bmw', 'gs93016-pbt+asa-gf20'),
 ('bmw', 'gs93016-pbt+asa-gf30'),
 ('bmw', 'gs93016-pbt'),
 ('bmw', 'gs93016-pbt-gf15'),
 ('bmw', 'gs93016-pbt-gf20'),
 ('bmw', 'gs93016-pbt-gf30'),
 ('bmw', 'gs93016-pbt-gb20'),
 ('bmw', 'gs93016-pet-gf30'),
 ('bmw', 'gs93016-pet-gf45'),
 ('bmw', 'gs93016-pet-(mx20+gf15)'),
 ('bmw', 'gs93042'),
 ('bmw', 'gs93016-pa66'),
 ('bmw', 'gs93016-pa66-gf30'),
 ('bmw', 'gs93016-pa66-gf35'),
 ('bmw', 'gs93016-pa66-gf50'),
 ('bmw', 'gs93016-pa6-gf15'),
 ('bmw', 'gs93016-pa6-gf30'),
 ('bmw', 'gs93016-pa66-gf15'),
 ('bmw', 'gs93016-pa66-gf25'),
 ('byd', 'ty-d13f05-0106'),
 ('byd', 'ty-d13f05-0043'),
 ('byd', 'ty-d1

In [182]:
additional_autocerts = [
    ('li auto', 'lia5310020'),
    ('li auto', 'lia5310038'),
    ('li auto', 'lia5310050'),
    ('li auto', 'lia5310057'),
    ('li auto', 'q-lia5310020'),
    ('li auto', 'q-lia5310038'),
    ('li auto', 'q-lia5310050'),
    ('li auto', 'q-lia5310057'),
    ("geely", "q/jly j7111001a-2016"),
    ("geely", "q/jly j7110235b-2018"),
    ("geely", "q/jly j7111001a-2016"),
    ("geely", "q/jly j7110235b-2018"),
]

auto_certifications += additional_autocerts
auto_certifications

[('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('baic', 'q-bjev 01.59'),
 ('baic', 'q-bjev 01.33'),
 ('baic', 'bas-491'),
 ('baic', 'bas-492'),
 ('bmw', 'gs93017'),
 ('bmw', 'gs93016'),
 ('bmw', 'gs97014'),
 ('bmw', 'gs93016-pbt+asa-gf20'),
 ('bmw', 'gs93016-pbt+asa-gf30'),
 ('bmw', 'gs93016-pbt'),
 ('bmw', 'gs93016-pbt-gf15'),
 ('bmw', 'gs93016-pbt-gf20'),
 ('bmw', 'gs93016-pbt-gf30'),
 ('bmw', 'gs93016-pbt-gb20'),
 ('bmw', 'gs93016-pet-gf30'),
 ('bmw', 'gs93016-pet-gf45'),
 ('bmw', 'gs93016-pet-(mx20+gf15)'),
 ('bmw', 'gs93042'),
 ('bmw', 'gs93016-pa66'),
 ('bmw', 'gs93016-pa66-gf30'),
 ('bmw', 'gs93016-pa66-gf35'),
 ('bmw', 'gs93016-pa66-gf50'),
 ('bmw', 'gs93016-pa6-gf15'),
 ('bmw', 'gs93016-pa6-gf30'),
 ('bmw', 'gs93016-pa66-gf15'),
 ('bmw', 'gs93016-pa66-gf25'),
 ('byd', 'ty-d13f05-0106'),
 ('byd', 'ty-d13f05-0043'),
 ('byd', 'ty-d1

In [183]:
all_oems = []
for i in auto_certifications:
    if i[0] not in all_oems:
        all_oems.append(i[0])

all_oems

['baic',
 'bmw',
 'byd',
 'bosch',
 'catl',
 'changan',
 'chery',
 'continental',
 'dongfeng motor',
 'evergrande auto',
 'faw group',
 'ford',
 'geely',
 'general motors (gm)',
 'great wall motor',
 'honda',
 'hyundai',
 'iveco',
 'li auto',
 'man',
 'mercedes-benz',
 'nio',
 'nissan',
 'saic motor',
 'stellantis-chrysler',
 'stellantis-fca group',
 'stellantis-psa group',
 'tesla',
 'vw group',
 'zf group']

In [184]:
ignore_common_values =  processing_types + product_functions + delivery_forms + all_brands + all_polymers
temp = list(processing_syns.values()) + list(feature_syns.values()) + list(delivery_form_syns.values()) + list(polymer_syns.values()) + list(brand_syns.values())
for i in temp:
    ignore_common_values.extend(i)

ignore_common_values

['injection molding',
 'other extrusion',
 'film extrusion',
 'profile extrusion',
 'sheet extrusion',
 'multi injection molding',
 'coextrusion',
 'compression molding',
 'blow molding',
 'calendering',
 'selective reinforcement',
 'thermoforming',
 'transfer molding',
 'coatable',
 'casting',
 'porous sintering',
 'gel extrusion',
 'fiber spinning / gel spinning',
 'ram extrusion',
 'extrusion blow molding',
 'rotational molding',
 'foam processing',
 'extrusion - wire and cable',
 'extrusion - small tubing',
 'injection blow molding',
 'extrusion - hose',
 'blowmolding',
 'bm',
 'blow moulding',
 'blow moldable',
 'blow molded',
 'calandering',
 'calendered',
 'calendr',
 'calenderable',
 'castable',
 'coating',
 'polymeric coating',
 'spin coating',
 'compression moulding',
 'matched-die moulding',
 'matched-die molding',
 'matched die molding',
 'matched die moulding',
 'cold compression',
 'hot compression',
 'compmold',
 'compression moldable',
 'compression mouldable',
 'hose e

In [185]:
auto_syns_mapping

{'mercedes-benz': 'mercedes-benz',
 'daimler': 'mercedes-benz',
 'daimler-benz': 'mercedes-benz',
 'daimler mercedes': 'mercedes-benz',
 'daimler mercedes-benz': 'mercedes-benz',
 'daimler mercedes-benz group': 'mercedes-benz',
 'mercedes': 'mercedes-benz',
 'mercedes-benz group (daimier)': 'mercedes-benz',
 'vw group': 'vw group',
 'vw': 'vw group',
 'bentley': 'vw group'}

In [186]:
# needs to be added to the synonym table
auto_syns_mapping['mercedes'] = 'mercedes-benz'

## Applications

In [187]:
app_industry_df = app_industry_df[['Medical & Pharma', 'Consumer Goods', 'Automotive & Transportation',
       'Electrical & Electronics', 'Industrial', 'Not Mapped',]]
app_industry_df

,Medical & Pharma,Consumer Goods,Automotive & Transportation,Electrical & Electronics,Industrial,Not Mapped
0,connection ring for natural glove,box for specimen collection,anti-slip mat for door storage box,bobbin housing for dishwasher,rod for knee replacement sizer,aerosol valve for over the counter (otc) valve...
1,autoinjector-anaphylaxis,"sprayers-throat tubes, nebulization inserts, j...",backlit knob-actuator,bracket for smartphone,filter bottle-shaft parts,antenna for wireless temperature probe
2,autoinjector-retraction assembly,tube for disinfection cabinet,carrier for back rest,projector-lock,hose for technical gardening,gear housing for pain management drug delivery...
3,clamp for feeding catheter,guide for paper transport,clip for recliner,lithium ion battery separator (libs) for power...,refrigerator-spring finger,tablet dispenser
4,dispension monitoring for diabetes pharmaceuti...,remote control for rudder,duct for air conditioner (ac),protection channel for wiring harness,wire control kit for firearm furniture,vending machine for hospital linen
...,...,...,...,...,...,...
213,NaN,NaN,trunk components,NaN,NaN,NaN
214,NaN,NaN,tube for air suspension,NaN,NaN,NaN
215,NaN,NaN,turn signal-shaft base,NaN,NaN,NaN
216,NaN,NaN,web sense retractor for seat belt,NaN,NaN,NaN


In [188]:
app_industry_df = app_industry_df.fillna('')
app_industry_data = app_industry_df.to_dict('list')
app_industry_data

{'Medical & Pharma': ['connection ring for natural glove',
  'autoinjector-anaphylaxis',
  'autoinjector-retraction assembly',
  'clamp for feeding catheter',
  'dispension monitoring for diabetes pharmaceutical preparation',
  'dose button for insulin pen',
  'drug delivery systems',
  'handle for flexible endoscopy',
  'hospital diagnostics & imaging',
  'metered dose inhaler (mdi)-dose counter',
  'plunger for intra ocular lens (iol)',
  'power transmission for metered dose inhaler (mdi)',
  'rod for orthopedic sizing device',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  

In [189]:
app_industry_df2 = app_industry_df2[['Medical & Pharma', 'Consumer Goods', 'Automotive & Transportation',
       'Electrical & Electronics', 'Industrial', 'Not Mapped',]]
app_industry_df2

,Medical & Pharma,Consumer Goods,Automotive & Transportation,Electrical & Electronics,Industrial,Not Mapped
0,hips implant,washing machine pump,active grille,Electrical & Electronics,abrasion sheet,junction box
1,syringe stopper,electric stove,adas,circuit breaker,water meter housing,junction box plastic
2,prosthetic dog ear,cooker handle,ags vanes,5g antenna,conveyor belts,busbar
3,hip implant,refrigerator,aircraft,5g antenna for electrical & electronics,seal for industrial headlights,oil pump plastic
4,orthopedic implant,beer,airspring,power electronics insulation,sliding for farm machine,nylbond
...,...,...,...,...,...,...
410,NaN,NaN,NaN,NaN,NaN,wine adapter
411,NaN,NaN,NaN,NaN,NaN,profiles
412,NaN,NaN,NaN,NaN,NaN,bus bars
413,NaN,NaN,NaN,NaN,NaN,overmolding


In [190]:
app_industry_df2 = app_industry_df2.fillna('')
app_industry_data2 = app_industry_df2.to_dict('list')
app_industry_data2

{'Medical & Pharma': ['hips implant',
  'syringe stopper',
  'prosthetic dog ear',
  'hip implant',
  'orthopedic implant',
  'medical',
  'peristaltic pump gears',
  'peristaltic tub',
  'housing in a medical',
  'medical',
  'cap for laboratory glass bottle',
  'peristaltic tubing',
  'dental part',
  'glucose monitor',
  'drug delivery',
  'metered dose counter',
  'cryogenic',
  'dental',
  'screw in injection device',
  'drug elution',
  'peristaltic pump',
  'implanted medical devices',
  'medical devices',
  'insulin pen',
  'wearable insulin pump',
  'auto-injector',
  'wearable pump',
  'plunger stopper for anesthesia syringe',
  'drug delivery device',
  'autoinjectors that is a good sliding partner',
  'lead screws in autoinjectors',
  'healtcare',
  'hip replacement',
  'x-ray',
  'connector port for iv bag',
  'implantable',
  'controlled release of drugs',
  'autoinjector pen plunger',
  'knees implant',
  'autoinjectors',
  'implant',
  'insulin pump',
  'mdi valve',
  '

In [191]:
app_industry_data_cleaned = {}
for i in app_industry_data:
    app_industry_data_cleaned[i.lower()] = [re.sub("(\s+)", " ", x.strip().lower()) for x in app_industry_data[i] if x] + [re.sub("(\s+)", " ", x.strip().lower()) for x in app_industry_data2[i] if x]

app_industry_data_cleaned

{'medical & pharma': ['connection ring for natural glove',
  'autoinjector-anaphylaxis',
  'autoinjector-retraction assembly',
  'clamp for feeding catheter',
  'dispension monitoring for diabetes pharmaceutical preparation',
  'dose button for insulin pen',
  'drug delivery systems',
  'handle for flexible endoscopy',
  'hospital diagnostics & imaging',
  'metered dose inhaler (mdi)-dose counter',
  'plunger for intra ocular lens (iol)',
  'power transmission for metered dose inhaler (mdi)',
  'rod for orthopedic sizing device',
  'hips implant',
  'syringe stopper',
  'prosthetic dog ear',
  'hip implant',
  'orthopedic implant',
  'medical',
  'peristaltic pump gears',
  'peristaltic tub',
  'housing in a medical',
  'medical',
  'cap for laboratory glass bottle',
  'peristaltic tubing',
  'dental part',
  'glucose monitor',
  'drug delivery',
  'metered dose counter',
  'cryogenic',
  'dental',
  'screw in injection device',
  'drug elution',
  'peristaltic pump',
  'implanted medi

In [192]:
for ind in app_industry_data_cleaned:
    print(len(app_industry_data_cleaned[ind]))

74
180
550
83
94
519


In [193]:
for ind in app_industry_data_cleaned:
    for app in app_industry_data_cleaned[ind]:
        for ind2 in app_industry_data_cleaned:
            if ind != ind2 and app in app_industry_data_cleaned[ind2]:
                print(f"'{app}' present in {ind} and {ind2}")

In [194]:
for ind in app_industry_data_cleaned:
    temp = []
    for app in app_industry_data_cleaned[ind]:
        if app not in temp:
            temp.append(app)

    app_industry_data_cleaned[ind] = temp

for ind in app_industry_data_cleaned:
    print(len(app_industry_data_cleaned[ind]))

70
178
546
83
94
508


In [195]:
for ind in app_industry_data_cleaned:
    for app in app_industry_data_cleaned[ind]:
        if len(app) < 4:
            print(app)

wig
cac
chc
fcv
grc
ip
tmm
ev
ddr
rod
5g
pv
ccm
ewp
fan
oil


In [196]:
ignore_apps = [
    'ip',
    'pv',
    'oil',
    'water',
    'medical',
]

for ind in app_industry_data_cleaned:
    app_industry_data_cleaned[ind] = [app for app in app_industry_data_cleaned[ind] if app not in ignore_apps]

for ind in app_industry_data_cleaned:
    for app in app_industry_data_cleaned[ind]:
        if len(app) < 4:
            print(app)

wig
cac
chc
fcv
grc
tmm
ev
ddr
rod
5g
ccm
ewp
fan


In [197]:
for ind in app_industry_data_cleaned:
    print(len(app_industry_data_cleaned[ind]))

69
178
545
83
94
505


In [198]:
app_industry_data_cleaned

{'medical & pharma': ['connection ring for natural glove',
  'autoinjector-anaphylaxis',
  'autoinjector-retraction assembly',
  'clamp for feeding catheter',
  'dispension monitoring for diabetes pharmaceutical preparation',
  'dose button for insulin pen',
  'drug delivery systems',
  'handle for flexible endoscopy',
  'hospital diagnostics & imaging',
  'metered dose inhaler (mdi)-dose counter',
  'plunger for intra ocular lens (iol)',
  'power transmission for metered dose inhaler (mdi)',
  'rod for orthopedic sizing device',
  'hips implant',
  'syringe stopper',
  'prosthetic dog ear',
  'hip implant',
  'orthopedic implant',
  'peristaltic pump gears',
  'peristaltic tub',
  'housing in a medical',
  'cap for laboratory glass bottle',
  'peristaltic tubing',
  'dental part',
  'glucose monitor',
  'drug delivery',
  'metered dose counter',
  'cryogenic',
  'dental',
  'screw in injection device',
  'drug elution',
  'peristaltic pump',
  'implanted medical devices',
  'medical d

In [199]:
all_apps_industries = []
for ind in app_industry_data_cleaned:
    if ind=='not mapped':
        all_apps_industries.append((app, None))
    else:
        for app in app_industry_data_cleaned[ind]:
            all_apps_industries.append((app, ind))

all_apps_industries

[('connection ring for natural glove', 'medical & pharma'),
 ('autoinjector-anaphylaxis', 'medical & pharma'),
 ('autoinjector-retraction assembly', 'medical & pharma'),
 ('clamp for feeding catheter', 'medical & pharma'),
 ('dispension monitoring for diabetes pharmaceutical preparation',
  'medical & pharma'),
 ('dose button for insulin pen', 'medical & pharma'),
 ('drug delivery systems', 'medical & pharma'),
 ('handle for flexible endoscopy', 'medical & pharma'),
 ('hospital diagnostics & imaging', 'medical & pharma'),
 ('metered dose inhaler (mdi)-dose counter', 'medical & pharma'),
 ('plunger for intra ocular lens (iol)', 'medical & pharma'),
 ('power transmission for metered dose inhaler (mdi)', 'medical & pharma'),
 ('rod for orthopedic sizing device', 'medical & pharma'),
 ('hips implant', 'medical & pharma'),
 ('syringe stopper', 'medical & pharma'),
 ('prosthetic dog ear', 'medical & pharma'),
 ('hip implant', 'medical & pharma'),
 ('orthopedic implant', 'medical & pharma'),


### Region

In [200]:
am_region_df = am_region_df.fillna('')
am_region_data = am_region_df.to_dict('list')
am_region_data

{'Country': ['United States',
  'Canada',
  'Mexico',
  'Brazil',
  'Argentina',
  'Colombia',
  'Chile',
  'Peru',
  'Venezuela',
  'Ecuador',
  'Bolivia',
  'Paraguay',
  'Uruguay',
  'Guyana',
  'Suriname',
  'Belize',
  'Costa Rica',
  'El Salvador',
  'Guatemala',
  'Honduras',
  'Nicaragua',
  'Panama',
  'Cuba',
  'Haiti',
  'Dominican Republic',
  'Jamaica',
  'Trinidad and Tobago',
  'Bahamas',
  'Barbados',
  'Saint Lucia',
  'USA',
  'US',
  'CA',
  'MX',
  'BR',
  'AR'],
 'State': ['California',
  'Texas',
  'New York',
  'Florida',
  'Illinois',
  'Pennsylvania',
  'Ohio',
  'Georgia, U.S.',
  'North Carolina',
  'Michigan',
  'Ontario',
  'Quebec',
  'British Columbia',
  'Alberta',
  'Nova Scotia',
  'Jalisco',
  'Nuevo León',
  'São Paulo',
  'Rio de Janeiro',
  'Minas Gerais',
  'Buenos Aires',
  'Córdoba',
  'Santa Fe',
  'Antioquia',
  'Valle del Cauca',
  'Lima',
  'Cusco',
  'Santiago Metropolitan',
  'Valparaíso',
  'Mendoza',
  'NY',
  'TX',
  'BC',
  '',
  '',
 

In [201]:
am_region_data_cleaned = {}
for i in am_region_data:
    am_region_data_cleaned[i.lower()] = [re.sub("(\s+)", " ", x.strip().lower()) for x in am_region_data[i] if x]

am_region_data_cleaned

{'country': ['united states',
  'canada',
  'mexico',
  'brazil',
  'argentina',
  'colombia',
  'chile',
  'peru',
  'venezuela',
  'ecuador',
  'bolivia',
  'paraguay',
  'uruguay',
  'guyana',
  'suriname',
  'belize',
  'costa rica',
  'el salvador',
  'guatemala',
  'honduras',
  'nicaragua',
  'panama',
  'cuba',
  'haiti',
  'dominican republic',
  'jamaica',
  'trinidad and tobago',
  'bahamas',
  'barbados',
  'saint lucia',
  'usa',
  'us',
  'ca',
  'mx',
  'br',
  'ar'],
 'state': ['california',
  'texas',
  'new york',
  'florida',
  'illinois',
  'pennsylvania',
  'ohio',
  'georgia, u.s.',
  'north carolina',
  'michigan',
  'ontario',
  'quebec',
  'british columbia',
  'alberta',
  'nova scotia',
  'jalisco',
  'nuevo león',
  'são paulo',
  'rio de janeiro',
  'minas gerais',
  'buenos aires',
  'córdoba',
  'santa fe',
  'antioquia',
  'valle del cauca',
  'lima',
  'cusco',
  'santiago metropolitan',
  'valparaíso',
  'mendoza',
  'ny',
  'tx',
  'bc'],
 'city': ['n

In [202]:
emea_region_df = emea_region_df.fillna('')
emea_region_data = emea_region_df.to_dict('list')
emea_region_data

{'Country': ['France',
  'Germany',
  'Italy',
  'Spain',
  'United Kingdom',
  'Netherlands',
  'Belgium',
  'Switzerland',
  'Austria',
  'Sweden',
  'Norway',
  'Denmark',
  'Finland',
  'Greece',
  'Portugal',
  'Turkey',
  'Israel',
  'Saudi Arabia',
  'United Arab Emirates',
  'Qatar',
  'Kuwait',
  'Oman',
  'Bahrain',
  'Jordan',
  'Lebanon',
  'Egypt',
  'South Africa',
  'Nigeria',
  'Kenya',
  'Morocco',
  'UK',
  'UAE',
  'FR',
  'NL',
  'SA',
  'ZA',
  'DE',
  'GB',
  'Albania',
  'Algeria',
  'Andorra',
  'Angola',
  'Belarus',
  'Benin',
  'Bosnia and Herzegovina',
  'Botswana',
  'Bulgaria',
  'Burkina Faso',
  'Burundi',
  'Cameroon',
  'Cape Verde',
  'Central African Republic',
  'Chad',
  'Comoros',
  'Croatia',
  'Cyprus',
  'Czech Republic',
  'Democratic Republic of the Congo',
  'Djibouti',
  'Equatorial Guinea',
  'Eritrea',
  'Estonia',
  'Ethiopia',
  'Faroe Islands',
  'Gabon',
  'Gambia',
  'Ghana',
  'Gibraltar',
  'Guernsey',
  'Guinea',
  'Guinea-Bissau'

In [203]:
emea_region_data_cleaned = {}
for i in emea_region_data:
    emea_region_data_cleaned[i.lower()] = [re.sub("(\s+)", " ", x.strip().lower()) for x in emea_region_data[i] if x]

emea_region_data_cleaned

{'country': ['france',
  'germany',
  'italy',
  'spain',
  'united kingdom',
  'netherlands',
  'belgium',
  'switzerland',
  'austria',
  'sweden',
  'norway',
  'denmark',
  'finland',
  'greece',
  'portugal',
  'turkey',
  'israel',
  'saudi arabia',
  'united arab emirates',
  'qatar',
  'kuwait',
  'oman',
  'bahrain',
  'jordan',
  'lebanon',
  'egypt',
  'south africa',
  'nigeria',
  'kenya',
  'morocco',
  'uk',
  'uae',
  'fr',
  'nl',
  'sa',
  'za',
  'de',
  'gb',
  'albania',
  'algeria',
  'andorra',
  'angola',
  'belarus',
  'benin',
  'bosnia and herzegovina',
  'botswana',
  'bulgaria',
  'burkina faso',
  'burundi',
  'cameroon',
  'cape verde',
  'central african republic',
  'chad',
  'comoros',
  'croatia',
  'cyprus',
  'czech republic',
  'democratic republic of the congo',
  'djibouti',
  'equatorial guinea',
  'eritrea',
  'estonia',
  'ethiopia',
  'faroe islands',
  'gabon',
  'gambia',
  'ghana',
  'gibraltar',
  'guernsey',
  'guinea',
  'guinea-bissau'

In [204]:
ap_region_df = ap_region_df.fillna('')
ap_region_data = ap_region_df.to_dict('list')
ap_region_data

{'Country': ['Australia',
  'China',
  'India',
  'Japan',
  'South Korea',
  'Indonesia',
  'Malaysia',
  'New Zealand',
  'Philippines',
  'Singapore',
  'Thailand',
  'Vietnam',
  'Bangladesh',
  'Pakistan',
  'Sri Lanka',
  'Nepal',
  'Myanmar',
  'Cambodia',
  'Laos',
  'Brunei',
  'Mongolia',
  'Fiji',
  'Papua New Guinea',
  'Maldives',
  'Bhutan',
  'Timor-Leste',
  'Solomon Islands',
  'Vanuatu',
  'Samoa',
  'Tonga',
  'AUS',
  'CHN',
  'IND',
  'KOR',
  'JPN',
  'SGP',
  'NZ',
  'indian'],
 'State': ['New South Wales',
  'Victoria',
  'Queensland',
  'Western Australia',
  'Tasmania',
  'Maharashtra',
  'Karnataka',
  'Tamil Nadu',
  'Gujarat',
  'Beijing',
  'Shanghai',
  'Guangdong',
  'Sichuan',
  'Hokkaido',
  'Kyoto',
  'Osaka',
  'Seoul',
  'Busan',
  'Jakarta',
  'Bali',
  'Selangor',
  'Kuala Lumpur',
  'Auckland',
  'Wellington',
  'Manila',
  'Bangkok',
  'Ho Chi Minh City',
  'Hanoi',
  'Colombo',
  'Punjab',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 'City

In [205]:
ap_region_data_cleaned = {}
for i in ap_region_data:
    ap_region_data_cleaned[i.lower()] = [re.sub("(\s+)", " ", x.strip().lower()) for x in ap_region_data[i] if x]

ap_region_data_cleaned

{'country': ['australia',
  'china',
  'india',
  'japan',
  'south korea',
  'indonesia',
  'malaysia',
  'new zealand',
  'philippines',
  'singapore',
  'thailand',
  'vietnam',
  'bangladesh',
  'pakistan',
  'sri lanka',
  'nepal',
  'myanmar',
  'cambodia',
  'laos',
  'brunei',
  'mongolia',
  'fiji',
  'papua new guinea',
  'maldives',
  'bhutan',
  'timor-leste',
  'solomon islands',
  'vanuatu',
  'samoa',
  'tonga',
  'aus',
  'chn',
  'ind',
  'kor',
  'jpn',
  'sgp',
  'nz',
  'indian'],
 'state': ['new south wales',
  'victoria',
  'queensland',
  'western australia',
  'tasmania',
  'maharashtra',
  'karnataka',
  'tamil nadu',
  'gujarat',
  'beijing',
  'shanghai',
  'guangdong',
  'sichuan',
  'hokkaido',
  'kyoto',
  'osaka',
  'seoul',
  'busan',
  'jakarta',
  'bali',
  'selangor',
  'kuala lumpur',
  'auckland',
  'wellington',
  'manila',
  'bangkok',
  'ho chi minh city',
  'hanoi',
  'colombo',
  'punjab'],
 'city': ['sydney',
  'melbourne',
  'brisbane',
  'pe

In [206]:
region_abbr_list = []
imp_abbr = ['usa', 'us', 'ny', 'uk', 'uae', 'eu', 'aus', 'chn', 'ind', 'kor', 'jpn']
all_regions_data = {'continent': [], 'country': [], 'state': [], 'city': []}

for k in am_region_data_cleaned:
    for v in am_region_data_cleaned[k]:
        if len(v)<4 and v not in imp_abbr:
            region_abbr_list.append(v)
        else:
            all_regions_data[k].append((v, 'americas'))

for k in emea_region_data_cleaned:
    for v in emea_region_data_cleaned[k]:
        if len(v)<4 and v not in imp_abbr:
            region_abbr_list.append(v)
        else:
            all_regions_data[k].append((v, 'europe middle east africa'))

for k in ap_region_data_cleaned:
    for v in ap_region_data_cleaned[k]:
        if len(v)<4 and v not in imp_abbr:
            region_abbr_list.append(v)
        else:
            all_regions_data[k].append((v, 'asia pacific'))
            
region_abbr_list

['ca',
 'mx',
 'br',
 'ar',
 'tx',
 'bc',
 'fr',
 'nl',
 'sa',
 'za',
 'de',
 'gb',
 'dxb',
 'sgp',
 'nz']

In [207]:
all_regions_data

{'continent': [('americas', 'americas'),
  ('north america', 'americas'),
  ('south america', 'americas'),
  ('eu', 'europe middle east africa'),
  ('europe', 'europe middle east africa'),
  ('europe middle east africa', 'europe middle east africa'),
  ('emea', 'europe middle east africa'),
  ('asia', 'asia pacific'),
  ('apac', 'asia pacific'),
  ('asia pacific', 'asia pacific')],
 'country': [('united states', 'americas'),
  ('canada', 'americas'),
  ('mexico', 'americas'),
  ('brazil', 'americas'),
  ('argentina', 'americas'),
  ('colombia', 'americas'),
  ('chile', 'americas'),
  ('peru', 'americas'),
  ('venezuela', 'americas'),
  ('ecuador', 'americas'),
  ('bolivia', 'americas'),
  ('paraguay', 'americas'),
  ('uruguay', 'americas'),
  ('guyana', 'americas'),
  ('suriname', 'americas'),
  ('belize', 'americas'),
  ('costa rica', 'americas'),
  ('el salvador', 'americas'),
  ('guatemala', 'americas'),
  ('honduras', 'americas'),
  ('nicaragua', 'americas'),
  ('panama', 'americas

### Cleaning Grades and CGrades

In [208]:
all_cgrades = [x.replace('®', '').replace('™', '').strip() for x in all_cgrades]
all_gradenames = [x.replace('®', '').replace('™', '').strip() for x in all_gradenames]

In [209]:
len(all_cgrades), len(all_gradenames)

(12257, 18148)

In [210]:
remove_suffixes = ['(ok1)', '(extra)', '(f1 apply gy/bk only)', '(r&d sample)', '(short version)', '(complete data)', '(before new yc)', '(extrusion)', '(condensed data)', '(r&d trial sample)', '(spcl)', '(eu)', '(us)', '(inte)', '(inte', '(condensed)', '(simplified)', '(dev)', '(old version)', '(old)', '(developmental)', ' - asia', ' - europe', ' - americas']

temp = []
for cg in all_cgrades:   
    for suffix in remove_suffixes:
        cg = re.sub(r'{}$'.format(re.escape(suffix.lower())), '', cg.lower())
    temp.append(re.sub("(\s+)", " ", cg.replace('®', '').replace('™', '').strip())) 

all_cgrades = [value.strip() for value in temp]

temp = []
for g in all_gradenames:   
    for suffix in remove_suffixes:
        g = re.sub(r'{}$'.format(re.escape(suffix.lower())), '', g.lower())
    temp.append(re.sub("(\s+)", " ", g.replace('®', '').replace('™', '').strip())) 

all_gradenames = [value.strip() for value in temp]
all_gradenames

['kep f20-03',
 'cnl b3 j gf15 nc 1102/r',
 'frianyl b3 gf25 x v0',
 'celcon uv140lg',
 'ze 650',
 'cs pp-gf20-02 black',
 'rynite 530hte nc010',
 'lftr pp-gf40-0403 black',
 'stp 251-80w232',
 'santoprene 281-55med',
 'hf s 9364uv',
 'pultrusion pa66-gf50-02 af3001 natural',
 'celcon m270 eco-b',
 'hostaform ec270tx',
 'hostaform slidex c0304 xap2',
 'lft pp-gf60-0453 p10/10',
 'santoprene 8221-85m300',
 'zenite 5115l',
 'pultrusion pp-gf40-20 ad3004 black',
 'hyt htr8332 bk320',
 'stp 691-73w175',
 'htr sc976 nc010',
 'celanyl b3 n hh bk 9004/z/uv',
 'celstran pp-gf30-0403 p7',
 'cnl a3 gf35 bk 9005/u',
 'hf c 2521 xap',
 'hf s 9364 lpb',
 'santoprene 101-87 eco-r',
 'hyt 40cb',
 'celstran cfr-tp pet gf60-10',
 'zytel lc6210 bk010',
 'kepital f10-52h lof',
 'crastin fr684nh1 nc010',
 'xgc',
 'gur 4130',
 'kepital f20-03 colored',
 'cs lft tpu-gf50-01 ad3002 black',
 'fo 6165a4',
 'hf c 27021 xap2 ls',
 'lftr pp-gf40-0455 eco-b352soul 4pk blk',
 'celstran lft pp-gf40-03-black',
 'lft 

In [211]:
all_brands_without_syns

['ecomid',
 'zenite',
 'thermx',
 'fortron',
 'santoprene',
 'frianyl',
 'celstran',
 'celanex',
 'hostaform',
 'celcon',
 'kepital',
 'coolpoly',
 'rynite',
 'vectra',
 'crastin',
 'hytrel',
 'celanyl',
 'zytel',
 'gur',
 'forflex',
 'kepamid',
 'abistir',
 'impet',
 'blueridge',
 'blendfor',
 'omnilon',
 'nylfor',
 'ateva',
 'selar',
 'factor',
 'tecnoprene',
 'neolast',
 'omnicarb',
 'sofpur',
 'micromax',
 'tarnoform',
 'nilamid',
 'celapex',
 'omnitech',
 'pibifor',
 'vamac',
 'kepex',
 'pipelon',
 'vitaldose',
 'forprene',
 'cecopoly',
 'clarifoil',
 'compel',
 'talcoprene',
 'litepol',
 'sikamid',
 'laprene',
 'elvamide',
 'keploy',
 'vandar',
 'stirofor',
 'amcel',
 'polifor',
 'omnipro',
 'sofprene',
 'maximid',
 'minlon',
 'pibiter']

In [212]:
grades_without_brands = []
grade_prefix = {}
for i in all_gradenames:
    brand_found = False
    for j in all_brands:
        if not brand_found and j == i.split()[0]:
            brand_found = True
            break
    if not brand_found:
        grades_without_brands.append(i)
        if i.split()[0] in grade_prefix:
            grade_prefix[i.split()[0]] += 1 
        else:
            grade_prefix[i.split()[0]] = 1 


In [213]:
{k: v for k, v in sorted(grade_prefix.items(), key=lambda item: item[1], reverse=True)}

{'htr': 283,
 'im': 79,
 'at': 26,
 'bexloy': 25,
 'etpv': 12,
 'sofpurle': 9,
 'ghr': 5,
 'active': 5,
 'nylind': 5,
 'forgrin': 5,
 'celanese': 5,
 'pet': 4,
 'sofpurl': 4,
 'holo': 4,
 'sofpurs': 4,
 'fxj9507': 3,
 'eco': 2,
 'carboprene': 2,
 'reblend': 2,
 'zytelrs': 2,
 'shine-e': 2,
 'zytelhtn51g35eft': 2,
 'retelan': 2,
 'xgc': 1,
 'dym': 1,
 'hhr': 1,
 'ice': 1,
 'eco-b': 1,
 'xap': 1,
 'slidex': 1,
 'lof2': 1,
 'icf': 1,
 'hrlm': 1,
 'scxxx': 1,
 'hsl': 1,
 'sea': 1,
 'hte': 1,
 'wrf': 1,
 'pcxxx': 1,
 'pls/xt': 1,
 'frhr': 1,
 'fit': 1,
 'lof': 1,
 'xfr': 1,
 'hslr': 1,
 'hrt': 1,
 'hfs': 1,
 'eco-r': 1,
 'htn': 1,
 'xap2': 1,
 'nkx-101': 1,
 'cabofor': 1,
 'jkx-1292': 1,
 'dev.': 1,
 'holrsn01': 1,
 'nkx-1010a': 1,
 'vizilon': 1,
 'kematal': 1,
 'jkx-1295': 1,
 'tecnoprenefk5u': 1,
 'b3': 1,
 'terra': 1,
 'ch0281-1': 1,
 'jkx-1291': 1,
 's224-55a': 1,
 'sd001': 1,
 'pryltex': 1,
 'zytelhtnfe270083': 1,
 'volo': 1,
 'k3045.x01': 1,
 'lumid': 1,
 'zytelhtn51g45eft': 1,
 'jkx-

In [214]:
grade_prefixes = all_brands + ['lanyl', 'omax', 'fprene', 'lanex', 'htr', 'lifor', 'tin', 'lstran', 'ynite', 'nicarb', 'omid', 'duct', 'lcon', 'teva', 'pet', 'nlon', 'fpur', 'nipro']
grade_prefixes = list(set(grade_prefixes))

In [215]:
len(grades_without_brands)

543

In [216]:
random.shuffle(all_gradenames)
all_gradenames = [x for x in all_gradenames if x not in ['pet']]

grade_brand_prefix_count = {}
filtered_gradenames = []
without_prefix = []
without_prefix2 = []
for i in all_gradenames:
    brand_found = False
    for j in grade_prefixes:
        if not brand_found and j == i.split()[0]:
            brand_found = True
            if j not in grade_brand_prefix_count:
                try:
                    if i.split(j+' ')[1] not in without_prefix:
                        without_prefix.append(i.split(j+' ')[1])
                        grade_brand_prefix_count[j] = 1
                        filtered_gradenames.append(i)
                except:
                    if i not in without_prefix:
                        without_prefix.append(i)
                        grade_brand_prefix_count[j] = 1
                        filtered_gradenames.append(i)

                
            elif j in grade_brand_prefix_count and grade_brand_prefix_count[j] < 70:
                try:
                    if i.split(j+' ')[1] not in without_prefix:
                        without_prefix.append(i.split(j+' ')[1])
                        grade_brand_prefix_count[j] += 1
                        filtered_gradenames.append(i)
                except:
                    if i not in without_prefix:
                        without_prefix.append(i)
                        grade_brand_prefix_count[j] += 1
                        filtered_gradenames.append(i)
            else:
                try:
                    if i.split(j+' ')[1] not in without_prefix:
                        without_prefix2.append(i.split(j+' ')[1])
                except:
                    if i not in without_prefix:
                        without_prefix2.append(i)
                
            break
            
    if not brand_found:
        without_prefix.append(i)
        filtered_gradenames.append(i)

len(filtered_gradenames)

4434

In [217]:
without_prefix2

['73g15hsl bk363',
 'efe1091 bk010',
 'fe3667 nc010',
 'htn92g45dh2 bk083',
 '70g35hslr bk416lm_bu',
 'htnfr55g50nhlw gy061',
 'htn53g60lrhf nc010',
 'rs lc3030 nc010',
 'htnwrf51g30 nc010',
 '158 nc010 - bu',
 'fn727 nc010',
 'htnwrf51mp20 nc010',
 'mt409ahs nc010',
 '70g25ef nc010',
 '75cg30hsl bk409',
 '72g33l bk031',
 'e50 nc010',
 '153hsl bkb038',
 'lc7601 nc010',
 'rs lc3090 nc010 - bu',
 'htnfe150062 bk544',
 'fr70m30v0 bk010',
 'fn718 bk230',
 '70g33hs1l bk031x',
 'htnfe350174 nc010',
 '101f nc010-bu',
 'sc315 nc010 - bu',
 'fe5313 bk032',
 'fe15004 bk032d',
 'st800hsl bk152',
 'fr15 nc010',
 '73g30hslc bk416',
 '74g35arx eco-r 311 blk1',
 '70g30psr nc010',
 'nvh70g35hsla bk152',
 '70g35hsl bk020a',
 '450hslx nc010',
 '101f bk009',
 '80g33l nc010',
 'lc7202 bk',
 '80g30arx eco-r 311 blk1',
 'pls95g40dh1t bk261',
 'fe170032 bk155',
 '159 nc010',
 'bm7300fnh bk317',
 'rs htn59g50ldp bk108',
 '70g50hslr bk509',
 '70g20hsl bk039b',
 'fe5510 bk512j',
 'st801aw bk195',
 'fg77g33l nc0

In [218]:
filtered_gradenames = filtered_gradenames + random.sample(without_prefix2, 500)
filtered_gradenames = [x.strip('+/-') for x in filtered_gradenames]
filtered_gradenames[-20:]

['5450h',
 '9c602',
 'z ce198-60a',
 'fr50 nc010',
 '540200a45 neutro',
 '830308914 bianco',
 'xs3 gf60 nc 1102/e',
 'mt8f01',
 'jkx-1049',
 'b3 h gf35 bk 9005/2',
 '219n10155r nero',
 'b3 hh gfb1020 bk 9005/2',
 'xs3 gf60 wt 9016/c ef',
 '8ks901a90/h nero',
 'a3 v2 a bk 9005/g',
 '350phs3 bk010',
 '185bs8060 beige micro',
 'lft roving fv vergine ms500',
 'xt4 gf30 v0i bk 9005/d',
 'b3 w gf33 nc 1102/fd/1']

In [219]:
len(filtered_gradenames)

4934

In [220]:
sorted(filtered_gradenames, key=len)[:100]

['im',
 'pb',
 'hte',
 'xgc',
 'lof',
 'hfs',
 'ice',
 'xap',
 'fit',
 'dym',
 'icf',
 'htn',
 'wrf',
 'hsl',
 'xfr',
 'sea',
 'hhr',
 'hrt',
 'frhr',
 'hslr',
 'lof2',
 'xap2',
 'hrlm',
 '0030',
 '5731',
 '3216',
 '5028',
 '4321',
 '2004',
 '5727',
 '7881',
 '2071',
 '1106',
 'lft ',
 '2031',
 '5418',
 '3316',
 '4047',
 '6474',
 '7105',
 '2011',
 '7105',
 '7082',
 '2041',
 '4195',
 's150',
 '8173',
 '1959',
 'pcxxx',
 'eco-r',
 'eco r',
 'sd001',
 'eco-b',
 'eco b',
 'scxxx',
 'impet',
 '5450h',
 'pe316',
 'thr35',
 '1401a',
 'j-600',
 'me614',
 'qp603',
 '3300a',
 'hfb21',
 '1748r',
 'qs874',
 '5082r',
 'hfb31',
 'bq331',
 'qm44h',
 'ht602',
 'pe825',
 '5504n',
 '5063d',
 'bq131',
 'e8 uv',
 'qq620',
 '0022z',
 'll602',
 'bq922',
 '9475r',
 '4597r',
 '0001b',
 '0f79a',
 '2008a',
 'pe510',
 '4879d',
 'lf79a',
 'cb200',
 '0f30e',
 'j600w',
 '7499r',
 'ht603',
 '8152b',
 '1642z',
 'qp601',
 'lf79a',
 'bq226',
 '5450h']

## Templates and Data Generation

In [221]:
st = time.time() 

feature_count = 0
processing_count = 0
del_count = 0

brand_count = 0
polymer_count = 0
app_count = 0

nsf_count = 0
water_count = 0
railway_count = 0
auto_cert_count = 0

prop_count = 0
prop_range_count = 0
prop_abb_count = 0

ul_prop_count = 0
ul_sub_prop_count = 0
filler_count = 0

region_count = 0
ignore_terms_count = 0

# queries_thershold = 1000
auto_th = 2500
rail_th = 1500
water_th = 1000
nsf_th = 1000


prop_th = 2000
prop_rm_th = 1000
prop_abb_th = 1000

fill_th = 2500
ul_prop_th = 1500
ul_sub_prop_th = 1400


process_th = 3500
feat_th = 6500
del_th = 2000

brand_th = 6000
poly_th = 6500
app_th = 3000
region_th = 3000
ignore_terms_th = 2000

auto_th_reached = False
rail_th_reached = False
water_th_reached = False
nsf_th_reached = False

prop_th_reached = False
prop_rm_th_reached = False
prop_abb_th_reached = False

fill_th_reached = False
ul_prop_th_reached = False
ul_sub_prop_th_reached = False

process_th_reached = False
feat_th_reached = False
del_th_reached = False

brand_th_reached = False
poly_th_reached = False
app_th_reached = False

region_th_reached = False
ignore_terms_th_reached = False

status = [  
    'auto_th_not_reached',  
    'rail_th_not_reached',  
    'water_th_not_reached',  
    'nsf_th_not_reached',  
    'prop_th_not_reached',  
    'prop_rm_th_not_reached',  
    'prop_abb_th_not_reached',  
    'fill_th_not_reached',  
    'ul_prop_th_not_reached',  
    'ul_sub_prop_th_not_reached',  
    'process_th_not_reached',  
    'feat_th_not_reached',  
    'del_th_not_reached',  
    'brand_th_not_reached',  
    'poly_th_not_reached',  
    'app_th_not_reached',
    'region_th_not_reached',
    'ignore_terms_th_not_reached',   
]

generated_data = []
# while any(x < queries_thershold for x in [feature_count, processing_count, del_count, brand_count, polymer_count]):    
while auto_cert_count < auto_th or railway_count < rail_th or nsf_count < nsf_th or water_count < water_th \
    or prop_count < prop_th or prop_range_count < prop_rm_th or prop_abb_count < prop_abb_th \
    or filler_count < fill_th or  ul_prop_count < ul_prop_th or  ul_sub_prop_count < ul_sub_prop_th \
    or processing_count < process_th or feature_count < feat_th or del_count < del_th \
    or brand_count < brand_th or polymer_count < poly_th or app_count < app_th or region_count < region_th:        
  
    if not auto_th_reached and auto_cert_count >= (auto_th - 1):  
        status.remove('auto_th_not_reached')  
        auto_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not rail_th_reached and railway_count >= (rail_th - 1):  
        status.remove('rail_th_not_reached')  
        rail_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not water_th_reached and water_count >= (water_th - 1):  
        status.remove('water_th_not_reached')  
        water_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not nsf_th_reached and nsf_count >= (nsf_th - 1):  
        status.remove('nsf_th_not_reached')  
        nsf_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not prop_th_reached and prop_count >= (prop_th - 1):  
        status.remove('prop_th_not_reached')  
        prop_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not prop_rm_th_reached and prop_range_count >= (prop_rm_th - 1):  
        status.remove('prop_rm_th_not_reached')  
        prop_rm_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not prop_abb_th_reached and prop_abb_count >= (prop_abb_th - 1):  
        status.remove('prop_abb_th_not_reached')  
        prop_abb_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not fill_th_reached and filler_count >= (fill_th - 1):  
        status.remove('fill_th_not_reached')  
        fill_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not ul_prop_th_reached and ul_prop_count >= (ul_prop_th - 1):  
        status.remove('ul_prop_th_not_reached')  
        ul_prop_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not ul_sub_prop_th_reached and ul_sub_prop_count >= (ul_sub_prop_th - 1):  
        status.remove('ul_sub_prop_th_not_reached')  
        ul_sub_prop_th_reached = True  
        print("time taken: ", int(time.time() - st))  
        print("\n", status, "\n")  
      
    if not process_th_reached and processing_count >= (process_th - 1):
        status.remove('process_th_not_reached')
        process_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")
    
    if not feat_th_reached and feature_count >= (feat_th - 1):
        status.remove('feat_th_not_reached')
        feat_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")
    
    if not del_th_reached and del_count >= (del_th - 1):
        status.remove('del_th_not_reached')
        del_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")
    
    if not brand_th_reached and brand_count >= (brand_th - 1):
        status.remove('brand_th_not_reached')
        brand_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")
    
    if not poly_th_reached and polymer_count >= (poly_th - 1):
        status.remove('poly_th_not_reached')
        poly_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")
    
    if not app_th_reached and app_count >= (app_th - 1):
        status.remove('app_th_not_reached')
        app_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n") 

    if not region_th_reached and region_count >= (region_th - 1):
        status.remove('region_th_not_reached')
        region_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")

    if not ignore_terms_th_reached and ignore_terms_count >= (ignore_terms_th - 1):
        status.remove('ignore_terms_th_not_reached')
        ignore_terms_th_reached = True
        print("time taken: ", int(time.time() - st))
        print("\n", status, "\n")
    
    processing = random.choice(processing_types).lower()
    feature = random.choice(product_functions).lower()
    delivery_form = random.choice(delivery_forms).lower()   
    brand = random.choice(all_brands_without_syns).lower()       
    polymer = random.choice(all_polymers).lower()       
    grade = random.choice(filtered_gradenames).lower()
    is_grade_without_space = random.choice([False, False, False, False, False, False, True])
    if is_grade_without_space:
        grade = grade.replace(' ', '')
    nsf = random.choice(nsf_certifications).lower()
    water = random.choice(water_certifications_combinations)
    water_name = water[0].lower()
    railway = random.choice(railway_certifications_combinations)

    app_ind = random.choice(all_apps_industries)
    application = app_ind[0]
    industry = app_ind[1]

    continent, continent_region = random.choice(all_regions_data['continent'])
    country, country_region = random.choice(all_regions_data['country'])
    state, state_region = random.choice(all_regions_data['state'])
    city, city_region = random.choice(all_regions_data['city'])
    competitor_grade = random.choice(all_cgrades)
    # for suffix in remove_suffixes:
    #     competitor_grade = re.sub(r'{}$'.format(re.escape(suffix)), '', competitor_grade) 
        
    competitor_grade = re.sub(r'^\((.*)\)$', r'\1', competitor_grade)
    competitor_grade = competitor_grade.strip()
    is_comp_grade_without_space = random.choice([False, False, False, False, False, False, True])
    if is_comp_grade_without_space:
        competitor_grade = competitor_grade.replace(' ', '')

    ignore_term = random.choice(ignore_terms)
    ignore_term2 = random.choice(ignore_terms)
    while ignore_term2 == ignore_term:
        ignore_term2 = random.choice(ignore_terms)
    
    k = random.choice([0, 1, 2])
    if k==0:
        railway_name = railway[0]
    elif k==1:
        railway_name = railway[0].split(' (')[0]
    elif k==2:
        railway_name = "".join(railway[0].split(' (')[0].split('-'))
        
    railway_name = ' '.join(railway_name.split()).lower()
    
    
    processing2 = random.choice(processing_types).lower()        
    while processing2 == processing:
        processing2 = random.choice(processing_types).lower()
        
    feature2 = random.choice(product_functions).lower()
    while feature2 == feature:
        feature2 = random.choice(product_functions).lower()
         
    delivery_form2 = random.choice(delivery_forms).lower()
    while delivery_form2 == delivery_form:
        delivery_form2 = random.choice(delivery_forms).lower()

    brand2 = random.choice(all_brands_without_syns).lower()      
    while brand2 == brand:
        brand2 = random.choice(all_brands_without_syns).lower()
    
    polymer2 = random.choice(all_polymers).lower()
    while polymer2 == polymer:
        polymer2 = random.choice(all_polymers).lower()

    is_processing_syn = random.choice([False, True, True, True])
    if is_processing_syn and processing in processing_syns:
        processing = random.choice(processing_syns[processing])
        
    is_processing_syn = random.choice([False, True, True, True])
    if is_processing_syn and processing2 in processing_syns:
        processing2 = random.choice(processing_syns[processing2])

    is_feature_syn = random.choice([False, True, True, True])
    if is_feature_syn and feature in feature_syns:
        feature = random.choice(feature_syns[feature])
        
    is_feature_syn = random.choice([False, True, True, True])
    if is_feature_syn and feature2 in feature_syns:
        feature2 = random.choice(feature_syns[feature2])

    is_delivery_syn = random.choice([False, True, True, True])
    if is_delivery_syn and delivery_form in delivery_form_syns:
        delivery_form = random.choice(delivery_form_syns[delivery_form])
        
    is_delivery_syn = random.choice([False, True, True, True])
    if is_delivery_syn and delivery_form2 in delivery_form_syns:
        delivery_form2 = random.choice(delivery_form_syns[delivery_form2])

    is_polymer_syn = random.choice([False, True, True, True])
    if is_polymer_syn and polymer in polymer_syns:
        polymer = random.choice(polymer_syns[polymer])
        
    is_polymer_syn = random.choice([False, True, True, True])
    if is_polymer_syn and polymer2 in polymer_syns:
        polymer2 = random.choice(polymer_syns[polymer2])

    is_brand_syn = random.choice([False, True, True, True])
    if is_brand_syn and brand in brand_syns:
        brand = random.choice(brand_syns[brand])

    is_brand_syn = random.choice([False, True, True, True])
    if len(brand)<4:
        while len(brand2)>3:
            brand2 = random.choice(all_brands_without_syns).lower()
        
            brand_temp = brand            
            if brand_temp in brand_syn_mapping:
                brand_temp = brand_syn_mapping[brand_temp]

            while brand2 == brand_temp:
                brand2 = random.choice(all_brands_without_syns).lower()

            brand2 = random.choice(brand_syns[brand2])

    elif is_brand_syn and brand2 in brand_syns:
        brand2 = random.choice(brand_syns[brand2])
        if len(brand2)<4:
            while len(brand2)<4:
                brand2 = random.choice(all_brands_without_syns).lower()
            
                brand_temp = brand            
                if brand_temp in brand_syn_mapping:
                    brand_temp = brand_syn_mapping[brand_temp]
    
                while brand2 == brand_temp:
                    brand2 = random.choice(all_brands_without_syns).lower() 

                brand2 = random.choice(brand_syns[brand2])

    auto_cert = random.choice(auto_certifications)
    oem_name = auto_cert[0].lower()
    if oem_name in auto_syns:
        oem_name = random.choice(auto_syns[oem_name])
    auto_cert_name = auto_cert[1].lower()
    
    auto_cert2 = random.choice(auto_certifications)
    oem_name2 = auto_cert2[0].lower()
    if oem_name2 in auto_syns:
        oem_name2 = random.choice(auto_syns[oem_name2])
    auto_cert_name2 = auto_cert2[1].lower()
    
    while auto_cert2==auto_cert:
        auto_cert2 = random.choice(auto_certifications)
        oem_name2 = auto_cert2[0].lower()
        if oem_name2 in auto_syns:
            oem_name2 = random.choice(auto_syns[oem_name2])
        auto_cert_name2 = auto_cert2[1].lower()

    auto_cert_name = random.choice([auto_cert_name, auto_cert_name, auto_cert_name, auto_cert_name.replace(' ', ''), re.sub('[^A-Za-z0-9]+', '', auto_cert_name), re.sub('[^A-Za-z0-9]+', '', auto_cert_name), re.sub("(\s+)", " ", re.sub('[^A-Za-z0-9]+', ' ', auto_cert_name)).strip(), re.sub("(\s+)", " ", re.sub('[^A-Za-z0-9]+', ' ', auto_cert_name)).strip()])    
    auto_cert_name2 = random.choice([auto_cert_name2, auto_cert_name2, auto_cert_name2, auto_cert_name2.replace(' ', ''), re.sub('[^A-Za-z0-9]+', '', auto_cert_name2), re.sub('[^A-Za-z0-9]+', '', auto_cert_name2), re.sub("(\s+)", " ", re.sub('[^A-Za-z0-9]+', ' ', auto_cert_name2)).strip(), re.sub("(\s+)", " ", re.sub('[^A-Za-z0-9]+', ' ', auto_cert_name2)).strip()])

    railway2 = random.choice(railway_certifications_combinations)
    k = random.choice([0, 1, 2])
    if k==0:
        railway_name2 = railway2[0]
    elif k==1:
        railway_name2 = railway2[0].split(' (')[0]
    elif k==2:
        railway_name2 = "".join(railway2[0].split(' (')[0].split('-'))
        
    railway_name2 = ' '.join(railway_name2.split()).lower()
    
    while railway2==railway:
        railway2 = random.choice(railway_certifications_combinations)
        k = random.choice([0, 1, 2])
        if k==0:
            railway_name2 = railway2[0]
        elif k==1:
            railway_name2 = railway2[0].split(' (')[0]
        elif k==2:
            railway_name2 = "".join(railway2[0].split(' (')[0].split('-'))
            
        railway_name2 = ' '.join(railway_name2.split()).lower()

    property_details , property = get_property()
    property2_details , property2 = get_property()
    
    while property2 == property:
        property2_details , property2 = get_property()

    property_abb_details , property_abb = get_property_abb()
    property_abb2_details , property_abb2 = get_property_abb()

    while property_abb2 == property_abb or (" " in property_abb2 and " " not in property_abb) or (" " in property_abb and " " not in property_abb2):
        property_abb2_details , property_abb2 = get_property_abb()

    common_modifier = random.choice([False, False, False, True])
    property_rm_details , property_rm = get_property_range_modifier(common_modifier)

    property_rm2_details , property_rm2 = get_property_range_modifier(common_modifier)
    while property_rm2 == property_rm:
        property_rm2_details , property_rm2 = get_property_range_modifier(common_modifier)

    filler_details, filler = get_random_filler()
    
    ul_property_details, ul_property = get_ul_property()
    ul_property2_details, ul_property2 = get_ul_property()
    while ul_property2 == ul_property:
        ul_property2_details, ul_property2 = get_ul_property()

    ul_sub_property_details, ul_sub_property = get_ul_sub_property()
    # this will add two min thichness
    # ul_sub_property2_details, ul_sub_property2 = get_ul_sub_property()
    # while ul_sub_property2 == ul_sub_property:
    #     ul_sub_property2_details, ul_sub_property2 = get_ul_sub_property()

    is_feature_spell_correct = False
    corrected_feature = ''
    if len(feature.replace('-', ' ').replace('/', ' ').split()[0])>4:
        is_feature_spell_correct = random.choice([False, False, False, False, False, False, False, False, False, False, False, False, False, True])
        if is_feature_spell_correct:
            corrected_feature = feature
            n = random.choice(range(len(feature)))
            if len(feature.split()[0])>4:
                feature = random.choice([feature[1:], feature[:n] + feature[n+1:]])
            else:
                feature = feature[:n] + feature[n+1:]

    templates = [ 
        (f"{railway_name.lower().replace('r', 'r ').replace('h', 'h ').replace('v0', 'v 0')}", "{railway_name}"),
        (f"{railway_name.lower().replace('r', 'r-').replace('h', 'h-').replace('v0', 'v-0')}", "{railway_name}"),
        (f"{railway_name.lower().replace('(en 45545-2)', '(en 455452)')}", "{railway_name}"),
        (f"{railway_name.lower().replace('(en 45545-2)', 'en 455452')}", "{railway_name}"),
        (f"{railway_name.lower().replace('(en 45545-2)', 'en')}", "{railway_name}"),
        (f"{railway_name.lower().replace(' ', '')}", "{railway_name}"),
        (f"{railway_name}", "{railway_name}"),
        
        (f"{railway_name} {railway_name2}", "{railway_name} {railway_name2}"),
        (f"{railway_name} and {railway_name2}", "{railway_name} and {railway_name2}"),
        (f"{railway_name} or {railway_name2}", "{railway_name} or {railway_name2}"),
        (f"{railway_name}, {railway_name2}", "{railway_name}, {railway_name2}"),
        
        (f"{auto_cert_name}", "{auto_cert_name}"),
        (f"{auto_cert_name} approval", "{auto_cert_name} approval"),
        (f"{auto_cert_name} auto spec", "{auto_cert_name} auto spec"),
        (f"{auto_cert_name} auto approval", "{auto_cert_name} auto approval"),
        (f"{auto_cert_name}", "{auto_cert_name}"),
        (f"{auto_cert_name} approval", "{auto_cert_name} approval"),
        (f"{auto_cert_name} auto spec", "{auto_cert_name} auto spec"),
        (f"{auto_cert_name} auto approval", "{auto_cert_name} auto approval"),
        
        (f"{oem_name}", "{oem_name}"),
        (f"{oem_name} approval", "{oem_name} approval"),
        (f"{oem_name} auto spec", "{oem_name} auto spec"),
        (f"{oem_name} auto approval", "{oem_name} auto approval"),

        (f"{oem_name} {auto_cert_name}", "{oem_name} {auto_cert_name}"),
        # (f"{oem_name}: {auto_cert_name}", "{oem_name}: {auto_cert_name}"),
        (f"{oem_name} - {auto_cert_name}", "{oem_name} - {auto_cert_name}"),
        (f"{oem_name} ({auto_cert_name})", "{oem_name} ({auto_cert_name})"),
        (f"{oem_name} auto specs {auto_cert_name}", "{oem_name} auto specs {auto_cert_name}"),
        # (f"{oem_name} auto specs approved {auto_cert_name}", "{oem_name} auto specs {auto_cert_name}"),
        
        (f"{auto_cert_name} {auto_cert_name2}", "{auto_cert_name} {auto_cert_name2}"),
        (f"{auto_cert_name} {auto_cert_name2}", "{auto_cert_name} {auto_cert_name2}"),
        (f"{auto_cert_name} {auto_cert_name2} approval", "{auto_cert_name} {auto_cert_name2} approval"),
        (f"{auto_cert_name} {auto_cert_name2} auto spec", "{auto_cert_name} {auto_cert_name2} auto spec"),
        # (f"{auto_cert_name} {auto_cert_name2} auto approval", "{auto_cert_name} {auto_cert_name2} auto approval"),
        
        (f"{auto_cert_name} and {auto_cert_name2}", "{auto_cert_name} and {auto_cert_name2}"),
        (f"{auto_cert_name} + {auto_cert_name2}", "{auto_cert_name} + {auto_cert_name2}"),
        (f"{auto_cert_name} or {auto_cert_name2}", "{auto_cert_name} or {auto_cert_name2}"),
        (f"{auto_cert_name}, {auto_cert_name2}", "{auto_cert_name}, {auto_cert_name2}"),
        
        (f"{oem_name} {auto_cert_name} {oem_name2} {auto_cert_name2}", "{oem_name} {auto_cert_name} {oem_name2} {auto_cert_name2}"),
        (f"{oem_name} {auto_cert_name} and {oem_name2} {auto_cert_name2}", "{oem_name} {auto_cert_name} and {oem_name2} {auto_cert_name2}"),
        (f"{oem_name} {auto_cert_name} or {oem_name2} {auto_cert_name2}", "{oem_name} {auto_cert_name} or {oem_name2} {auto_cert_name2}"),
        # (f"{oem_name}: {auto_cert_name} {oem_name2}: {auto_cert_name2}", "{oem_name}: {auto_cert_name} {oem_name2}: {auto_cert_name2}"),
        (f"{oem_name} - {auto_cert_name} {oem_name2} - {auto_cert_name2}", "{oem_name} - {auto_cert_name} {oem_name2} - {auto_cert_name2}"),
        (f"{oem_name} ({auto_cert_name}) {oem_name2} ({auto_cert_name2})", "{oem_name} ({auto_cert_name}) {oem_name2} ({auto_cert_name2})"),
        
        (f"{property_rm}", "{property_rm}"),
        (f"{property_rm} {property_rm2}", "{property_rm} {property_rm2}"),
        (f"{property_rm} and {property_rm2}", "{property_rm} and {property_rm2}"),
        (f"{property_rm} or {property_rm2}", "{property_rm} or {property_rm2}"),
        (f"{property_rm},  {property_rm2}", "{property_rm}, {property_rm2}"),
        
        (f"{property_rm} {property} {feature} {filler}", "{property_rm} {property} {feature} {filler}"),
        (f"{property_rm} {property} good {feature} {filler}", "{property_rm} {property} good {feature} {filler}"),
        (f"{property_rm} and {property} high {feature} with {filler}", "{property_rm} and {property} high {feature} with {filler}"),
        (f"{property_rm} and {property} excellent {feature} with {filler}", "{property_rm} and {property} excellent {feature} with {filler}"),
        
        (f"{filler} {feature} {property_rm} {property}", "{filler} {feature} {property_rm} {property}"),
        (f"{property_rm}, {property}, {feature}, {filler}", "{property_rm}, {property}, {feature}, {filler}"),
        (f"{property} good {feature} {filler} {property_rm}", "{property} good {feature} {filler} {property_rm}"),
        (f"{property} and high {feature} {property_rm} with {filler}", "{property} and high {feature} {property_rm} with {filler}"),
        (f"excellent {feature} {property_rm} and {property} with {filler}", "excellent {feature} {property_rm} and {property} with {filler}"),
        
        (f"{property_rm} {property}", "{property_rm} {property}"),
        (f"{property_rm}, {property}", "{property_rm}, {property}"),
        (f"{property_rm} and {property}", "{property_rm} and {property}"),
        (f"{property_rm} or {property}", "{property_rm} or {property}"),
        (f"{property_rm} + {property}", "{property_rm} + {property}"),
        
        (f"{property_rm} {feature}", "{property_rm} {feature}"),
        (f"{property_rm} with excellent {feature}", "{property_rm} with excellent {feature}"),
        (f"{property_rm} good {feature}", "{property_rm} good {feature}"),
        (f"{property_rm}, high {feature}", "{property_rm}, high {feature}"),
        (f"{property_rm} and {feature}", "{property_rm} and {feature}"),
        (f"{property_rm} + {feature}", "{property_rm} + {feature}"),
        
        (f"{property_rm} {filler}", "{property_rm} {filler}"),
        (f"{property_rm}, {filler}", "{property_rm}, {filler}"),
        (f"{property_rm} and {filler}", "{property_rm} and {filler}"),
        (f"{property_rm} + {filler}", "{property_rm} + {filler}"),
        
        (f"good {feature}", "good {feature}"),
        (f"{grade} with excellent {feature}", "{grade} with excellent {feature}"),
        (f"{brand} with good {feature} {feature2}", "{brand} with good {feature} {feature2}"),
        (f"{brand} with good {feature} and {feature2}", "{brand} with good {feature} and {feature2}"),
        
        (f"grade with {property_abb}", "grade with {property_abb}"),
        (f"material with {property_abb}", "material with {property_abb}"),
        (f"{property_abb}", "{property_abb}"),
        (f"{property_abb}", "{property_abb}"),
        (f"{property_abb}", "{property_abb}"),
        (f"{property_abb} {property_abb2}", "{property_abb} {property_abb2}"),
        (f"{property_abb}, {property_abb2}", "{property_abb}, {property_abb2}"),
        (f"{property_abb} and {property_abb2}", "{property_abb} and {property_abb2}"),
        (f"{property_abb} or {property_abb2}", "{property_abb} or {property_abb2}"),
        
        (f"{ul_property}", "{ul_property}"),
        (f"{ul_property}", "{ul_property}"),
        
        (f"{ul_property} {ul_property2}", "{ul_property} {ul_property2}"),
        (f"{ul_property} {ul_property2}", "{ul_property} {ul_property2}"),
        
        
        (f"{ul_sub_property}", "{ul_sub_property}"),       
        (f"{ul_sub_property}", "{ul_sub_property}"),       

        # (f"{ul_sub_property} {ul_sub_property2}", "{ul_sub_property} {ul_sub_property2}"),
        # (f"{ul_sub_property} {ul_sub_property2}", "{ul_sub_property} {ul_sub_property2}"),
        
        (f"{filler}", "{filler}"),
        (f"{filler}", "{filler}"),

        (f"{brand} {brand2}", "{brand} {brand2}"),
        (f"{brand} {brand2}", "{brand} {brand2}"),
        (f"{brand} and {brand2}", "{brand} and {brand2}"),
        (f"{brand}, {brand2}", "{brand}, {brand2}"),

        (f"{brand} {brand2} {polymer}", "{brand} {brand2} {polymer}"),
        (f"{brand} {brand2} {polymer}", "{brand} {brand2} {polymer}"),
        (f"{brand} {brand2} and {polymer}", "{brand} {brand2} and {polymer}"),
        (f"{brand}, {brand2}, {polymer}", "{brand}, {brand2}, {polymer}"),

        (f"{brand} {brand2} {feature}", "{brand} {brand2} {feature}"),
        (f"{brand} {brand2} {feature}", "{brand} {brand2} {feature}"),
        (f"{brand} {brand2} and {feature}", "{brand} {brand2} and {feature}"),
        (f"{brand}, {brand2}, {feature}", "{brand}, {brand2}, {feature}"),

        (f"{brand} {brand2} {filler}", "{brand} {brand2} {filler}"),
        (f"{brand} {brand2} {filler}", "{brand} {brand2} {filler}"),
        (f"{brand} {brand2} and {filler}", "{brand} {brand2} and {filler}"),
        (f"{brand}, {brand2}, {filler}", "{brand}, {brand2}, {filler}"),    
        
        (f"{brand} grade with {property_abb}, and {ul_property}", "{brand} grade with {property_abb}, and {ul_property}"),
        (f"{brand} grade with {property_abb}, and {ul_sub_property}", "{brand} grade with {property_abb}, and {ul_sub_property}"),
        (f"{filler} {feature} {polymer}", "{filler} {feature} {polymer}"),
        (f"{polymer} with {property_abb} {property_abb2}", "{polymer} with {property_abb} {property_abb2}"),
        (f"{feature} {property_abb} {polymer} {processing}", "{feature} {property_abb} {polymer} {processing}"),
        (f"products with {property_abb}", "products with {property_abb}"),
        (f"{brand} {property_abb}; {property_abb2}", "{brand} {property_abb}; {property_abb2}"),
        (f"Find a material that is {property_abb}", "Find a material that is {property_abb}"),
        (f"{property_abb}", "{property_abb}"),
        (f"{polymer} {property_abb} {polymer2}", "{polymer} {property_abb} {polymer2}"),
        (f"looking for a {polymer} {filler} with {property_abb}", "looking for a {polymer} {filler} with {property_abb}"),
        (f"get {polymer} material with the condition, {property_abb}", "get {polymer} material with the condition, {property_abb}"),
        (f"{polymer} with {property_abb}", "{polymer} with {property_abb}"),
        (f"{feature}, {polymer}, with {property_abb}", "{feature}, {polymer}, with {property_abb}"),
        (f"{property_abb} {polymer}", "{property_abb} {polymer}"),
        (f"{property_abb} {feature}", "{property_abb} {feature}"),
        (f"{polymer} {property_abb}, {feature}", "{polymer} {property_abb}, {feature}"),
        (f"{brand} {property_abb}", "{brand} {property_abb}"),
        
        (f"{brand} grade with {property_rm}, and {ul_property}", "{brand} grade with {property_rm}, and {ul_property}"),
        (f"{brand} grade with {property_rm}, and {ul_sub_property}", "{brand} grade with {property_rm}, and {ul_sub_property}"),
        (f"{filler} {feature} {polymer}", "{filler} {feature} {polymer}"),
        (f"{polymer} with {property_rm} {property_rm2}", "{polymer} with {property_rm} {property_rm2}"),
        (f"{feature} {property_rm} {polymer} {processing}", "{feature} {property_rm} {polymer} {processing}"),
        (f"products with {property_rm}", "products with {property_rm}"),
        (f"{brand} {property_rm}; {property_rm2}", "{brand} {property_rm}; {property_rm2}"),
        (f"Find a material that is {property_rm}", "Find a material that is {property_rm}"),
        (f"{property_rm}", "{property_rm}"),
        (f"{polymer} {property_rm} {polymer2}", "{polymer} {property_rm} {polymer2}"),
        (f"looking for a {polymer} {filler} with {property_rm}", "looking for a {polymer} {filler} with {property_rm}"),
        (f"get {polymer} material with the condition, {property_rm}", "get {polymer} material with the condition, {property_rm}"),
        (f"{polymer} with {property_rm}", "{polymer} with {property_rm}"),
        (f"{feature}, {polymer}, with {property_rm}", "{feature}, {polymer}, with {property_rm}"),
        (f"{property_rm} {polymer}", "{property_rm} {polymer}"),
        (f"{property_rm} {feature}", "{property_rm} {feature}"),
        (f"{polymer} {property_rm}, {feature}", "{polymer} {property_rm}, {feature}"),
        (f"{brand} {property_rm}", "{brand} {property_rm}"),
        
        
        (f"{application}", "{application}"),
        (f"{polymer}", "{polymer}"),
        (f"{polymer} {feature}", "{polymer} {feature}"),
        # (f"{application} {polymer}", "{application} {polymer}"),
        (f"{feature}", "{feature}"),
        (f"good {feature}", "good {feature}"),
        (f"{brand} {feature}", "{brand} {feature}"),
        (f"{polymer} {processing}", "{polymer} {processing}"),
        # (f"{application} {brand}", "{application} {brand}"),
        (f"{brand} {polymer}", "{brand} {polymer}"),
        (f"{grade} {polymer}", "{grade} {polymer}"),
        # (f"{application} {feature}", "{application} {feature}"),
        # (f"{application} {polymer} {feature}", "{application} {polymer} {feature}"),
        (f"{brand} {processing}", "{brand} {processing}"),
        (f"{processing}", "{processing}"),
        # (f"{polymer} {competitor_grade}", "{polymer} {competitor_grade}"),
        # (f"{application} {processing}", "{application} {processing}"),
        (f"{grade} {brand}", "{grade} {brand}"),
        # (f"{application} {brand} {feature}", "{application} {brand} {feature}"),
        # (f"{brand} {competitor_grade}", "{brand} {competitor_grade}"),
        (f"{brand} {polymer} {feature}", "{brand} {polymer} {feature}"),
        # (f"{grade} {application}", "{grade} {application}"),
        # (f"{grade} {competitor_grade}", "{grade} {competitor_grade}"),
        (f"{polymer} {feature} {processing}", "{polymer} {feature} {processing}"),
        # (f"{polymer} {nsf_cert}", "{polymer} {nsf_cert}"),
        # (f"{application} {competitor_grade}", "{application} {competitor_grade}"),
        (f"{brand} {delivery_form}", "{brand} {delivery_form}"),
        # (f"{application} {polymer} {processing}", "{application} {polymer} {processing}"),
        # (f"{nsf_cert}", "{nsf_cert}"),
        (f"{grade} {feature}", "{grade} {feature}"),
        (f"{grade} with excellent {feature}", "{grade} with excellent {feature}"),
        (f"{feature} {processing}", "{feature} {processing}"),
        (f"{polymer} {delivery_form}", "{polymer} {delivery_form}"),
        # (f"{grade} {brand} {polymer}", "{grade} {brand} {polymer}"),
        (f"{brand} {polymer} {processing}", "{brand} {polymer} {processing}"),
        # (f"{application} {brand} {processing}", "{application} {brand} {processing}"),
        # (f"{brand} {nsf_cert}", {brand} {nsf_cert}"),
        (f"{grade} {processing}", "{grade} {processing}"),
        # (f"{feature} {competitor_grade}", "{feature} {competitor_grade}"),
        # (f"{grade} {polymer} {competitor_grade}", "{grade} {polymer} {competitor_grade}"),
        # (f"{application} {delivery_form}", "{application} {delivery_form}"),
        (f"{grade} {feature} {delivery_form}", "{grade} {feature} {delivery_form}"),
        # (f"{grade} {application} {feature}", "{grade} {application} {feature}"),
        (f"{delivery_form}", "{delivery_form}"),
        (f"{grade} {polymer} {feature}", "{grade} {polymer} {feature}"),
        (f"{polymer} {feature} {delivery_form}", "{polymer} {feature} {delivery_form}"),
        # (f"{application} {brand} {delivery_form}", "{application} {brand} {delivery_form}" ),
        (f"{grade} {polymer} {feature} {delivery_form}", "{grade} {polymer} {feature} {delivery_form}"),
        (f"{brand} {feature} {processing}", "{brand} {feature} {processing}"),
        # (f"{delivery_form} {competitor_grade}", "{delivery_form} {competitor_grade}"),
        
        (f"{polymer}, {feature}", "{polymer}, {feature}"),
        (f"{application}, {polymer}", "{application}, {polymer}"),
        (f"{brand}, {feature}", "{brand}, {feature}"),
        (f"{polymer}, {processing}", "{polymer}, {processing}"),
        (f"{application}, {brand}", "{application}, {brand}"),
        (f"{brand}, {polymer}", "{brand}, {polymer}"),
        (f"{grade}, {polymer}", "{grade}, {polymer}"),
        (f"{application}, {feature}", "{application}, {feature}"),
        (f"{application}, {polymer}, {feature}", "{application}, {polymer}, {feature}"),
        (f"{brand}, {processing}", "{brand}, {processing}"),
        # (f"{polymer}, {competitor_grade}", "{polymer}, {competitor_grade}"),
        (f"{application}, {processing}", "{application}, {processing}"),
        (f"{grade}, {brand}", "{grade}, {brand}"),
        (f"{application}, {brand}, {feature}", "{application}, {brand}, {feature}"),
        # (f"{brand}, {competitor_grade}", "{brand}, {competitor_grade}"),
        (f"{brand}, {polymer}, {feature}", "{brand}, {polymer}, {feature}"),
        (f"{grade}, {application}", "{grade}, {application}"),
        # (f"{grade}, {competitor_grade}", "{grade}, {competitor_grade}"),
        (f"{polymer}, {feature}, {processing}", "{polymer}, {feature}, {processing}"),
        # (f"{polymer}, {nsf_cert}", "{polymer}, {nsf_cert}"),
        # (f"{application}, {competitor_grade}", ),
        (f"{brand}, {delivery_form}", "{brand}, {delivery_form}"),
        (f"{application}, {polymer}, {processing}", "{application}, {polymer}, {processing}"),
        # (f"{nsf_cert}", "{nsf_cert}"),
        (f"{grade}, {feature}", "{grade}, {feature}"),
        (f"{feature}, {processing}", "{feature}, {processing}"),
        (f"{polymer}, {delivery_form}", "{polymer}, {delivery_form}"),
        (f"{grade}, {brand}, {polymer}", "{grade}, {brand}, {polymer}"),
        (f"{brand}, {polymer}, {processing}", "{brand}, {polymer}, {processing}"),
        (f"{application}, {brand}, {processing}", "{application}, {brand}, {processing}"),
        # (f"{brand}, {nsf_cert}", "{brand}, {nsf_cert}"),
        (f"{grade}, {processing}", "{grade}, {processing}"),
        # (f"{feature}, {competitor_grade}", "{feature}, {competitor_grade}"),
        # (f"{grade}, {polymer}, {competitor_grade}", "{grade}, {polymer}, {competitor_grade}"),
        (f"{application}, {delivery_form}", "{application}, {delivery_form}"),
        (f"{grade}, {feature}, {delivery_form}", "{grade}, {feature}, {delivery_form}"),
        (f"{grade}, {application}, {feature}", "{grade}, {application}, {feature}"),
        (f"{delivery_form}", "{delivery_form}"),
        (f"{grade}, {polymer}, {feature}", "{grade}, {polymer}, {feature}"),
        (f"{polymer}, {feature}, {delivery_form}", "{polymer}, {feature}, {delivery_form}"),
        (f"{application}, {brand}, {delivery_form}", "{application}, {brand}, {delivery_form}"),
        (f"{grade}, {polymer}, {feature}, {delivery_form}", "{grade}, {polymer}, {feature}, {delivery_form}"),
        (f"{brand}, {feature}, {processing}", "{brand}, {feature}, {processing}"),
        # (f"{delivery_form}, {competitor_grade}", "{delivery_form}, {competitor_grade}"),
        
        # twice same entity
        (f"{polymer}, {polymer2}", "{polymer}, {polymer2}"),
        (f"{polymer}, {feature}, {feature2}", "{polymer}, {feature}, {feature2}"),
        (f"{polymer}, {polymer2}, {feature}", "{polymer}, {polymer2}, {feature}"),
        (f"{application}, {polymer}, {polymer2}", "{application}, {polymer}, {polymer2}"),
        (f"{feature}, {feature2}", "{feature}, {feature2}"),
        (f"{brand}, {feature}, {feature2}", "{brand}, {feature}, {feature2}"),
        (f"{polymer}, {polymer2}, {processing}", "{polymer}, {polymer2}, {processing}"),
        (f"{polymer}, {processing}, {processing2}", "{polymer}, {processing}, {processing2}"),
        
        # (f"{brand}, {polymer}, {polymer2}", "{brand}, {polymer}, {polymer2}"),
        # (f"{grade}, {polymer}, {polymer2}", "{grade}, {polymer}, {polymer2}"),
        (f"{application}, {feature}, {feature2}", "{application}, {feature}, {feature2}"),
        # (f"{application}, {polymer}, {polymer2}, {feature}", "{application}, {polymer}, {polymer2}, {feature}"),
        (f"{application}, {polymer}, {feature}, {feature2}", "{application}, {polymer}, {feature}, {feature2}"),
        (f"{brand}, {processing}, {processing2}", "{brand}, {processing}, {processing2}"),
        
        (f"{brand}, {delivery_form}, {delivery_form2}", "{brand}, {delivery_form}, {delivery_form2}"),
        (f"{polymer}, {delivery_form} , {delivery_form2}", "{polymer}, {delivery_form} , {delivery_form2}"),
        # (f"{polymer}, {polymer2} , {delivery_form2}", "{polymer}, {polymer2} , {delivery_form2}"),
        (f"{application}, {delivery_form}, {delivery_form2}", "{application}, {delivery_form}, {delivery_form2}"),
    
        
        (f"{polymer} {polymer2}", "{polymer} {polymer2}"),
        (f"{polymer} {feature} {feature2}", "{polymer} {feature} {feature2}"),
        (f"{polymer} {polymer2} {feature}", "{polymer} {polymer2} {feature}"),
        (f"{application} {polymer} {polymer2}", "{application} {polymer} {polymer2}"),
        (f"{feature} {feature2}", "{feature} {feature2}"),
        (f"{brand} {feature} {feature2}", "{brand} {feature} {feature2}"),
        (f"{polymer} {polymer2} {processing}", "{polymer} {polymer2} {processing}"),
        (f"{polymer} {processing} {processing2}", "{polymer} {processing} {processing2}"),
        
        # (f"{brand} {polymer} {polymer2}", "{brand} {polymer} {polymer2}"),
        # (f"{grade} {polymer} {polymer2}", "{grade} {polymer} {polymer2}"),
        (f"{application} {feature} {feature2}", "{application} {feature} {feature2}"),
        # (f"{application} {polymer} {polymer2} {feature}", "{application} {polymer} {polymer2} {feature}"),
        (f"{application} {polymer} {feature} {feature2}", "{application} {polymer} {feature} {feature2}"),
        (f"{brand} {processing} {processing2}", "{brand} {processing} {processing2}"),
        
        (f"{brand} {delivery_form} {delivery_form2}", "{brand} {delivery_form} {delivery_form2}"),
        (f"{polymer} {delivery_form} , {delivery_form2}", "{polymer} {delivery_form} , {delivery_form2}"),
        # (f"{polymer} {polymer2} , {delivery_form2}", "{polymer} {polymer2} , {delivery_form2}"),
        (f"{application} {delivery_form} {delivery_form2}", "{application} {delivery_form} {delivery_form2}"),
        
        (f"{polymer} and {polymer2}", "{polymer} and {polymer2}"),
        (f"{polymer} and {feature} and {feature2}", "{polymer} and {feature} and {feature2}"),
        (f"{polymer} and {polymer2} and {feature}", "{polymer} and {polymer2} and {feature}"),
        (f"{application} and {polymer} and {polymer2}", "{application} and {polymer} and {polymer2}"),
        (f"{feature} and {feature2}", "{feature} and {feature2}"),
        (f"{brand} and {feature} and {feature2}", "{brand} and {feature} and {feature2}"),
        (f"{polymer} and {polymer2} and {processing}", "{polymer} and {polymer2} and {processing}"),
        (f"{polymer} and {processing} and {processing2}", "{polymer} and {processing} and {processing2}"),
        
        # (f"{brand} and {polymer} and {polymer2}", "{brand} and {polymer} and {polymer2}"),
        # (f"{grade} and {polymer} and {polymer2}", "{grade} and {polymer} and {polymer2}"),
        (f"{application} and {feature} and {feature2}", "{application} and {feature} and {feature2}"),
        # (f"{application} and {polymer} and {polymer2} and {feature}", "{application} and {polymer} and {polymer2} and {feature}"),
        (f"{application} and {polymer} and {feature} and {feature2}", "{application} and {polymer} and {feature} and {feature2}"),
        (f"{brand} and {processing} and {processing2}", "{brand} and {processing} and {processing2}"),
        
        (f"{brand} and {delivery_form} and {delivery_form2}", "{brand} and {delivery_form} and {delivery_form2}"),
        (f"{polymer} and {delivery_form} , {delivery_form2}", "{polymer} and {delivery_form} , {delivery_form2}"),
        # (f"{polymer} and {polymer2} , {delivery_form2}", "{polymer} and {polymer2} , {delivery_form2}"),
        (f"{application} and {delivery_form} and {delivery_form2}", "{application} and {delivery_form} and {delivery_form2}"),
        
        # nsf water railway
        (f"{nsf}", "{nsf}"),
        (f"{nsf} certification", "{nsf} certification"),
        (f"{nsf} certified", "{nsf} certified"),
        (f"{nsf} grades", "{nsf} grades"),
        (f"grade with {nsf}", "grade with {nsf}"),

        (f"{water_name}", "{water_name}"),
        (f"{water_name} certification", "{water_name} certification"),
        (f"{water_name} certified", "{water_name} certified"),
        (f"{water_name} grades", "{water_name} grades"),
        (f"grade with {water_name}", "grade with {water_name}"),

        (f"{nsf}, {feature}", "{nsf}, {feature}"),
        (f"{delivery_form}, {nsf}", "{delivery_form}, {nsf}"),
        (f"{nsf}, {processing}", "{nsf}, {processing}"),
        (f"{application}, {nsf}", "{application}, {nsf}"),
        (f"{nsf}, {feature}", "{nsf}, {feature}"),
        (f"{brand}, {nsf}", "{brand}, {nsf}"),
        (f"{nsf}, {processing}", "{nsf}, {processing}"),
        (f"{nsf}, {brand}", "{nsf}, {brand}"),
        
        (f"{water_name}, {feature}", "{water_name}, {feature}"),
        (f"{delivery_form}, {water_name}", "{delivery_form}, {water_name}"),
        (f"{water_name}, {processing}", "{water_name}, {processing}"),
        (f"{application}, {water_name}", "{application}, {water_name}"),
        (f"{water_name}, {feature}", "{water_name}, {feature}"),
        (f"{brand}, {water_name}", "{brand}, {water_name}"),
        (f"{water_name}, {processing}", "{water_name}, {processing}"),
        (f"{water_name}, {brand}", "{water_name}, {brand}"),

        (f"{railway_name} certification", "{railway_name} certification"),
        (f"{railway_name} certified", "{railway_name} certified"),
        (f"{railway_name}, {feature}", "{railway_name}, {feature}"),
        (f"{delivery_form}, {railway_name}", "{delivery_form}, {railway_name}"),
        (f"{railway_name}, {processing}", "{railway_name}, {processing}"),
        (f"{application}, {railway_name}", "{application}, {railway_name}"),
        (f"{railway_name}, {feature}", "{railway_name}, {feature}"),
        (f"{brand}, {railway_name}", "{brand}, {railway_name}"),
        (f"{railway_name}, {processing}", "{railway_name}, {processing}"),
        (f"{railway_name}, {brand}", "{railway_name}, {brand}"),
        
        (f"{nsf} {feature}", "{nsf} {feature}"),
        (f"{delivery_form} {nsf}", "{delivery_form} {nsf}"),
        (f"{nsf} {processing}", "{nsf} {processing}"),
        (f"{application} {nsf}", "{application} {nsf}"),
        (f"{nsf} {feature}", "{nsf} {feature}"),
        (f"{brand} {nsf}", "{brand} {nsf}"),
        (f"{nsf} {processing}", "{nsf} {processing}"),
        (f"{nsf} {brand}", "{nsf} {brand}"),
        
        (f"{water_name} {feature}", "{water_name} {feature}"),
        (f"{delivery_form} {water_name}", "{delivery_form} {water_name}"),
        (f"{water_name} {processing}", "{water_name} {processing}"),
        (f"{application} {water_name}", "{application} {water_name}"),
        (f"{water_name} {feature}", "{water_name} {feature}"),
        (f"{brand} {water_name}", "{brand} {water_name}"),
        (f"{water_name} {processing}", "{water_name} {processing}"),
        (f"{water_name} {brand}", "{water_name} {brand}"),
        
        (f"{railway_name} {feature}", "{railway_name} {feature}"),
        (f"{delivery_form} {railway_name}", "{delivery_form} {railway_name}"),
        (f"{railway_name} {processing}", "{railway_name} {processing}"),
        (f"{application} {railway_name}", "{application} {railway_name}"),
        (f"{railway_name} {feature}", "{railway_name} {feature}"),
        (f"{brand} {railway_name}", "{brand} {railway_name}"),
        (f"{railway_name} {processing}", "{railway_name} {processing}"),
        (f"{railway_name} {brand}", "{railway_name} {brand}"),
        
        (f"{nsf} and {feature}", "{nsf} and {feature}"),
        (f"{delivery_form} and {nsf}", "{delivery_form} and {nsf}"),
        (f"{nsf} and {processing}", "{nsf} and {processing}"),
        (f"{application} and {nsf}", "{application} and {nsf}"),
        (f"{nsf} and {feature}", "{nsf} and {feature}"),
        (f"{brand} and {nsf}", "{brand} and {nsf}"),
        (f"{nsf} and {processing}", "{nsf} and {processing}"),
        (f"{nsf} and {brand}", "{nsf} and {brand}"),
        
        (f"{water_name} and {feature}", "{water_name} and {feature}"),
        (f"{delivery_form} and {water_name}", "{delivery_form} and {water_name}"),
        (f"{water_name} and {processing}", "{water_name} and {processing}"),
        (f"{application} and {water_name}", "{application} and {water_name}"),
        (f"{water_name} and {feature}", "{water_name} and {feature}"),
        (f"{brand} and {water_name}", "{brand} and {water_name}"),
        (f"{water_name} and {processing}", "{water_name} and {processing}"),
        (f"{water_name} and {brand}", "{water_name} and {brand}"),
        
        (f"{railway_name} and {feature}", "{railway_name} and {feature}"),
        (f"{delivery_form} and {railway_name}", "{delivery_form} and {railway_name}"),
        (f"{railway_name} and {processing}", "{railway_name} and {processing}"),
        (f"{application} and {railway_name}", "{application} and {railway_name}"),
        (f"{railway_name} and {feature}", "{railway_name} and {feature}"),
        (f"{brand} and {railway_name}", "{brand} and {railway_name}"),
        (f"{railway_name} and {processing}", "{railway_name} and {processing}"),
        (f"{railway_name} and {brand}", "{railway_name} and {brand}"),

        # auto and other certiications
        (f"{railway_name} {auto_cert_name}", "{railway_name} {auto_cert_name}"),
        (f"{railway_name} {auto_cert_name}", "{railway_name} {auto_cert_name}"),
        (f"{railway_name}, {auto_cert_name} certifications", "{railway_name}, {auto_cert_name} certifications"),
        (f"{railway_name} and {auto_cert_name}", "{railway_name} and {auto_cert_name}"),
        (f"{railway_name} certification {auto_cert_name} approval", "{railway_name} certification {auto_cert_name} approval"),
        (f"{railway_name} {auto_cert_name} {oem_name}", "{railway_name} {auto_cert_name} {oem_name}"),
        (f"{railway_name} {oem_name} {auto_cert_name}", "{railway_name} {oem_name} {auto_cert_name}"),
        (f"{railway_name} {oem_name}", "{railway_name} {oem_name}"),
        (f"{oem_name} {railway_name}", "{oem_name} {railway_name}"),
        (f"{oem_name} {auto_cert_name} {railway_name}", "{oem_name} {auto_cert_name} {railway_name}"),
        (f"{auto_cert_name} {oem_name} {railway_name}", "{auto_cert_name} {oem_name} {railway_name}"),
        (f"{auto_cert_name} {railway_name}", "{auto_cert_name} {railway_name}"),
        (f"{auto_cert_name} {railway_name}", "{auto_cert_name} {railway_name}"),

        (f"{water_name} {auto_cert_name}", "{water_name} {auto_cert_name}"),
        (f"{water_name} {auto_cert_name}", "{water_name} {auto_cert_name}"),
        (f"{water_name}, {auto_cert_name} certifications", "{water_name}, {auto_cert_name} certifications"),
        (f"{water_name} and {auto_cert_name}", "{water_name} and {auto_cert_name}"),
        (f"{water_name} certified and {auto_cert_name} approved", "{water_name} certified and {auto_cert_name} approved"),
        (f"{water_name} {auto_cert_name} {oem_name}", "{water_name} {auto_cert_name} {oem_name}"),
        (f"{water_name} {oem_name} {auto_cert_name}", "{water_name} {oem_name} {auto_cert_name}"),
        (f"{water_name} {oem_name}", "{water_name} {oem_name}"),
        (f"{oem_name} {water_name}", "{oem_name} {water_name}"),
        (f"{oem_name} {auto_cert_name} {water_name}", "{oem_name} {auto_cert_name} {water_name}"),
        (f"{auto_cert_name} {oem_name} {water_name}", "{auto_cert_name} {oem_name} {water_name}"),
        (f"{auto_cert_name} {water_name}", "{auto_cert_name} {water_name}"),
        (f"{auto_cert_name} {water_name}", "{auto_cert_name} {water_name}"),

        (f"{nsf} {auto_cert_name}", "{nsf} {auto_cert_name}"),
        (f"{nsf} {auto_cert_name}", "{nsf} {auto_cert_name}"),
        (f"{nsf}, {auto_cert_name} certifications", "{nsf}, {auto_cert_name} certifications"),
        (f"{nsf} and {auto_cert_name}", "{nsf} and {auto_cert_name}"),
        (f"{nsf} certification and {auto_cert_name} approved", "{nsf} certification and {auto_cert_name} approved"),
        (f"{nsf} {auto_cert_name} {oem_name}", "{nsf} {auto_cert_name} {oem_name}"),
        (f"{nsf} {oem_name} {auto_cert_name}", "{nsf} {oem_name} {auto_cert_name}"),
        (f"{nsf} {oem_name}", "{nsf} {oem_name}"),
        (f"{oem_name} {nsf}", "{oem_name} {nsf}"),
        (f"{oem_name} {auto_cert_name} {nsf}", "{oem_name} {auto_cert_name} {nsf}"),
        (f"{auto_cert_name} {oem_name} {nsf}", "{auto_cert_name} {oem_name} {nsf}"),
        (f"{auto_cert_name} {nsf}", "{auto_cert_name} {nsf}"),
        (f"{auto_cert_name} {nsf}", "{auto_cert_name} {nsf}"),

        (f"{nsf}, {water_name}, {railway_name}, {auto_cert_name} certifications", "{nsf}, {water_name}, {railway_name}, {auto_cert_name} certifications"),
        (f"{water_name}, {railway_name}, {nsf}, {auto_cert_name} certifications", "{water_name}, {railway_name}, {nsf}, {auto_cert_name} certifications"),
        (f"{railway_name}, {nsf}, {water_name}, {auto_cert_name} certifications", "{railway_name}, {nsf}, {water_name}, {auto_cert_name} certifications"),
        (f"{auto_cert_name}, {railway_name}, {nsf}, {water_name} certifications", "{auto_cert_name}, {railway_name}, {nsf}, {water_name} certifications"),
        (f"{nsf}, {water_name}, {railway_name}, {oem_name} certifications", "{nsf}, {water_name}, {railway_name}, {oem_name} certifications"),
        (f"{water_name}, {railway_name}, {nsf}, {oem_name} certifications", "{water_name}, {railway_name}, {nsf}, {oem_name} certifications"),
        (f"{railway_name}, {nsf}, {water_name}, {oem_name} certifications", "{railway_name}, {nsf}, {water_name}, {oem_name} certifications"),
        (f"{oem_name}, {railway_name}, {nsf}, {water_name} certifications", "{oem_name}, {railway_name}, {nsf}, {water_name} certifications"),
        
        # no space
        (f"{processing.replace(' ', '')}", "{processing}"),
        (f"{feature.replace(' ', '')}", "{feature}"),

        (f"{polymer} {filler}", "{polymer} {filler}"),
        # (f"{ul_property} in UL listing", "{ul_property} in UL listing"),
        (f"{ul_property}", "{ul_property}"),
        (f"Need a grade with a {ul_property}", "Need a grade with a {ul_property}"),
        (f"{ul_property} specification for {grade}", "{ul_property} specification for {grade}"),
        (f"{brand} grade with a {ul_property}", "{brand} grade with a {ul_property}"),
        (f"{brand} grade with {property}, and {ul_property}", "{brand} grade with {property}, and {ul_property}"),
        (f"{polymer} {ul_property}", "{polymer} {ul_property}"),
        (f"{polymer} with {ul_property}", "{polymer} with {ul_property}"),
        (f"{ul_property} and {ul_property2} grade", "{ul_property} and {ul_property2} grade"),
        (f"grade for {application} with {ul_property}", "grade for {application} with {ul_property}"),
        (f"can you recommend a {ul_property} {polymer} {brand} grade", "can you recommend a {ul_property} {polymer} {brand} grade"),
        (f"{polymer} {processing} grade that is {ul_property}", "{polymer} {processing} grade that is {ul_property}"),
        (f"{brand} {ul_property}", "{brand} {ul_property}"),
        (f"{ul_property}", "{ul_property}"),
        
        # (f"{ul_sub_property} in UL listing", "{ul_sub_property} in UL listing"),
        (f"{ul_sub_property}", "{ul_sub_property}"),
        (f"Need a grade with a {ul_sub_property}", "Need a grade with a {ul_sub_property}"),
        (f"{ul_sub_property} specification for {grade}", "{ul_sub_property} specification for {grade}"),
        (f"{brand} grade with a {ul_sub_property}", "{brand} grade with a {ul_sub_property}"),
        (f"{brand} grade with {property}, and {ul_sub_property}", "{brand} grade with {property}, and {ul_sub_property}"),
        (f"{polymer} {ul_sub_property}", "{polymer} {ul_sub_property}"),
        (f"{polymer} with {ul_sub_property}", "{polymer} with {ul_sub_property}"),
        # (f"{ul_sub_property} and {ul_sub_property2} grade", "{ul_sub_property} and {ul_sub_property2} grade"),
        (f"grade for {application} with {ul_sub_property}", "grade for {application} with {ul_sub_property}"),
        (f"can you recommend a {ul_sub_property} {polymer} {brand} grade", "can you recommend a {ul_sub_property} {polymer} {brand} grade"),
        (f"{polymer} {processing} grade that is {ul_sub_property}", "{polymer} {processing} grade that is {ul_sub_property}"),
        (f"{brand} {ul_sub_property}", "{brand} {ul_sub_property}"),
        (f"{ul_sub_property}", "{ul_sub_property}"),
        (f"flexible {polymer} grades", "flexible {polymer} grades"),
        (f"{filler} {feature} {polymer}", "{filler} {feature} {polymer}"),
        (f"{polymer} with {property} {property2}", "{polymer} with {property} {property2}"),
        (f"{feature} {property} {polymer} {processing}", "{feature} {property} {polymer} {processing}"),
        (f"{polymer} for {processing}", "{polymer} for {processing}"),
        (f"{feature} material", "{feature} material"),
        (f"products with {property}", "products with {property}"),
        (f"{feature} {polymer} {application} {feature2}", "{feature} {polymer} {application} {feature2}"),
        # (f"{MODIFIER_RANGE} {property} grade", "{MODIFIER_RANGE} {property} grade"),
        (f"{filler} {polymer}", "{filler} {polymer}"),
        (f"{brand} {property}; {property2}", "{brand} {property}; {property2}"),
        (f"Find a material that is {property}", "Find a material that is {property}"),
        (f"{feature} {filler}", "{feature} {filler}"),
        (f"{polymer} {filler}", "{polymer} {filler}"),
        (f"{property}", "{property}"),
        (f"{processing} {grade} {brand} that is {feature}", "{processing} {grade} {brand} that is {feature}"),
        # (f"grade meet {AUTO_CERT}", "grade meet {AUTO_CERT}"),
        (f"{polymer} {property} {polymer2}", "{polymer} {property} {polymer2}"),
        (f"looking for a {polymer} {filler} with {property}", "looking for a {polymer} {filler} with {property}"),
        (f"{application} {brand}", "{application} {brand}"),
        (f"{application} grades", "{application} grades"),
        (f"{polymer} {filler} {feature}", "{polymer} {filler} {feature}"),
        # (f"{polymer} with {MODIFIER_RANGE} {property}", "{polymer} with {MODIFIER_RANGE} {property}"),
        (f"get {polymer} material with the condition, {property}", "get {polymer} material with the condition, {property}"),
        (f"a {polymer} with {filler}", "a {polymer} with {filler}"),
        (f"{polymer} with {property}", "{polymer} with {property}"),
        (f"{feature}, {polymer}, with {property}", "{feature}, {polymer}, with {property}"),
        # (f"Yellow Card {brand} {grade}", "Yellow Card {brand} {grade}"),
        # (f"{polymer} silver color", "{polymer} silver color"),
        (f"{property} {polymer}", "{property} {polymer}"),
        (f"{polymer} for {application} with {property}", "{polymer} for {application} with {property}"),
        (f"{polymer} {filler} {feature}", "{polymer} {filler} {feature}"),
        (f"{polymer} {filler} with good {feature}", "{polymer} {filler} with good {feature}"),
        (f"{property} {feature}", "{property} {feature}"),
        (f"{polymer} with {nsf} approval", "{polymer} with {nsf} approval"),
        (f"flexible {polymer} for {processing}", "flexible {polymer} for {processing}"),
        (f"{polymer} with good {feature}", "{polymer} with good {feature}"),
        # (f"{feature} and {feature2} {polymer} for {application} application with {AUTO_CERT} approval", "{feature} and {feature2} {polymer} for {application} application with {AUTO_CERT} approval"),
        (f"{filler} {feature} {polymer}", "{filler} {feature} {polymer}"),
        (f"{polymer} {property}, {feature}", "{polymer} {property}, {feature}"),
        (f"{polymer} {filler} with {feature}", "{polymer} {filler} with {feature}"),
        (f"{feature} grade for {application} solution", "{feature} grade for {application} solution"),
        (f"can you recommend a {feature} {filler} {polymer} grade that is approved for {application}", "can you recommend a {feature} {filler} {polymer} grade that is approved for {application}"),
        # (f"SEBS {property}", "SEBS {property}"),
        (f"material with {filler}", "material with {filler}"),
        (f"{polymer} for {application} {feature}", "{polymer} for {application} {feature}"),
        # (f"is {polymer} anti-oxidate", "is {polymer} anti-oxidate"),
        # (f"{polymer} can reduce the vibration and noise", "{polymer} can reduce the vibration and noise"),
        (f"{brand} {property}", "{brand} {property}"),
        (f"{filler} in {polymer}", "{filler} in {polymer}"),
        (f"{property} grade", "{property} grade"),
        
        (f"{brand} grade with {property_rm}, and {ul_property}", "{brand} grade with {property_rm}, and {ul_property}"),
        (f"{brand} grade with {property_rm}, and {ul_sub_property}", "{brand} grade with {property_rm}, and {ul_sub_property}"),
        (f"{polymer} with {property_rm} {property_rm2}", "{polymer} with {property_rm} {property_rm2}"),
        (f"{feature} {property_rm} {polymer} {processing}", "{feature} {property_rm} {polymer} {processing}"),
        (f"products with {property_rm}", "products with {property_rm}"),
        (f"{brand} {property_rm}; {property_rm2}", "{brand} {property_rm}; {property_rm2}"),
        (f"Find a material that is {property_rm}", "Find a material that is {property_rm}"),
        (f"{property_rm}", "{property_rm}"),
        (f"{polymer} {property_rm} {polymer2}", "{polymer} {property_rm} {polymer2}"),
        (f"looking for a {polymer} {filler} with {property_rm}", "looking for a {polymer} {filler} with {property_rm}"),
        (f"get {polymer} material with the condition, {property_rm}", "get {polymer} material with the condition, {property_rm}"),
        (f"{polymer} with {property_rm}", "{polymer} with {property_rm}"),
        (f"{feature}, {polymer}, with {property_rm}", "{feature}, {polymer}, with {property_rm}"),
        (f"{property_rm} {polymer}", "{property_rm} {polymer}"),
        (f"{polymer} for {application} with {property_rm}", "{polymer} for {application} with {property_rm}"),
        (f"{property_rm} {feature}", "{property_rm} {feature}"),
        (f"{polymer} {property_rm}, {feature}", "{polymer} {property_rm}, {feature}"),
        (f"material with {filler}", "material with {filler}"),
        (f"{brand} {property_rm}", "{brand} {property_rm}"),
        (f"{property_rm} grade", "{property_rm} grade"),
        
        (f"{brand} grade with {property_abb}, and {ul_property}", "{brand} grade with {property_abb}, and {ul_property}"),
        (f"{brand} grade with {property_abb}, and {ul_sub_property}", "{brand} grade with {property_abb}, and {ul_sub_property}"),
        (f"{polymer} with {property_abb} {property_abb2}", "{polymer} with {property_abb} {property_abb2}"),
        (f"{feature} {property_abb} {polymer} {processing}", "{feature} {property_abb} {polymer} {processing}"),
        (f"products with {property_abb}", "products with {property_abb}"),
        (f"{brand} {property_abb}; {property_abb2}", "{brand} {property_abb}; {property_abb2}"),
        (f"Find a material that is {property_abb}", "Find a material that is {property_abb}"),
        (f"{property_abb}", "{property_abb}"),
        (f"{polymer} {property_abb} {polymer2}", "{polymer} {property_abb} {polymer2}"),
        (f"looking for a {polymer} {filler} with {property_abb}", "looking for a {polymer} {filler} with {property_abb}"),
        (f"get {polymer} material with the condition, {property_abb}", "get {polymer} material with the condition, {property_abb}"),
        (f"{polymer} with {property_abb}", "{polymer} with {property_abb}"),
        (f"{feature}, {polymer}, with {property_abb}", "{feature}, {polymer}, with {property_abb}"),
        (f"{property_abb} {polymer}", "{property_abb} {polymer}"),
        (f"{polymer} for {application} with {property_abb}", "{polymer} for {application} with {property_abb}"),
        (f"{property_abb} {feature}", "{property_abb} {feature}"),
        (f"{polymer} {property_abb}, {feature}", "{polymer} {property_abb}, {feature}"),
        (f"material with {filler}", "material with {filler}"),
        (f"{brand} {property_abb}", "{brand} {property_abb}"),
        (f"{property_abb} grade", "{property_abb} grade"),

        # adding filler templates
        (f"{feature} {polymer} {filler}", "{feature} {polymer} {filler}"),
        (f"{polymer} {filler}", "{polymer} {filler}"),
        (f"material with {polymer} {filler}", "material with {polymer} {filler}"),
        (f"{oem_name} {filler}", "{oem_name} {filler}"),
        (f"{auto_cert_name} {filler}", "{auto_cert_name} {filler}"),
        (f"{auto_cert_name} {filler}", "{auto_cert_name} {filler}"),
        (f"{filler} {oem_name}", "{filler} {oem_name}"),
        (f"{filler} {auto_cert_name}", "{filler} {auto_cert_name}"),
        (f"{filler} {auto_cert_name}", "{filler} {auto_cert_name}"),
        (f"{auto_cert_name} {oem_name} {filler}", "{auto_cert_name} {oem_name} {filler}"),
        (f"{filler} {auto_cert_name} {oem_name}", "{filler} {auto_cert_name} {oem_name}"),
        (f"{ul_sub_property} {filler}", "{ul_sub_property} {filler}"),
        (f"{filler} {ul_sub_property}", "{filler} {ul_sub_property}"),

        # region templates
        (f"{filler} {brand} available in {continent}", "{filler} {brand} available in {continent}"),
        (f"{polymer} for {continent} {application}", "{polymer} for {continent} {application}"),
        (f"{polymer} available in {country}", "{polymer} available in {country}"),
        (f"what {polymer} grades are available in {country}?", "what {polymer} grades are available in {country}?"),
        (f"{application} for {country}", "{application} for {country}"),
        (f"{continent} sourced {polymer}", "{continent} sourced {polymer}"),
        (f"{continent} {feature} grades", "{continent} {feature} grades"),
        (f"i want a {continent} {polymer} grade with {feature}", "i want a {continent} {polymer} grade with {feature}"),
        (f"i want an offset to {competitor_grade} for {country}", "i want an offset to {competitor_grade} for {country}"),
        (f"{brand} offset to {competitor_grade} for {city}", "{brand} offset to {competitor_grade} for {city}"),
        (f"{feature}, {feature2} grade for {city}", "{feature}, {feature2} grade for {city}"),
        (f"{polymer} grade available for {country} to use in {application}", "{polymer} grade available for {country} to use in {application}"),
        (f"{grade} grades available to {country} and {feature}", "{grade} grades available to {country} and {feature}"),
        (f"grades which are sold in {state} and have {oem_name} approvals", "grades which are sold in {state} and have {oem_name} approvals"),
        (f"{competitor_grade} offset, black, {country}, {feature}", "{competitor_grade} offset, black, {country}, {feature}"),
        (f"{feature} {brand} importable to {country}", "{feature}{brand} importable to {country}"),
        (f"{property} {polymer} to buy in {country}", "{property} {polymer} to buy in {country}"),
        (f"{country} approved food grade {brand}", "{country} approved food grade {brand}"),
        (f"{feature} {polymer} to ship to {country}", "{feature} {polymer} to ship to {country}"),
        (f"{country} sourceable {polymer} for {application}", "{country} sourceable {polymer} for {application}"),
        (f"{polymer} sourced from {country} for {application}", "{polymer} sourced from {country} for {application}"),
        (f"{water_name} approved {polymer} purchasable in {country}", "{water_name} approved {polymer} purchasable in {country}"),
        (f"{feature} {brand} obtainable in {country}", "{feature} {brand} obtainable in {country}"),
        (f"{competitor_grade} {continent}", "{competitor_grade} {continent}"),

        (f"what {polymer} grades are available in {state}?", "what {polymer} grades are available in {state}?"),
        (f"what {polymer} grades are available in {city}?", "what {polymer} grades are available in {city}?"),
        (f"{polymer} grade available for {state} to use in {application}", "{polymer} grade available for {state} to use in {application}"),
        (f"{polymer} grade available for {city} to use in {application}", "{polymer} grade available for {city} to use in {application}"),
        (f"{feature} {brand} importable to {state}", "{feature}{brand} importable to {state}"),
        (f"{feature} {brand} importable to {city}", "{feature}{brand} importable to {city}"),		
        (f"{property} {polymer} to buy in {state}", "{property} {polymer} to buy in {state}"),
        (f"{property} {polymer} to buy in {city}", "{property} {polymer} to buy in {city}"),
		(f"{polymer}, {feature}, {state}", "{polymer}, {feature}, {state}"),
		(f"{polymer} {feature} {state}", "{polymer}, {feature}, {state}"),
		(f"{polymer}, {feature}, {city}", "{polymer}, {feature}, {city}"),
		(f"{polymer} {feature} {city}", "{polymer}, {feature}, {city}"),
		(f"{country} sourced {brand}", "{country} sourced {brand}"),
		(f"{continent} sourced {brand}", "{continent} sourced {brand}"),

		(f"{continent}", "{continent}"),
		(f"{country}", "{country}"),
		(f"{state}", "{state}"),
		(f"{city}", "{city}"),
		(f"{continent}", "{continent}"),
		(f"{country}", "{country}"),
		(f"{state}", "{state}"),
		(f"{city}", "{city}"),

        (f"{application} for {oem_name}", "{application} for {oem_name}"),
        (f"{application} for {oem_name}", "{application} for {oem_name}"),
        (f"{application} for {oem_name}", "{application} for {oem_name}"),
        (f"{application} for {oem_name}", "{application} for {oem_name}"),
        (f"{application} for {oem_name}", "{application} for {oem_name}"),
        (f"{application}, {oem_name}", "{application}, {oem_name}"),
        (f"{oem_name}, {application}", "{oem_name}, {application}"),
        (f"{oem_name} for {application}", "{oem_name} for {application}"),
        (f"{oem_name} approved for {application}", "{oem_name} approved for {application}"),
        (f"{auto_cert_name} certification for {application}", "{auto_cert_name} certification for {application}"),
        (f"{oem_name}-{auto_cert_name} approved for {application}", "{oem_name}-{auto_cert_name} approved for {application}"),
        (f"{application} with {oem_name}-{auto_cert_name} approval", "{application} with {oem_name}-{auto_cert_name} approval}"),

        (f"{ignore_term}", "{ignore_term}"),
        (f"{ul_property} {ignore_term}", "{ul_property} {ignore_term}"),
        (f"{ul_sub_property} {ignore_term}", "{ul_sub_property} {ignore_term}"),
        (f"{ignore_term} {property}", "{ignore_term} {property}"),
        (f"{filler} {ignore_term}", "{filler} {ignore_term}"),

        (f"{oem_name} {ignore_term}", "{oem_name} {ignore_term}"),
        (f"{auto_cert_name} {ignore_term}", "{auto_cert_name} {ignore_term}"),
        (f"{auto_cert_name} {ignore_term}", "{auto_cert_name} {ignore_term}"),
        (f"{feature} {ignore_term}", "{feature} {ignore_term}"),
        (f"{brand} {ignore_term}", "{brand} {ignore_term}"),
        (f"{polymer} {ignore_term}", "{polymer} {ignore_term}"),

        (f"{ignore_term} {oem_name}", "{ignore_term} {oem_name}"),
        (f"{ignore_term} {auto_cert_name}", "{ignore_term} {auto_cert_name}"),
        (f"{ignore_term} {auto_cert_name}", "{ignore_term} {auto_cert_name}"),
        (f"{ignore_term} {feature}", "{ignore_term} {feature}"),
        (f"{ignore_term} {brand}", "{ignore_term} {brand}"),
        (f"{ignore_term} {polymer}", "{ignore_term} {polymer}"),

        (f"{oem_name} and {ignore_term}", "{oem_name} and {ignore_term}"),
        (f"{auto_cert_name} and {ignore_term}", "{auto_cert_name} and {ignore_term}"),
        (f"{auto_cert_name} and {ignore_term}", "{auto_cert_name} and {ignore_term}"),
        (f"{feature} and {ignore_term}", "{feature} and {ignore_term}"),
        (f"{brand} and {ignore_term}", "{brand} and {ignore_term}"),
        (f"{polymer} and {ignore_term}", "{polymer} and {ignore_term}"),

        (f"{ignore_term} and {oem_name}", "{ignore_term} and {oem_name}"),
        (f"{ignore_term} and {auto_cert_name}", "{ignore_term} and {auto_cert_name}"),
        (f"{ignore_term} and {auto_cert_name}", "{ignore_term} and {auto_cert_name}"),
        (f"{ignore_term} and {feature}", "{ignore_term} and {feature}"),
        (f"{ignore_term} and {brand}", "{ignore_term} and {brand}"),
        (f"{ignore_term} and {polymer}", "{ignore_term} and {polymer}"),


        (f"{oem_name}, {ignore_term} and {ignore_term2}", "{oem_name}, {ignore_term} and {ignore_term2}"),
        (f"{auto_cert_name}, {ignore_term} and {ignore_term2}", "{auto_cert_name}, {ignore_term} and {ignore_term2}"),
        (f"{auto_cert_name}, {ignore_term} and {ignore_term2}", "{auto_cert_name}, {ignore_term} and {ignore_term2}"),
        (f"{feature}, {ignore_term} and {ignore_term2}", "{feature}, {ignore_term} and {ignore_term2}"),
        (f"{brand}, {ignore_term} and {ignore_term2}", "{brand}, {ignore_term} and {ignore_term2}"),
        (f"{polymer}, {ignore_term} and {ignore_term2}", "{polymer}, {ignore_term} and {ignore_term2}"),

        (f"{ignore_term}, {ignore_term2} and {oem_name}", "{ignore_term}, {ignore_term2} and {oem_name}"),
        (f"{ignore_term}, {ignore_term2} and {auto_cert_name}", "{ignore_term}, {ignore_term2} and {auto_cert_name}"),
        (f"{ignore_term}, {ignore_term2} and {auto_cert_name}", "{ignore_term}, {ignore_term2} and {auto_cert_name}"),
        (f"{ignore_term}, {ignore_term2} and {feature}", "{ignore_term}, {ignore_term2} and {feature}"),
        (f"{ignore_term}, {ignore_term2} and {brand}", "{ignore_term}, {ignore_term2} and {brand}"),
        (f"{ignore_term}, {ignore_term2} and {polymer}", "{ignore_term}, {ignore_term2} and {polymer}"),

        (f"{filler} materials", "{filler} materials"),
        (f"{filler} grades", "{filler} grades"),
        (f"grades with {filler}", "grades with {filler}"),
        (f"materials with {filler}", "materials with {filler}"),
        (f"grades that have {property}", "grades that have {property}"),
        (f"materials that have {property}", "materials that have {property}"),
        (f"grades that have {ul_property}", "grades that have {ul_property}"),
        (f"materials that have {ul_property}", "materials that have {ul_property}"),
        (f"grades that have {ul_sub_property}", "grades that have {ul_sub_property}"),
        (f"materials that have {ul_sub_property}", "materials that have {ul_sub_property}"),

    ]

    output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [],  'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
    random_query = random.choice(templates)
    query = random_query[0]

    if "{processing}" in random_query[1]:
        if processing in processing_syn_mapping:
            processing = processing_syn_mapping[processing]
            
        output['PROCESSING'].append(processing)
        if processing_count < process_th:
            processing_count += 1
        else:
            continue
    
    if "{processing2}" in random_query[1]:
        if processing2 in processing_syn_mapping:
            processing2 = processing_syn_mapping[processing2]
            
        output['PROCESSING'].append(processing2)
        if processing_count < process_th:
            processing_count += 1
        else:
            continue

    if "{feature}" in random_query[1]:
        if is_feature_spell_correct and corrected_feature:
            print(feature, random_query)
            feature = corrected_feature
        
        if feature in feature_syn_mapping:
            feature = feature_syn_mapping[feature]
            
        output['FEATURE'].append(feature)
        if feature_count < feat_th:
            feature_count += 1
        else:
            continue

    if "{feature2}" in random_query[1]:
        if feature2 in feature_syn_mapping:
            feature2 = feature_syn_mapping[feature2]
            
        output['FEATURE'].append(feature2)
        if feature_count < feat_th:
            feature_count += 1
        else:
            continue

    if "{delivery_form}" in random_query[1]:
        if delivery_form in delivery_syn_mapping:
            delivery_form = delivery_syn_mapping[delivery_form]
            
        output['DELIVERY_FORM'].append(delivery_form)
        if del_count < del_th:
            del_count += 1
        else:
            continue

    if "{delivery_form2}" in random_query[1]:
        if delivery_form2 in delivery_syn_mapping:
            delivery_form2 = delivery_syn_mapping[delivery_form2]
            
        output['DELIVERY_FORM'].append(delivery_form2)
        if del_count < del_th:
            del_count += 1
        else:
            continue

    if "{brand}" in random_query[1]:
        if brand in brand_syn_mapping:
            brand = brand_syn_mapping[brand]
            
        output['BRAND'].append(brand)
        if brand_count < brand_th:
            brand_count += 1
        else:
            continue

    if "{brand2}" in random_query[1]:
        if brand2 in brand_syn_mapping:
            brand2 = brand_syn_mapping[brand2]
            
        output['BRAND'].append(brand2)
        if brand_count < brand_th:
            brand_count += 1
        else:
            continue

    if "{polymer}" in random_query[1]:
        if polymer in polymer_syn_mapping:
            polymer = polymer_syn_mapping[polymer]
            
        output['POLYMER'].append(polymer)
        if polymer_count < poly_th:
            polymer_count += 1
        else:
            continue

    if "{polymer2}" in random_query[1]:
        if polymer2 in polymer_syn_mapping:
            polymer2 = polymer_syn_mapping[polymer2]
            
        output['POLYMER'].append(polymer2)
        if polymer_count < poly_th:
            polymer_count += 1
        else:
            continue

    if "{application}" in random_query[1]:
        if "oem_name" in random_query[1] or 'auto_cert_name' in random_query[1]:
            if industry == 'automotive & transportation':
                 print("#"*25, '\n', random_query[0], '\n', "#"*25)
            else:
                continue
            
        output['APPLICATION'].append(application)
        if industry != 'not mapped':
            output['INDUSTRY'].append(industry)
        if app_count < app_th:
            app_count += 1
        else:
            continue

    if "{continent}" in random_query[1]:
        output['REGION'].append(continent_region)
        if region_count < region_th:
            region_count += 1
        else:
            continue

    if "{country}" in random_query[1]:
        output['REGION'].append(country_region)
        if region_count < region_th:
            region_count += 1
        else:
            continue

    if "{state}" in random_query[1]:
        output['REGION'].append(state_region)
        if region_count < region_th:
            region_count += 1
        else:
            continue

    if "{city}" in random_query[1]:
        output['REGION'].append(city_region)
        if region_count < region_th:
            region_count += 1
        else:
            continue

    if "{grade}" in random_query[1]:
        output['GRADE'].append(grade)

    if "{competitor_grade}" in random_query[1]:
        output['COMPETITOR_GRADE'].append(competitor_grade)

    if "{nsf}" in random_query[1]:
        output['NSF_CERT'].append(nsf)
        if nsf_count < nsf_th:
            nsf_count += 1
            print("nsf: ", nsf_count)
        else:
            continue

    if "{ignore_term}" in random_query[1]:
        # here we are considering count based on query
        if ignore_terms_count < ignore_terms_th:
            ignore_terms_count += 1
        else:
            continue

    if "{water_name}" in random_query[1]:
        output['WATER_CERT'].append({"Standard": water[1].lower(), "Temp": [water[2].lower()]})
        if water_count < water_th:
            water_count += 1
            print("water: ", water_count)
        else:
            continue

    if "{railway_name}" in random_query[1]:
        output['RAILWAY_CERT'].append({"Standard": railway[3].lower(), "Hazard_Level": [railway[2].lower()], "Req_Set": [railway[1].lower()]})
        if railway_count < rail_th:
            railway_count += 1
        else:
            continue

    if "{railway_name2}" in random_query[1]:
        output['RAILWAY_CERT'][0]['Hazard_Level'].append(railway2[2].lower())
        output['RAILWAY_CERT'][0]['Req_Set'].append(railway2[1].lower())
        if railway_count < rail_th:
            railway_count += 1
        else:
            continue

    if  "{auto_cert_name}" in random_query[1]:
        output['AUTO_CERT'].append({'OEM': auto_cert[0], 'CERTS': [auto_cert_name]})
        if auto_cert_count < auto_th:
            auto_cert_count += 1
        else:
            continue
    
    if "{auto_cert_name2}" in random_query[1]:
        if auto_cert[0].lower() == auto_cert2[0].lower():
            output['AUTO_CERT'][0]['CERTS'].append(auto_cert_name2)
        else:
            output['AUTO_CERT'].append({'OEM': auto_cert2[0], 'CERTS': [auto_cert_name2]})
            
        if auto_cert_count < auto_th:
            auto_cert_count += 1
        else:
            continue
        
    if "{oem_name}" in random_query[1] and not "{auto_cert_name}" in random_query[1]:
        output['AUTO_CERT'].append({'OEM': auto_cert[0].lower(), 'CERTS': ['all']})
        if auto_cert_count < auto_th:
            auto_cert_count += 1
        else:
            continue

    if "{oem_name2}" in random_query[1] and not "{auto_cert_name2}" in random_query[1]:
        if auto_cert[0].lower() == auto_cert2[0].lower():
            continue
            
        output['AUTO_CERT'].append({'OEM': auto_cert2[0].lower(), 'CERTS': ['all']})
        if auto_cert_count < auto_th:
            auto_cert_count += 1
        else:
            continue

    if "{property}" in random_query[1]:
        output['PROPERTY'].append(property_details)
        if prop_count < prop_th:
            prop_count += 1
        else:
            continue

    if "{property2}" in random_query[1]:
        output['PROPERTY'].append(property2_details)
        if prop_count < prop_th:
            prop_count += 1
        else:
            continue

    if "{property_abb}" in random_query[1]:
        output['PROPERTY'].append(property_abb_details)
        if prop_abb_count < prop_abb_th:
            prop_abb_count += 1
        else:
            continue

    if "{property_abb2}" in random_query[1]:
        output['PROPERTY'].append(property_abb2_details)
        if prop_abb_count < prop_abb_th:
            prop_abb_count += 1
        else:
            continue

    if "{ul_property}" in random_query[1]:
        output['PROPERTY'].append(ul_property_details)
        if ul_prop_count < ul_prop_th:
            ul_prop_count += 1
        else:
            continue

    if "{ul_property2}" in random_query[1]:
        output['PROPERTY'].append(ul_property2_details)
        if ul_prop_count < ul_prop_th:
            ul_prop_count += 1
        else:
            continue
      
    # list of dicts
    if "{filler}" in random_query[1]:
        output['FILLER'].extend(filler_details)
        if filler_count < fill_th:
            filler_count += 1
        else:
            continue

    if "{property_rm}" in random_query[1]:
        output['PROPERTY'].extend(property_rm_details)
        if prop_range_count < prop_rm_th:
            prop_range_count += 1
        else:
            continue

    if "{property_rm2}" in random_query[1]:
        output['PROPERTY'].extend(property_rm2_details)
        if prop_range_count < prop_rm_th:
            prop_range_count += 1
        else:
            continue

    if "{ul_sub_property}" in random_query[1]:
        output['PROPERTY'].extend(ul_sub_property_details)
        if ul_sub_prop_count < ul_sub_prop_th:
            ul_sub_prop_count += 1
        else:
            continue

    # if "{ul_sub_property2}" in random_query[1]:
    #     output['PROPERTY'].extend(ul_sub_property2_details)
    #     if ul_sub_prop_count < ul_sub_prop_th:
    #         ul_sub_prop_count += 1
    #     else:
    #         continue

    row = {'Query': query,'Output': output} 
    if row not in generated_data:
        generated_data.append(row)
            
    
# generated_data[:50]

C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\1632742568.py:10: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(i[1],i[2])
C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\2640670372.py:21: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(p_metadata[1],p_metadata[2])
C:\Users\DSCAK2\AppData\Local\Temp\ipykernel_6740\164086352.py:13: DeprecationWarning: randrange() will raise TypeError in the future
  value = random.randrange(p_metadata[1],p_metadata[2])


$$$$$$$$$$$$$$$$$$$$ value was 0.0 for property range 1.3e-07 to 3.8e-07, picking another random number: 2.16e-07
************************* negative exponential
{'property_name': 'effective thermal diffusivity iso 22007-4 through plane (m²/s)', 'modifier': {'value': '2.16e-07', 'min': '1.94e-07', 'max': '2.38e-07', 'unit': 'm²/s'}, 'property_type': 'property'}
$$$$$$$$$$$$$$$$$$$$ value was 0.0 for property range 1.3e-07 to 3.8e-07, picking another random number: 1.76e-07
************************* negative exponential
{'property_name': 'effective thermal diffusivity', 'modifier': {'value': '1.76e-07', 'min': '1.58e-07', 'max': '1.94e-07', 'unit': 'm²/s'}, 'property_type': 'property'}
water:  1
nsf:  1
nsf:  2
$$$$$$$$$$$$$$$$$$$$ value was 0.0 for property range 2.2e-07 to 1.08e-06, picking another random number: 9.63e-07
nsf:  3
nsf:  4
mpact resistant ('mpact resistant flow233.41 semi aromatic pa compression moldable', '{feature} {property_abb} {polymer} {processing}')
nsf:  5
contai

In [222]:
templates

[('r 24 h l2', '{railway_name}'),
 ('r-24 h-l2', '{railway_name}'),
 ('r24 hl2', '{railway_name}'),
 ('r24 hl2', '{railway_name}'),
 ('r24 hl2', '{railway_name}'),
 ('r24hl2', '{railway_name}'),
 ('r24 hl2', '{railway_name}'),
 ('r24 hl2 r22 - hl3 (en 45545-2)', '{railway_name} {railway_name2}'),
 ('r24 hl2 and r22 - hl3 (en 45545-2)', '{railway_name} and {railway_name2}'),
 ('r24 hl2 or r22 - hl3 (en 45545-2)', '{railway_name} or {railway_name2}'),
 ('r24 hl2, r22 - hl3 (en 45545-2)', '{railway_name}, {railway_name2}'),
 ('gmw15812p-tpv(epdm+pp)-type 8m', '{auto_cert_name}'),
 ('gmw15812p-tpv(epdm+pp)-type 8m approval', '{auto_cert_name} approval'),
 ('gmw15812p-tpv(epdm+pp)-type 8m auto spec', '{auto_cert_name} auto spec'),
 ('gmw15812p-tpv(epdm+pp)-type 8m auto approval',
  '{auto_cert_name} auto approval'),
 ('gmw15812p-tpv(epdm+pp)-type 8m', '{auto_cert_name}'),
 ('gmw15812p-tpv(epdm+pp)-type 8m approval', '{auto_cert_name} approval'),
 ('gmw15812p-tpv(epdm+pp)-type 8m auto spec',

## Generated Data Value Count

In [223]:
print("total queries", len(generated_data))

counts = {
    'feature_count': feature_count, 'processing_count': processing_count, 'del_count': del_count, 'brand_count': brand_count, 
    'polymer_count': polymer_count, 'app_count': app_count, 'nsf_count': nsf_count, 'water_count': water_count, 'railway_count': railway_count,
    'filler_count': filler_count, 'ul_prop_count': ul_prop_count, 'ul_sub_prop_count': ul_sub_prop_count, 'prop_abb_count': prop_abb_count, 
    'prop_range_count': prop_range_count, 'prop_count': prop_count, 'auto_cert_count': auto_cert_count,
}
counts

total queries 18763


{'feature_count': 6500,
 'processing_count': 3500,
 'del_count': 2000,
 'brand_count': 6000,
 'polymer_count': 6500,
 'app_count': 3000,
 'nsf_count': 1000,
 'water_count': 1000,
 'railway_count': 1500,
 'filler_count': 2500,
 'ul_prop_count': 1500,
 'ul_sub_prop_count': 1400,
 'prop_abb_count': 1000,
 'prop_range_count': 1000,
 'prop_count': 2000,
 'auto_cert_count': 2500}

In [224]:
gen_df = pd.DataFrame(generated_data)
cols = ["Sl. No."] + list(gen_df)
gen_df["Sl. No."] = list(range(1, len(gen_df)+1))
gen_df = gen_df[cols]
gen_df

,Sl. No.,Query,Output
0,1,dtul1.8mpa=high lubricated,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
1,2,"blueridge, impet, total load of 94","{'GRADE': [], 'APPLICATION': [], 'BRAND': ['bl..."
2,3,los angeles,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
3,4,high current arc ignition (hai): plc3,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
4,5,looking for a pa 666 full load upto 39 with ex...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
...,...,...,...
18758,18759,i want an offset to durethan bkv50h2.0ef 90151...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
18759,18760,i want an offset to tarolox200g3 for slovakia,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
18760,18761,polyshine pbt d223 gf15 fr emea,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
18761,18762,pbt2000-201d asia,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."


In [225]:
# gen_df.to_excel("./prod_generated_data_29_11_24.xlsx", header=True, index=False)
# gen_df.to_excel("./prod_generated_data_27_01_25.xlsx", header=True, index=False)
gen_df.to_excel(f"./prod_generated_data_{data_version}.xlsx", header=True, index=False)

## Additional Generated Data

### Data for Queries with Multiple UL Properties

In [226]:
multi_ul_sub_props = {'Query': [], 'Output': []}

for i in range(1, 50):
    p_details, query = get_ul_sub_properties()
    output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': p_details, 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
    multi_ul_sub_props['Query'].append(query)
    multi_ul_sub_props['Output'].append(output)
    
multi_ul_sub_props_df = pd.DataFrame(multi_ul_sub_props)

cols = ["Sl. No."] + list(multi_ul_sub_props_df)
multi_ul_sub_props_df["Sl. No."] = list(range(1, len(multi_ul_sub_props_df)+1))
multi_ul_sub_props_df = multi_ul_sub_props_df[cols]
multi_ul_sub_props_df

got same properties
got same properties
got same properties
got same properties
got same properties
got same properties
got same properties


,Sl. No.,Query,Output
0,1,flame rating at 3.1mm v0 and glow wire temp of...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
1,2,749°c glow wire ignition temperature @ 0.8mm a...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
2,3,"flammabilityclassification of v 0, gwi temp 3m...","{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
3,4,relative thermal index - mechanical impact (rt...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
4,5,"hotwireignition of plc 0, high-current arc ign...","{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
5,6,relative thermal index ele (145°c) around 15% ...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
6,7,"high ampere arc ignition plc1, flame class rat...","{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
7,8,flamerate v 1 and relative thermal index - mec...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
8,9,rti-e maximum 149°c and glow-wire flammability...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
9,10,glow wire: 569.76 °c plc 1 (60 - 119) high-cur...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."


In [227]:
# multi_ul_sub_props_df.to_excel("./multi_ul_sub_props_29_11_24.xlsx", header=True, index=False)
# multi_ul_sub_props_df.to_excel("./multi_ul_sub_props_27_01_25.xlsx", header=True, index=False)
multi_ul_sub_props_df.to_excel(f"./multi_ul_sub_props_{data_version}.xlsx", header=True, index=False)

### Data for queries with only Certification

In [228]:
auto_certifications_data = {'Query': [], 'Output': []}

for i in auto_certifications:
    query = random.choice([
        f"{i[0]} {i[1]}",
        f"{i[0]} {i[1]}",
        f"{i[0]} - {i[1]}",
        f"{i[0]} - {i[1]}",
        f"{i[0]} {i[1]} certification",
        f"{i[0]} {i[1]} auto approval",

        f"{i[1]} {i[0]}",

        f"{i[0]} {i[1]}",
        f"{i[0]} {i[1]}",
        f"{i[0]} - {i[1]}",
        f"{i[0]} - {i[1]}",
        f"{i[0]} {i[1]} certification",
        f"{i[0]} {i[1]} auto approval",

        f"{i[1]} {i[0]}",
        
        f"{i[0]} certifcation {i[1]} approval",
        f"{i[0]} auto approval {i[1]}",
        f"{i[0]} auto approval {i[1]} certifcation",

        f"{i[0]} certifcation {i[1]} approved",
        f"{i[0]} auto approved {i[1]}",
        f"{i[0]} auto approved {i[1]} certifcation",
        f"{i[0]} {i[1]} auto approved",

    ])
    output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [{'OEM': i[0], 'CERTS': [i[1]]}], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
    auto_certifications_data['Query'].append(query)
    auto_certifications_data['Output'].append(output)
    
auto_certifications_data_df = pd.DataFrame(auto_certifications_data)

cols = ["Sl. No."] + list(auto_certifications_data_df)
auto_certifications_data_df["Sl. No."] = list(range(1, len(auto_certifications_data_df)+1))
auto_certifications_data_df = auto_certifications_data_df[cols]
auto_certifications_data_df

,Sl. No.,Query,Output
0,1,baic q-bjev 01.59 certification,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
1,2,baic - q-bjev 01.33,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
2,3,baic bas-491,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
3,4,baic bas-492,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
4,5,baic - q-bjev 01.59,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
...,...,...,...
980,981,li auto q-lia5310057 certification,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
981,982,geely q/jly j7111001a-2016,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
982,983,geely q/jly j7110235b-2018 auto approval,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
983,984,geely auto approved q/jly j7111001a-2016 certi...,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."


In [229]:
# auto_certifications_data_df.to_excel("./auto_certifications_data_29_11_24.xlsx", header=True, index=False)
# auto_certifications_data_df.to_excel("./auto_certifications_data_27_01_25.xlsx", header=True, index=False)
auto_certifications_data_df.to_excel(f"./auto_certifications_data_{data_version}.xlsx", header=True, index=False)

### Data for Queries with only Grade / Competitor Grade

In [230]:
len(all_cgrades)

12257

In [231]:
len(filtered_gradenames)

4934

In [ ]:
filtered_gradenames[:25]

In [233]:
grade_cgrade_data = {'Query': [], 'Output': []}
for i in filtered_gradenames:
    i = i.replace('®', '').replace('™', '').strip()
    i = re.sub(r'^\((.*)\)$', r'\1', i)
    for suffix in remove_suffixes:
        i = re.sub(r'{}$'.format(re.escape(suffix)), '', i)

    i = i.strip()
    is_grade_without_space = random.choice([False, False, False, False, False, False, True])
    if is_grade_without_space:
        i = i.replace(' ', '')
        
    query = random.choice([
        f"{i} grade",
        f"looking for {i}",
        f"{i} material",
        f"{i}",
    ])
    output = {'GRADE': [i], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
    grade_cgrade_data['Query'].append(query)
    grade_cgrade_data['Output'].append(output)

for i in all_cgrades:
    i = i.replace('®', '').replace('™', '').strip()
    i = re.sub(r'^\((.*)\)$', r'\1', i)
    for suffix in remove_suffixes:
        i = re.sub(r'{}$'.format(re.escape(suffix)), '', i)
        
    i = re.sub(r'{}$'.format(re.escape('')), '', i) 
    i = i.strip()
    is_comp_grade_without_space = random.choice([False, False, False, False, False, False, True])
    if is_comp_grade_without_space:
        i = i.replace(' ', '')
    query = random.choice([
        f"alternative for {i}",
        f"alternative to {i}",
        f"offset for {i}",
        f"offset to {i}",
        f"{i} offset",
        f"{i}",
    ])
    output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [i], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
    grade_cgrade_data['Query'].append(query)
    grade_cgrade_data['Output'].append(output)

    
grade_cgrade_data_df = pd.DataFrame(grade_cgrade_data)

cols = ["Sl. No."] + list(grade_cgrade_data_df)
grade_cgrade_data_df["Sl. No."] = list(range(1, len(grade_cgrade_data_df)+1))
grade_cgrade_data_df = grade_cgrade_data_df[cols]
grade_cgrade_data_df

,Sl. No.,Query,Output
0,1,hostaform xgc15-lw01 xap material,"{'GRADE': ['hostaform xgc15-lw01 xap'], 'APPLI..."
1,2,looking for zenite 7140x,"{'GRADE': ['zenite 7140x'], 'APPLICATION': [],..."
2,3,lftr tpu-gf60-01-x,"{'GRADE': ['lftr tpu-gf60-01-x'], 'APPLICATION..."
3,4,kepex 3730gf,"{'GRADE': ['kepex 3730gf'], 'APPLICATION': [],..."
4,5,looking for hostaform c 9021 s1,"{'GRADE': ['hostaform c 9021 s1'], 'APPLICATIO..."
...,...,...,...
17186,17187,ultrasint pa6 fr black offset,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
17187,17188,alternative for epimix pbt nc q101,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
17188,17189,offset to tarolox gfr 2,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
17189,17190,nylene mach 6 offset,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."


In [234]:
# grade_cgrade_data_df.to_excel("./grade_cgrade_data_29_11_24.xlsx", header=True, index=False)
# grade_cgrade_data_df.to_excel("./grade_cgrade_data_27_01_25.xlsx", header=True, index=False)
grade_cgrade_data_df.to_excel(f"./grade_cgrade_data_{data_version}.xlsx", header=True, index=False)

### Data for Queries with only Certification Nubmber (to avoid any confusion with grade/cgrade numbers)

In [235]:
certs_with_numbers = {
    'bmw': ['gs93016', '93016'],
    'bosch': ['gf019', 'ox067', 'bn02-gf034', 'bn05-ox019'],
    'catl': ['gf25-a-i01', 'gf30-x-i0x'],
    'ford': ['m4d639', 'm4d1016-a1', 'm4d861-a4', 'm4d1014'],
    'geely': ['p1 0370', '0367', '0369'],
    'general motors (gm)': ['17025', '17327p', '17327', '15702-250051', '250051', '15702-250058', '250058', 'pet.002', '15702-120032', '120032', '16270p', '16270', '3038p', '15702-110080', '18066', '17961', '110052'],
    'honda': ['0094z', 'ghaf-9810-m1', '9810-m1', '9810'],
    'hyundai': ['216-03', '94103', '220-24', '220-08 type a', '211-72'],
    'mercedes-benz': ['5403', '5562', '5562 aa39', '5403.21', '5408.45'],
    'nio': ['sm.51.010', 'sm.51', '51.010', '51010', '51.003', '51.010-c4'],
    'opel (psa)': ['000637', '006611', '0006612', '0006615'],
    'stellantis - chrysl': ['50103', 'db-448', '50017'],
    'stellantis-psa group': ['0300', '620300', '62 0300'],
    'tesla': ['1006', '102160', '1006 102160'],
    'vw group': ['50136', '52683', '50127', '50125'],
    'valeo': ['15009', '15006'],
}

In [236]:
certs_with_numbers_data = {'Query': [], 'Output': []}

for oem in certs_with_numbers:
    for cert in certs_with_numbers[oem]:
        query = random.choice([
            f"{cert} certified",
            f"{cert} certification",
            f"{cert} certifications",
            f"grade with {cert} certified",
            f"material with {cert} certified",
            f"{cert} certified grade",
            f"{cert} certified material",
        ])
        output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [{'OEM': oem, 'CERTS': [cert]}], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
        certs_with_numbers_data['Query'].append(query)
        certs_with_numbers_data['Output'].append(output)
        certs_with_numbers_data['Query'].append(f"{cert}")
        certs_with_numbers_data['Output'].append(output)

gen_auto_num_df = pd.DataFrame(certs_with_numbers_data)
cols = ["Sl. No."] + list(gen_auto_num_df)
gen_auto_num_df["Sl. No."] = list(range(1, len(gen_auto_num_df)+1))
gen_auto_num_df = gen_auto_num_df[cols]
gen_auto_num_df

,Sl. No.,Query,Output
0,1,grade with gs93016 certified,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
1,2,gs93016,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
2,3,grade with 93016 certified,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
3,4,93016,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
4,5,material with gf019 certified,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
...,...,...,...
137,138,50125,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
138,139,15009 certification,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
139,140,15009,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."
140,141,15006 certifications,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ..."


In [237]:
# gen_auto_num_df.to_excel("./prod_generated_data_auto_num_29_11_24.xlsx", header=True, index=False)
# gen_auto_num_df.to_excel("./prod_generated_data_auto_num_27_01_25.xlsx", header=True, index=False)
gen_auto_num_df.to_excel(f"./prod_generated_data_auto_num_{data_version}.xlsx", header=True, index=False)

### Data for "Melting temperature of Celcon m90?" type of queries

In [238]:
prop_grade_data = {'Query': [], 'Output': []}
for i in range(50):
    g = random.choice(filtered_gradenames)
    g = g.replace('®', '').replace('™', '').strip()
    g = re.sub(r'^\((.*)\)$', r'\1', g)
    for suffix in remove_suffixes:
        g = re.sub(r'{}$'.format(re.escape(suffix)), '', g)

    g = g.strip()
    is_grade_without_space = random.choice([False, False, False, False, False, False, True])
    if is_grade_without_space:
        g = g.replace(' ', '')

    p_metadata = random.choice(property_and_values)
    property_name = p_metadata[-1].lower()
    property_name_meaning = p_metadata[0].lower()
    property_details = {'property_name': property_name_meaning, 'modifier': {'value': None, 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}

    query = random.choice([
        f"{property_name} of {g}",
        f"{property_name} of {g}",
        f"{property_name} of {g}",
        f"{property_name} of {g}?",
        f"{property_name} value of {g}?",
        f"what is the {property_name} of {g}?",
        f"what is the {property_name} value of {g}?",
    ])
    output = {'GRADE': [g], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [property_details], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
    prop_grade_data['Query'].append(query)
    prop_grade_data['Output'].append(output)

gen_prop_grade_df = pd.DataFrame(prop_grade_data)
cols = ["Sl. No."] + list(gen_prop_grade_df)
gen_prop_grade_df["Sl. No."] = list(range(1, len(gen_prop_grade_df)+1))
gen_prop_grade_df = gen_prop_grade_df[cols]
gen_prop_grade_df

,Sl. No.,Query,Output
0,1,"tangent delta, 1mhz of frianyl a3 rv0 or 2011/...","{'GRADE': ['frianyl a3 rv0 or 2011/p(s)'], 'AP..."
1,2,what is the strain at break 50 mm/min of a3 rv...,"{'GRADE': ['a3 rv0 bk 9005/aa'], 'APPLICATION'..."
2,3,what is the elongation at break 50 mm/min valu...,"{'GRADE': ['celcon uv270z'], 'APPLICATION': []..."
3,4,dissipation factor of htn51g45hslr bk420,"{'GRADE': ['htn51g45hslr bk420'], 'APPLICATION..."
4,5,what is the melt index of hf xt 90?,"{'GRADE': ['hf xt 90'], 'APPLICATION': [], 'BR..."
5,6,what is the yield strain 50 mm/min value of cr...,"{'GRADE': ['crastin t803 nc010'], 'APPLICATION..."
6,7,what is the nominal break elongation value of ...,"{'GRADE': ['lft pa66-cf40-01-us'], 'APPLICATIO..."
7,8,surface resistivity of lft pbt-gf40-09?,"{'GRADE': ['lft pbt-gf40-09'], 'APPLICATION': ..."
8,9,what is the izod impact strength unnotch value...,"{'GRADE': ['fortronfx4330t7'], 'APPLICATION': ..."
9,10,what is the break elongation 5mm/min value of ...,"{'GRADE': ['va 9116'], 'APPLICATION': [], 'BRA..."


In [239]:
# gen_prop_grade_df.to_excel("./prod_generated_data_prop_grade_29_11_24.xlsx", header=True, index=False)
# gen_prop_grade_df.to_excel("./prod_generated_data_prop_grade_27_11_25.xlsx", header=True, index=False)
gen_prop_grade_df.to_excel(f"./prod_generated_data_prop_grade_{data_version}.xlsx", header=True, index=False)

## Enhancement Data (enhancement_queries.xlsx)

### Data for Polymer-Total Load ("POM 15%") and Brand-Total Load ("Celcon 15%")

In [240]:
# poly_brand_total_load_data = {'Query': [], 'Output': []}
# for i in range(30):
#     tl = random.choice(list(range(10, 61)))
#     brand = random.choice(all_brands).lower()       
        
#     is_brand_syn = random.choice([False, False, True])
#     if is_brand_syn and brand in brand_syns:
#         brand = random.choice(brand_syns[brand])

#     query = random.choice([
#         f"{brand} {tl}%",
#         f"{brand} {tl}%",
#         f"{brand}{tl}%",
#         f"{tl}% {brand}",
#         f"{tl}% {brand}",
#         f"{tl}%{brand}",
#     ])

#     if brand in brand_syn_mapping:
#         brand_meaning = brand_syn_mapping[brand]
#     else:
#         brand_meaning = brand
        
#     output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}        
#     output['BRAND'].append(brand_meaning)
#     output['FILLER'].append({'total_load': {'value': tl, 'min': tl-5, 'max': tl+5}})
#     poly_brand_total_load_data['Query'].append(query)
#     poly_brand_total_load_data['Output'].append(output)

# for i in range(30):
#     tl = random.choice(list(range(10, 61)))
#     polymer = random.choice(all_polymers).lower()  
    
#     is_polymer_syn = random.choice([False, True, True])
#     if is_polymer_syn and polymer in polymer_syns:
#         polymer = random.choice(polymer_syns[polymer])
    
#     query = random.choice([
#         f"{polymer} {tl}%",
#         f"{polymer} {tl}%",
#         f"{polymer}{tl}%",
#         f"{tl}% {polymer}",
#         f"{tl}% {polymer}",
#         f"{tl}%{polymer}",
#     ])

#     if polymer in polymer_syn_mapping:
#         polymer_meaning = polymer_syn_mapping[polymer]
#     else:
#         polymer_meaning = polymer
            
#     output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}        
#     output['POLYMER'].append(polymer_meaning)
#     output['FILLER'].append({'total_load': {'value': tl, 'min': tl-5, 'max': tl+5}})
#     poly_brand_total_load_data['Query'].append(query)
#     poly_brand_total_load_data['Output'].append(output)

# gen_poly_brand_total_load_df = pd.DataFrame(poly_brand_total_load_data)
# cols = ["Sl. No."] + list(gen_poly_brand_total_load_df)
# gen_poly_brand_total_load_df["Sl. No."] = list(range(1, len(gen_poly_brand_total_load_df)+1))
# gen_poly_brand_total_load_df = gen_poly_brand_total_load_df[cols]
# gen_poly_brand_total_load_df

In [241]:
## gen_poly_brand_total_load_df.to_excel("./prod_generated_data_poly_brand_tl_29_11_24.xlsx", header=True, index=False)

### Data for the properties that has common name

In [242]:
# ambiguity_prop_syns_mapping = {
#     'charpy': 'charpy',
#     'izod': 'izod',
#     # 'flammability': 'flammability',
#     'impact': 'impact strength',
#     'notched impact': 'notched impact',
#     'shore': 'shore',
#     'shore hardness': 'shore',
#     'strain at break': 'strain at break',
#     'strength': 'tensile stress',
#     'stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
#     'stress at break': 'stress at break',
#     'temperature': 'temperature',
#     'tensile': 'tensile',
#     'tensile strength': 'tensile strength',
#     'tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
#     'tensile stress at break': 'stress at break',
#     'viscosity': 'viscosity',
# }

In [243]:
# prop_ambiguity_data = {'Query': [], 'Output': []}

# for i in range(5):
#     for syn in ambiguity_prop_syns_mapping:
#         v = random.randint(10, 200)
#         query = random.choice([
#             f"{syn} of {v}",
#             f"{syn} {v}",
#             f"{v} {syn}",
#             f"{syn} = {v}",
#             f"{v} = {syn}",
#             f"{syn}: {v}",
    
#         ])
#         output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}         
#         output['PROPERTY'].append({'property_name': ambiguity_prop_syns_mapping[syn], 'modifier': {'value': v, 'min': v*0.9, 'max': v*1.1, 'unit': ''}, 'property_type': 'property'})
#         prop_ambiguity_data['Query'].append(query)
#         prop_ambiguity_data['Output'].append(output)


# gen_prop_ambiguity_df = pd.DataFrame(prop_ambiguity_data)
# cols = ["Sl. No."] + list(gen_prop_ambiguity_df)
# gen_prop_ambiguity_df["Sl. No."] = list(range(1, len(gen_prop_ambiguity_df)+1))
# gen_prop_ambiguity_df = gen_prop_ambiguity_df[cols]
# gen_prop_ambiguity_df

In [244]:
## gen_prop_ambiguity_df.to_excel("./prod_generated_data_prop_ambiguity_29_11_24.xlsx", header=True, index=False)

### Data for Region

In [245]:
# addtional_region_data = {'asia pacific': ['indian',
#   'australian',
#   'japanese',
#   'russian',
#   'chinese',
#   'korean',
#   'thai',
#   'vietnamese',
#   'indonesian',
#   'malaysian',
#   'philippine',
#   'new zealander'],
#  'europe middle east africa': ['italian',
#   'french',
#   'german',
#   'south african',
#   'spanish',
#   'british',
#   'egyptian',
#   'swedish',
#   'norwegian',
#   'finnish',
#   'danish',
#   'polish',
#   'dutch',
#   'belgian',
#   'swiss',
#   'austrian'],
#  'americas': ['brazilian',
#   'canadian',
#   'mexican',
#   'argentinian',
#   'chilean',
#   'peruvian',
#   'colombian',
#   'venezuelan',
#   'uruguayan',
#   'cuban',
#   'jamaican']}

In [246]:
# region_data = {'Query': [], 'Output': []}

# for i in addtional_region_data:
#     for j in addtional_region_data[i]:
#         query = random.choice([
#             f"{j} grades",
#             f"{j} materials",
#             f"{j}",
#         ])
#         output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': [i]}
#         region_data['Query'].append(query)
#         region_data['Output'].append(output)


# for i in addtional_region_data:
#     for j in addtional_region_data[i]:
#         query = random.choice([
#             f"{j} grades",
#             f"{j} materials",
#             f"{j}",
#         ])
#         output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': [i]}
#         region_data['Query'].append(query)
#         region_data['Output'].append(output)

    
# region_data_df = pd.DataFrame(region_data)

# cols = ["Sl. No."] + list(region_data_df)
# region_data_df["Sl. No."] = list(range(1, len(region_data_df)+1))
# region_data_df = region_data_df[cols]
# region_data_df

In [247]:
## region_data_df.to_excel("./region_queries_29_11_24.xlsx", header=True, index=False)

### Data for "NB" value properties

In [248]:
# nb_properties = []
# for p_details in unique_values['Property']:
#     if p_details['String_value_for_toggle']:
#         nb_properties.append(p_details['PROPERTY_NAME_UI'].lower())

# nb_properties

In [249]:
# property_and_nb_values = []
# for i in nb_properties:
#     for j in property_and_values:
#         if j[0] in i:
#             if (j[0], j[-1], 'nb') not in property_and_nb_values:
#                 property_and_nb_values.append((j[0], j[-1], 'nb'))

# property_and_nb_values

In [250]:
# prop_nb_data = {'Query': [], 'Output': []}
# property_and_nb_values = [
#     ('charpy', 'charpy'),
#     ('izod', 'izod'),
#     ('impact strength', 'impact'),
# ]
# for i in range(25):
#     for p in property_and_nb_values:
#         pname = p[1]
#         query = random.choice([
#             f"{pname} nobreak",
#             f"{pname} no break",
#             f"{pname} no-break",
#             f"{pname} nb",
            
#             f"nobreak {pname}",
#             f"no break {pname}",
#             f"no-break {pname}",
#             f"nb {pname}",
            
#             f"{pname} (nobreak)",
#             f"{pname} (no break)",
#             f"{pname} (no-break)",
#             f"{pname} (nb)",
            
#             f"(nobreak) {pname}",
#             f"(no break) {pname}",
#             f"(no-break) {pname}",
#             f"(nb) {pname}",
#         ])
#         output = {'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [{'property_name': p[0], 'modifier': {'value': 'nb', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}
#         prop_nb_data['Query'].append(query)
#         prop_nb_data['Output'].append(output)
    
# prop_nb_data_df = pd.DataFrame(prop_nb_data)

# cols = ["Sl. No."] + list(prop_nb_data_df)
# prop_nb_data_df["Sl. No."] = list(range(1, len(prop_nb_data_df)+1))
# prop_nb_data_df = prop_nb_data_df[cols]
# prop_nb_data_df

In [251]:
## prop_nb_data_df.to_excel("./prop_nb_data_29_11_24.xlsx", header=True, index=False)